In [ ]:
import os, time, math, copy, random, warnings, logging, json
from collections import defaultdict, Counter
from typing import Dict, List, Set, Tuple, Optional, Any

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")

from torch_geometric.nn import MessagePassing, global_mean_pool, SAGEConv, GATConv
from torch_geometric.data import Data, Batch
from torch_geometric.loader import DataLoader

from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score,
    recall_score, f1_score, roc_curve,
)
from scipy.stats import spearmanr, chi2_contingency, wilcoxon

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)


###########################################################################
# §1  UNIFIED MOLECULE REPRESENTATION
###########################################################################

class MoleculeGraph:
    """Parse a SMILES string once and cache everything the pipeline needs.

    The graph is built over ALL atoms (including explicit H added by
    RDKit) so that message passing sees the full molecular topology.
    However, only heavy atoms are treated as *players* in the
    cooperative game.  Each heavy atom "owns" its bonded H atoms:
    masking heavy atom i also masks every H bonded exclusively to i.
    """

    def __init__(self, smiles: str, use_mask_bit: bool = False):
        self.smiles = smiles
        self.use_mask_bit = use_mask_bit

        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            raise ValueError(f"Cannot parse SMILES: {smiles}")
        mol = Chem.AddHs(mol)
        self.mol = mol
        self.n_total = mol.GetNumAtoms()

        # --- identify heavy vs H atoms ---
        self.heavy_idx: List[int] = []     # indices of heavy atoms in the full graph
        self.h_idx: List[int] = []         # indices of H atoms
        for atom in mol.GetAtoms():
            if atom.GetAtomicNum() == 1:
                self.h_idx.append(atom.GetIdx())
            else:
                self.heavy_idx.append(atom.GetIdx())
        self.n_heavy = len(self.heavy_idx)

        # mapping: heavy atom index → list of H-atom indices bonded to it
        self.heavy_to_h: Dict[int, List[int]] = defaultdict(list)
        for h in self.h_idx:
            h_atom = mol.GetAtomWithIdx(h)
            for nb in h_atom.GetNeighbors():
                if nb.GetAtomicNum() != 1:
                    self.heavy_to_h[nb.GetIdx()].append(h)
                    break  # H is bonded to exactly one heavy atom

        # --- build atom features (8-dim scalar) + optional mask bit ---
        raw_feats = []
        for atom in mol.GetAtoms():
            raw_feats.append([
                atom.GetAtomicNum(),
                atom.GetDegree(),
                atom.GetTotalNumHs(),
                atom.GetTotalValence(),
                int(atom.GetIsAromatic()),
                atom.GetFormalCharge(),
                int(atom.GetHybridization()),
                int(atom.IsInRing()),
            ])
        raw_feats = np.array(raw_feats, dtype=np.float32)

        if use_mask_bit:
            # append a 0-column (mask bit = 0 means "real")
            mask_col = np.zeros((self.n_total, 1), dtype=np.float32)
            raw_feats = np.concatenate([raw_feats, mask_col], axis=1)

        self.x = torch.tensor(raw_feats, dtype=torch.float32)
        self.feat_dim = self.x.shape[1]

        # --- edges (full graph including H–heavy edges) ---
        edge_list = []
        for bond in mol.GetBonds():
            i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
            edge_list.append([i, j])
            edge_list.append([j, i])
        if edge_list:
            self.edge_index = torch.tensor(
                edge_list, dtype=torch.long,
            ).t().contiguous()
        else:
            self.edge_index = torch.zeros((2, 0), dtype=torch.long)

        # --- heavy-atom adjacency (for partition connectedness checks) ---
        self.heavy_adj: Dict[int, Set[int]] = defaultdict(set)
        for bond in mol.GetBonds():
            a, b = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
            if a in set(self.heavy_idx) and b in set(self.heavy_idx):
                self.heavy_adj[a].add(b)
                self.heavy_adj[b].add(a)

        # --- baseline feature vector ---
        # For the mask-bit variant: all-zeros features with mask_bit=1
        # For the legacy variant: mean of the real features
        if use_mask_bit:
            self.baseline = torch.zeros(self.feat_dim, dtype=torch.float32)
            self.baseline[-1] = 1.0       # mask bit ON
        else:
            self.baseline = self.x.mean(dim=0)

    def pyg_data(self) -> Data:
        """Return the full (unmasked) PyG Data object."""
        return Data(x=self.x.clone(), edge_index=self.edge_index.clone())

    def heavy_atom_symbol(self, idx: int) -> str:
        return self.mol.GetAtomWithIdx(idx).GetSymbol()

    def heavy_atom_labels(self) -> List[str]:
        """E.g. ['0:C', '1:N', '2:O', ...]  for heavy atoms only."""
        return [
            f"{i}:{self.mol.GetAtomWithIdx(i).GetSymbol()}"
            for i in self.heavy_idx
        ]


In [ ]:
###########################################################################
# §2  MODEL DEFINITION  (single-logit, MPNN → 3×GAT → SAGE readout)
###########################################################################

class MPNN(MessagePassing):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__(aggr='add')
        self.lin = nn.Linear(in_channels, out_channels)
        self.lin_update = nn.Linear(out_channels * 2, out_channels)

    def forward(self, x, edge_index):
        m = self.propagate(edge_index, x=x)
        x = self.lin_update(torch.cat([self.lin(x), m], dim=1))
        return F.leaky_relu(x, negative_slope=0.1)

    def message(self, x_j):
        return self.lin(x_j)


class DumplingGNN(nn.Module):
    """MPNN → 3×GAT → GraphSAGE → global mean pool → single-logit.

    This is the architecture described in the paper (Section 5.1).
    The raw output z is the explanation target; p(toxic) = σ(z).

    Parameters
    ----------
    input_dim : int
        Number of atom features.  8 for legacy, 9 with mask bit.
    hidden_channels : int
        Width of every hidden layer.
    dropout : float
        Dropout probability during training.
    """

    def __init__(
        self,
        input_dim: int = 8,
        hidden_channels: int = 64,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.mpnn = MPNN(input_dim, hidden_channels)
        self.gat1 = GATConv(
            hidden_channels, hidden_channels,
            heads=8, concat=True, dropout=dropout,
        )
        self.gat2 = GATConv(
            hidden_channels * 8, hidden_channels,
            heads=8, concat=True, dropout=dropout,
        )
        self.gat3 = GATConv(
            hidden_channels * 8, hidden_channels,
            heads=8, concat=True, dropout=dropout,
        )
        self.sage = SAGEConv(hidden_channels * 8, hidden_channels)
        # ---- single output neuron ----
        self.out = nn.Linear(hidden_channels, 1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, data: Data) -> torch.Tensor:
        """Returns shape [B] – one raw toxicity logit per molecule."""
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = self.mpnn(x, edge_index)
        x = F.relu(x)
        x = self.dropout(x)

        x = self.gat1(x, edge_index)
        x = F.elu(x)
        x = self.dropout(x)

        x = self.gat2(x, edge_index)
        x = F.leaky_relu(x)
        x = self.dropout(x)

        x = self.gat3(x, edge_index)
        x = F.elu(x)

        x = self.sage(x, edge_index)
        x = F.relu(x)

        x = global_mean_pool(x, batch)
        z = self.out(x).squeeze(-1)
        return z

    @torch.no_grad()
    def predict_proba(self, data: Data) -> torch.Tensor:
        """Return p(toxic) = sigmoid(z)."""
        self.eval()
        return torch.sigmoid(self.forward(data))


In [ ]:
###########################################################################
# §3  VALUE FUNCTION  (masked-node evaluation)
###########################################################################

def evaluate_coalition(
    model: nn.Module,
    mol_graph: MoleculeGraph,
    active_heavy: Set[int],
    device: torch.device,
) -> float:
    """Compute f_GNN(S) — the raw toxicity logit when only heavy atoms
    in *active_heavy* (and their bonded H atoms) retain original
    features, and everything else is masked to baseline.

    This is the characteristic function w(S) = f_GNN(S) from the paper
    (Eq. 7), before subtracting f_GNN(∅).

    Parameters
    ----------
    model : nn.Module
        Single-logit DumplingGNN.
    mol_graph : MoleculeGraph
        Pre-parsed molecule.
    active_heavy : set of int
        Indices of *heavy atoms* that are "active" (unmasked).
    device : torch.device

    Returns
    -------
    float   Raw toxicity logit z.
    """
    # Determine which total-graph nodes are active:
    # active heavy atoms + their owned H atoms
    active_total: Set[int] = set()
    for h in active_heavy:
        active_total.add(h)
        active_total.update(mol_graph.heavy_to_h.get(h, []))

    x_masked = mol_graph.x.clone()
    for i in range(mol_graph.n_total):
        if i not in active_total:
            x_masked[i] = mol_graph.baseline

    data = Data(x=x_masked, edge_index=mol_graph.edge_index)
    batch = Batch.from_data_list([data]).to(device)

    with torch.no_grad():
        model.eval()
        z = model(batch)
        return z.item()


class ValueFunction:
    """Caching wrapper around evaluate_coalition.

    Stores every coalition→logit mapping so repeated evaluations
    (which are frequent in the ST2 estimator) hit the cache.
    """

    def __init__(
        self,
        model: nn.Module,
        mol_graph: MoleculeGraph,
        device: torch.device,
    ):
        self.model = model
        self.mol_graph = mol_graph
        self.device = device
        self._cache: Dict[FrozenSet[int], float] = {}
        self.n_calls = 0       # total calls (cached + uncached)
        self.n_evals = 0       # actual model forward passes

    def __call__(self, active_heavy: Set[int]) -> float:
        key = frozenset(active_heavy)
        self.n_calls += 1
        if key not in self._cache:
            self._cache[key] = evaluate_coalition(
                self.model, self.mol_graph, active_heavy, self.device,
            )
            self.n_evals += 1
        return self._cache[key]

    @property
    def baseline_score(self) -> float:
        return self(set())

    @property
    def full_score(self) -> float:
        return self(set(self.mol_graph.heavy_idx))

    def stats(self) -> Dict[str, Any]:
        bs = self.baseline_score
        fs = self.full_score
        return {
            "baseline_score": bs,
            "full_score": fs,
            "total_change": fs - bs,
            "n_calls": self.n_calls,
            "n_evals": self.n_evals,
            "cache_size": len(self._cache),
        }


###########################################################################
# §4  SHAPLEY–TAYLOR ORDER-2 INTERACTION ESTIMATOR
###########################################################################

def compute_l_hop_neighborhood(
    adj: Dict[int, Set[int]],
    source_nodes: Set[int],
    l: int,
) -> Set[int]:
    """Return the l-hop neighbourhood of *source_nodes* in the
    heavy-atom adjacency graph."""
    if l == 0:
        return source_nodes.copy()
    visited = source_nodes.copy()
    frontier = source_nodes.copy()
    for _ in range(l):
        next_frontier: Set[int] = set()
        for node in frontier:
            for nb in adj.get(node, set()):
                if nb not in visited:
                    next_frontier.add(nb)
                    visited.add(nb)
        frontier = next_frontier
        if not frontier:
            break
    return visited


def compute_st2_interaction_matrix(
    vfunc: ValueFunction,
    mol_graph: MoleculeGraph,
    n_samples: int = 200,
    l_hop: Optional[int] = None,
    antithetic: bool = True,
    seed: Optional[int] = 42,
) -> np.ndarray:
    """Compute the atom-level degree-2 Shapley–Taylor interaction matrix
    over heavy atoms.

    Estimator
    ---------
    For each pair (i, j) of heavy atoms and each Monte Carlo sample:
      1. Draw a random permutation π of the context atoms
         C = Universe \\ {i, j}.
      2. Draw a uniform random prefix length k ∈ {0, …, |C|}.
      3. Let S = {π(1), …, π(k)}.
      4. Compute Δ_{ij}(S) = v(S∪{i,j}) − v(S∪{i}) − v(S∪{j}) + v(S).

    This gives an unbiased estimate of the Shapley interaction index
    I(i,j) = Σ_{S⊆N\\{i,j}} w(|S|) Δ_{ij}(S)  with Shapley weights
    w(s) = s!(n−s−2)! / (n−1)!,  because the uniform-(π, k) distribution
    induces exactly these weights over subsets.

    Variance reduction (antithetic=True):
      For the same permutation, pair the prefix S with its complement
      S' = C \\ S and average: (Δ(S) + Δ(S')) / 2.  Unbiased because S'
      is itself a valid uniform sample, and negatively correlated with S.

    Parameters
    ----------
    vfunc : ValueFunction
        Cached characteristic function over heavy-atom coalitions.
    mol_graph : MoleculeGraph
        Pre-parsed molecule.
    n_samples : int
        MC samples per atom pair.
    l_hop : int or None
        If set, restrict the context universe for pair (i,j) to the
        l-hop heavy-atom neighbourhood of {i,j}.
    antithetic : bool
        Whether to use antithetic variance reduction.
    seed : int or None
        Random seed for reproducibility.

    Returns
    -------
    W : np.ndarray, shape (n_heavy, n_heavy)
        Symmetric matrix indexed by *position in mol_graph.heavy_idx*.
        W[a][b] = estimated Shapley interaction index between heavy
        atoms heavy_idx[a] and heavy_idx[b].  Diagonal is zero.
    """
    if seed is not None:
        rng = np.random.RandomState(seed)
    else:
        rng = np.random.RandomState()

    heavy = mol_graph.heavy_idx
    n = len(heavy)
    # positional index → global atom index
    pos_to_global = {pos: gidx for pos, gidx in enumerate(heavy)}
    global_to_pos = {gidx: pos for pos, gidx in enumerate(heavy)}

    W = np.zeros((n, n), dtype=np.float64)

    total_pairs = n * (n - 1) // 2
    pairs_done = 0

    for a in range(n):
        gi = pos_to_global[a]
        for b in range(a + 1, n):
            gj = pos_to_global[b]

            # determine context universe (heavy atoms only)
            if l_hop is not None:
                universe = compute_l_hop_neighborhood(
                    mol_graph.heavy_adj, {gi, gj}, l_hop,
                )
            else:
                universe = set(heavy)
            context = sorted(universe - {gi, gj})
            n_ctx = len(context)

            if n_ctx == 0:
                # exact computation — no sampling needed
                S: Set[int] = set()
                delta = (
                    vfunc(S | {gi, gj})
                    - vfunc(S | {gi})
                    - vfunc(S | {gj})
                    + vfunc(S)
                )
                W[a, b] = delta
                W[b, a] = delta
            else:
                accum = 0.0
                for _ in range(n_samples):
                    perm = rng.permutation(context)
                    k = rng.randint(0, n_ctx + 1)
                    S_arr = set(perm[:k].tolist())

                    f_S = vfunc(S_arr)
                    f_Si = vfunc(S_arr | {gi})
                    f_Sj = vfunc(S_arr | {gj})
                    f_Sij = vfunc(S_arr | {gi, gj})
                    delta_main = f_Sij - f_Si - f_Sj + f_S

                    if antithetic:
                        S_anti = set(context) - S_arr
                        f_S2 = vfunc(S_anti)
                        f_Si2 = vfunc(S_anti | {gi})
                        f_Sj2 = vfunc(S_anti | {gj})
                        f_Sij2 = vfunc(S_anti | {gi, gj})
                        delta_anti = f_Sij2 - f_Si2 - f_Sj2 + f_S2
                        accum += (delta_main + delta_anti) / 2.0
                    else:
                        accum += delta_main

                W[a, b] = accum / n_samples
                W[b, a] = W[a, b]

            pairs_done += 1
            if pairs_done % max(1, total_pairs // 5) == 0:
                logger.info(
                    f"ST2 progress: {pairs_done}/{total_pairs} "
                    f"({100 * pairs_done / total_pairs:.0f}%)"
                )

    return W


def compute_main_effects(
    vfunc: ValueFunction,
    mol_graph: MoleculeGraph,
    n_samples: int = 200,
    seed: Optional[int] = 42,
) -> np.ndarray:
    """Estimate degree-1 Shapley values for each heavy atom.

    Uses the standard permutation-based MC estimator:
    for each permutation π, the marginal contribution of atom i is
    v(predecessors(i) ∪ {i}) − v(predecessors(i)).

    Returns
    -------
    phi : np.ndarray, shape (n_heavy,)
    """
    if seed is not None:
        rng = np.random.RandomState(seed)
    else:
        rng = np.random.RandomState()

    heavy = mol_graph.heavy_idx
    n = len(heavy)
    phi = np.zeros(n, dtype=np.float64)

    for _ in range(n_samples):
        perm = rng.permutation(n)             # permutation of positions
        coalition: Set[int] = set()
        prev_val = vfunc(set())
        for pos_in_perm in perm:
            gi = heavy[pos_in_perm]
            coalition.add(gi)
            cur_val = vfunc(coalition)
            phi[pos_in_perm] += cur_val - prev_val
            prev_val = cur_val

    phi /= n_samples
    return phi


def shapley_taylor_adjust(
    phi: np.ndarray,
    W: np.ndarray,
) -> np.ndarray:
    """Convert raw Shapley values to Shapley–Taylor order-2 main effects.

    The standard Shapley value φ_a already absorbs half of every
    pairwise interaction with its neighbours.  To avoid double-counting
    when we later add within-group interactions W[a,b] in the group
    aggregation (Eq. 9), we subtract that half:

        T_a  =  φ_a  −  ½ Σ_{b≠a} W[a, b]

    Efficiency property (exact in expectation):
        Σ_a T_a  +  Σ_{a<b} W[a,b]  =  Δz

    This guarantees that group values and cross-group interactions
    sum to exactly Δz, so percentage-of-Δz displays are meaningful.

    Parameters
    ----------
    phi : (n,)   raw Shapley values (positional indexing)
    W   : (n,n)  symmetric interaction matrix, zero diagonal

    Returns
    -------
    T : (n,)   adjusted main effects
    """
    # W is symmetric with zero diagonal, so row sum = Σ_{b≠a} W[a,b]
    return phi - 0.5 * W.sum(axis=1)


###########################################################################
# §5  GRAPH CONNECTIVITY UTILITIES
###########################################################################

def is_connected_subgraph(
    nodes: Set[int],
    adj: Dict[int, Set[int]],
) -> bool:
    """Check whether *nodes* induces a connected subgraph."""
    if len(nodes) <= 1:
        return True
    start = next(iter(nodes))
    visited = {start}
    stack = [start]
    while stack:
        v = stack.pop()
        for nb in adj.get(v, set()):
            if nb in nodes and nb not in visited:
                visited.add(nb)
                stack.append(nb)
    return visited == nodes


def split_into_connected_components(
    nodes: Set[int],
    adj: Dict[int, Set[int]],
) -> List[Set[int]]:
    """Split *nodes* into its connected components w.r.t. *adj*."""
    remaining = set(nodes)
    components: List[Set[int]] = []
    while remaining:
        start = next(iter(remaining))
        comp = {start}
        stack = [start]
        while stack:
            v = stack.pop()
            for nb in adj.get(v, set()):
                if nb in remaining and nb not in comp:
                    comp.add(nb)
                    stack.append(nb)
        components.append(comp)
        remaining -= comp
    return components


def enforce_connected_partition(
    partition: List[Set[int]],
    adj: Dict[int, Set[int]],
) -> List[Set[int]]:
    """Repair a partition so every group is connected.

    Disconnected groups are split into their connected components.
    """
    repaired: List[Set[int]] = []
    for group in partition:
        if len(group) <= 1 or is_connected_subgraph(group, adj):
            repaired.append(group)
        else:
            repaired.extend(split_into_connected_components(group, adj))
    return repaired


###########################################################################
# §6  GROUP-LEVEL STII AGGREGATION
###########################################################################

def aggregate_group_stii(
    W: np.ndarray,
    phi: np.ndarray,
    partition: List[List[int]],
    heavy_idx: List[int],
) -> Dict[str, Any]:
    """Aggregate atom-level Shapley–Taylor values to group level.

    Given a partition P = {S_1, …, S_K} of the heavy atoms, compute:
      Value(S_i)   = Σ_{a ∈ S_i} T_a  +  Σ_{a<b ∈ S_i} W[a,b]   (Eq. 9)
      I(S_i, S_j)  = Σ_{a ∈ S_i} Σ_{b ∈ S_j} W[a,b]               (Eq. 10)

    IMPORTANT: *phi* must be the **adjusted** Shapley–Taylor main effects
    T_a = φ_a − ½ Σ_{b≠a} W[a,b], NOT raw Shapley values.  Use
    shapley_taylor_adjust() before calling this function.  This ensures:

        Σ_i Value(S_i)  +  Σ_{i<j} Cross(S_i, S_j)  =  Δz

    Parameters
    ----------
    W : (n_heavy, n_heavy)  interaction matrix (positional indexing)
    phi : (n_heavy,)        adjusted main effects T (positional indexing)
    partition : list of list of int
        Each inner list contains *global* heavy-atom indices.
    heavy_idx : list of int
        Positional → global index mapping (mol_graph.heavy_idx).

    Returns
    -------
    dict with keys:
        group_values   – list of Value(S_i)
        cross_scores   – K×K matrix of I(S_i, S_j)
        total_within   – sum of group_values
        total_cross    – sum of upper-triangle cross_scores
    """
    global_to_pos = {g: p for p, g in enumerate(heavy_idx)}
    K = len(partition)

    # --- group values (Eq. 9) ---
    group_values: List[float] = []
    for group in partition:
        positions = [global_to_pos[g] for g in group if g in global_to_pos]
        # main effects
        val = sum(phi[p] for p in positions)
        # within-group interactions
        for ii in range(len(positions)):
            for jj in range(ii + 1, len(positions)):
                val += W[positions[ii], positions[jj]]
        group_values.append(float(val))

    # --- cross-group interactions (Eq. 10) ---
    cross = np.zeros((K, K), dtype=np.float64)
    for a in range(K):
        pos_a = [global_to_pos[g] for g in partition[a] if g in global_to_pos]
        for b in range(a + 1, K):
            pos_b = [global_to_pos[g] for g in partition[b] if g in global_to_pos]
            s = 0.0
            for pa in pos_a:
                for pb in pos_b:
                    s += W[pa, pb]
            cross[a, b] = s
            cross[b, a] = s

    return {
        "group_values": group_values,
        "cross_scores": cross.tolist(),
        "total_within": sum(group_values),
        "total_cross": float(np.triu(cross, 1).sum()),
    }


In [ ]:
###########################################################################
# §7  FUNCTIONAL GROUP DETECTION  (heavy atoms only, condensed SMARTS)
###########################################################################

# fmt: off
_FUNCTIONAL_GROUPS: List[Tuple[str, str]] = [
    # --- Halogenated ---
    ('Fluoro',                    '[F]'),
    ('Chloro',                    '[Cl]'),
    ('Bromo',                     '[Br]'),
    ('Iodo',                      '[I]'),
    ('Trifluoromethyl',           'C(F)(F)F'),
    ('Aryl Halide',               'a[F,Cl,Br,I]'),
    ('Acyl Halide',               '[CX3](=O)[F,Cl,Br,I]'),
    # --- Oxygen ---
    ('Carboxylic Acid',           '[CX3](=O)[OX2H1]'),
    ('Ester',                     '[#6][CX3](=O)[OX2][#6]'),
    ('Acid Anhydride',            '[CX3](=O)[OX2][CX3](=O)'),
    ('Aldehyde',                  '[CX3H1](=O)'),
    ('Ketone',                    '[#6][CX3](=O)[#6]'),
    ('Alcohol (Primary)',         '[CX4H2][OX2H]'),
    ('Alcohol (Secondary)',       '[CX4H1]([#6])[OX2H]'),
    ('Alcohol (Tertiary)',        '[CX4]([#6])([#6])[OX2H]'),
    ('Phenol',                    '[OX2H]c'),
    ('Ether',                     '[OX2]([#6])[#6]'),
    ('Epoxide',                   '[OX2r3]1[CX4r3][CX4r3]1'),
    ('Peroxide',                  '[OX2][OX2]'),
    ('Carbonyl (generic)',        '[CX3]=[OX1]'),
    ('Quinone',                   'O=c1ccc(=O)cc1'),
    # --- Nitrogen ---
    ('Amine (Primary)',           '[NX3H2][CX4]'),
    ('Amine (Secondary)',         '[NX3H1]([CX4])[CX4]'),
    ('Amine (Tertiary)',          '[NX3H0]([CX4])([CX4])[CX4]'),
    ('Aniline',                   '[NX3H2]c'),
    ('Amide (Primary)',           '[NX3H2][CX3]=O'),
    ('Amide (Secondary)',         '[NX3H1]([#6])[CX3]=O'),
    ('Amide (Tertiary)',          '[NX3H0]([#6])([#6])[CX3]=O'),
    ('Nitrile',                   '[NX1]#[CX2]'),
    ('Nitro',                     '[NX3+](=O)[O-]'),
    ('Nitroaromatic',             'c[NX3+](=O)[O-]'),
    ('Azo',                       '[NX2]=[NX2]'),
    ('Azide',                     '[NX1-]=[N+;X2]=[NX1]'),
    ('Imine',                     '[CX3]=[NX2]'),
    ('Oxime',                     '[CX3]=[NX2][OX2H]'),
    ('Hydrazine',                 '[NX3][NX3]'),
    ('Urea',                      '[NX3][CX3](=O)[NX3]'),
    ('Carbamate',                 '[NX3][CX3](=O)[OX2]'),
    ('Guanidine',                 '[NX3][CX3](=[NX2])[NX3]'),
    ('Isocyanate',                '[N;X2]=[C;X2]=O'),
    ('Isothiocyanate',            '[N;X2]=[C;X2]=S'),
    # --- N-heterocycles ---
    ('Pyrrole',                   'n1cccc1'),
    ('Pyridine',                  'n1ccccc1'),
    ('Imidazole',                 'n1cncc1'),
    ('Pyrazole',                  'n1nccc1'),
    ('Pyrimidine',                'n1cncnc1'),
    ('Triazole (1,2,3)',          'n1nncc1'),
    ('Triazole (1,2,4)',          'n1cnnc1'),
    ('Indole',                    'n1c(cccc2)c[cH]21'),
    ('Quinoline',                 'n1c(cccc2)cccc21'),
    # --- O/S/N-heterocycles ---
    ('Furan',                     'o1cccc1'),
    ('Oxazole',                   'o1cncc1'),
    ('Thiazole',                  's1cncc1'),
    ('Thiophene',                 's1cccc1'),
    ('Morpholine',                'O1CCNCC1'),
    ('Piperidine',                'N1CCCCC1'),
    ('Piperazine',                'N1CCNCC1'),
    # --- Sulfur ---
    ('Thiol',                     '[SX2H]'),
    ('Thioether',                 '[SX2]([#6])[#6]'),
    ('Disulfide',                 '[SX2][SX2]'),
    ('Sulfoxide',                 '[SX3](=O)[#6]'),
    ('Sulfone',                   '[SX4](=O)(=O)[#6]'),
    ('Sulfonic Acid',             '[SX4](=O)(=O)[OX2H]'),
    ('Sulfonamide (Primary)',     '[SX4](=O)(=O)[NX3H2]'),
    ('Sulfonamide (Secondary)',   '[SX4](=O)(=O)[NX3H1][#6]'),
    ('Thioamide',                 '[NX3][CX3]=S'),
    ('Thiourea',                  '[NX3][CX3](=S)[NX3]'),
    # --- Phosphorus ---
    ('Phosphine',                 '[PX3]'),
    ('Phosphine Oxide',           '[PX4]=O'),
    ('Phosphonate',               '[PX4](=O)([OX2][#6])[OX2][#6]'),
    # --- Rings ---
    ('Phenyl',                    'c1ccccc1'),
    ('Naphthalene',               'c1ccc2ccccc2c1'),
    ('Biphenyl',                  'c1ccccc1-c2ccccc2'),
    # --- Unsaturated ---
    ('Alkene',                    '[CX3]=[CX3]'),
    ('Alkyne',                    '[CX2]#[CX2]'),
    ('Allene',                    '[CX3]=[CX2]=[CX3]'),
    ('Enone',                     '[CX3]=[CX3][CX3](=O)[#6]'),
    ('Michael Acceptor',          '[C]=[C][C]=[O,N,S]'),
]
# fmt: on


def find_functional_groups(
    mol_graph: MoleculeGraph,
) -> Dict[str, List[Tuple[int, ...]]]:
    """Match SMARTS patterns against the molecule.

    Returns only heavy-atom indices (H indices are filtered out).
    """
    mol = mol_graph.mol
    heavy_set = set(mol_graph.heavy_idx)
    results: Dict[str, List[Tuple[int, ...]]] = {}

    for name, smarts in _FUNCTIONAL_GROUPS:
        pat = Chem.MolFromSmarts(smarts)
        if pat is None:
            continue
        matches = mol.GetSubstructMatches(pat)
        if not matches:
            continue
        # keep only heavy-atom indices
        filtered: List[Tuple[int, ...]] = []
        for match in matches:
            heavy_only = tuple(idx for idx in match if idx in heavy_set)
            if heavy_only:
                filtered.append(heavy_only)
        if filtered:
            results[name] = filtered

    return results


def functional_groups_to_partition(
    mol_graph: MoleculeGraph,
    fg_result: Optional[Dict[str, List[Tuple[int, ...]]]] = None,
) -> Tuple[List[List[int]], List[str]]:
    """Create an initial partition from functional group matches.

    Heavy atoms that are not claimed by any group are assigned to
    singleton groups or merged with an adjacent assigned group.
    Guarantees: every heavy atom appears in exactly one group.
    """
    if fg_result is None:
        fg_result = find_functional_groups(mol_graph)

    heavy_set = set(mol_graph.heavy_idx)
    assigned: Set[int] = set()
    partition: List[List[int]] = []
    group_names: List[str] = []

    # Priority ordering: complex groups first, then simpler ones
    _PRIORITY = [
        'Carboxylic Acid', 'Ester', 'Acid Anhydride',
        'Amide (Primary)', 'Amide (Secondary)', 'Amide (Tertiary)',
        'Sulfonamide (Primary)', 'Sulfonamide (Secondary)',
        'Nitro', 'Nitroaromatic', 'Nitrile',
        'Urea', 'Carbamate', 'Guanidine',
        'Phenyl', 'Naphthalene', 'Biphenyl',
        'Ketone', 'Aldehyde',
        'Alcohol (Primary)', 'Alcohol (Secondary)', 'Alcohol (Tertiary)',
        'Phenol',
        'Amine (Primary)', 'Amine (Secondary)', 'Amine (Tertiary)',
        'Aniline',
    ]

    def try_assign(name: str):
        if name not in fg_result:
            return
        for match in fg_result[name]:
            atoms = [idx for idx in match if idx in heavy_set and idx not in assigned]
            if atoms:
                partition.append(atoms)
                group_names.append(name)
                assigned.update(atoms)

    # pass 1: priority groups
    for name in _PRIORITY:
        try_assign(name)

    # pass 2: remaining functional groups (skip overly generic ones)
    skip = set(_PRIORITY) | {'Carbonyl (generic)', 'Alkene'}
    for name in fg_result:
        if name not in skip:
            try_assign(name)

    # pass 3: collect remaining heavy atoms
    remaining = heavy_set - assigned
    if remaining:
        # try to merge each remaining atom with an adjacent assigned group
        for atom in sorted(remaining):
            merged = False
            for nb in mol_graph.heavy_adj.get(atom, set()):
                for g_idx, group in enumerate(partition):
                    if nb in group:
                        partition[g_idx].append(atom)
                        assigned.add(atom)
                        merged = True
                        break
                if merged:
                    break
            if not merged:
                partition.append([atom])
                group_names.append(
                    f"Atom({mol_graph.heavy_atom_symbol(atom)}{atom})"
                )
                assigned.add(atom)

    return partition, group_names


In [ ]:
###########################################################################
# §8  CAP PARTITION SCORER
###########################################################################

class CAPScorer:
    """Scale-invariant partition scoring (revised Section 4.6).

    The original formulation computed coverage in raw logit units
    (~6–30 for typical toxicity models) while entropy and size
    penalties lived in bounded ranges (~0–4).  This made the
    penalties invisible and the scorer always preferred 1 group.

    Additionally, the entropy term H_exp(L_m) was *zero* for a
    single-component solution, effectively rewarding the degenerate
    partition instead of penalising it.

    Revised scoring
    ───────────────
    S(P) = max_{1≤m≤ℓ} [ NormCov_m
                          − γ_C · ConcentrationPenalty(L_m)
                          − γ_S · SizePenalty(P) ]

    where:
      NormCov_m   = Σ_{j=1}^{m}  (v_j / V_total) / j
                    Coverage with each v_j divided by the sum of all
                    positive component values, so NormCov ∈ (0, 1].
                    Harmonic weighting rewards fewer, stronger components.

      ConcentrationPenalty = 1 − H_exp(L_m) / log₂(m)     ∈ [0, 1]
                    Penalises *low* entropy (one component dominates).
                    For m = 1 the penalty is 1.0 (maximum), driving
                    the scorer away from the degenerate 1-group solution.
                    For m equal components the penalty is 0.

      SizePenalty = (1/K) Σ_i (|S_i|/μ − 1)²
                    Linear-scale penalty (not log-scale) so large groups
                    are penalised more aggressively.  Applied to ALL groups
                    in P, not just the top-m subset.
    """

    def __init__(
        self,
        group_values: List[float],
        cross_scores: List[List[float]],
        group_sizes: List[int],
        gamma_C: float = 0.4,
        gamma_S: float = 0.3,
        mu_ideal: float = 5.0,
    ):
        self.gamma_C = gamma_C
        self.gamma_S = gamma_S
        self.mu_ideal = mu_ideal
        self.group_sizes = group_sizes

        # --- collect positive first- and second-order contributions ---
        self._components: List[Dict[str, Any]] = []
        for i, v in enumerate(group_values):
            if v > 0:
                self._components.append(
                    {"type": "group", "index": i, "value": v}
                )
        K = len(group_values)
        for i in range(K):
            for j in range(i + 1, K):
                v = cross_scores[i][j]
                if v > 0:
                    self._components.append(
                        {"type": "pair", "index": (i, j), "value": v}
                    )

        # sort descending
        self._components.sort(key=lambda c: c["value"], reverse=True)
        self._ell = len(self._components)

        # normalisation constant: sum of ALL positive component values
        self._V_total = sum(c["value"] for c in self._components)
        if self._V_total <= 0:
            self._V_total = 1.0          # guard against zero-division

        # precompute the global size penalty (independent of m)
        self._global_size_penalty = self._compute_size_penalty()

    # ------------------------------------------------------------------
    def _compute_size_penalty(self) -> float:
        """(1/K) Σ_i (|S_i|/μ − 1)²  over ALL groups in the partition."""
        K = len(self.group_sizes)
        if K == 0 or self.mu_ideal <= 0:
            return 0.0
        penalty = 0.0
        for sz in self.group_sizes:
            ratio = sz / self.mu_ideal
            penalty += (ratio - 1.0) ** 2
        return penalty / K

    # ------------------------------------------------------------------
    @staticmethod
    def _concentration_penalty(L_m: List[Dict], V_total: float) -> float:
        """1 − H(L_m) / log₂(m).

        Returns a value in [0, 1]:
          - 1.0  when m = 1  (maximally concentrated → worst)
          - 0.0  when m components are equal (maximally spread → best)
        """
        m = len(L_m)
        if m <= 1:
            return 1.0                    # single component → max penalty

        total = sum(c["value"] for c in L_m)
        if total <= 0:
            return 1.0

        H = 0.0
        for c in L_m:
            p = c["value"] / total
            if p > 0:
                H -= p * math.log2(p)

        H_max = math.log2(m)              # entropy of uniform over m items
        if H_max <= 0:
            return 1.0
        return 1.0 - H / H_max

    # ------------------------------------------------------------------
    def score(self) -> Tuple[float, int, List[Dict]]:
        """Return (best_score, optimal_m, optimal_components)."""
        if self._ell == 0:
            return 0.0, 0, []

        best_score = -float("inf")
        best_m = 0
        best_Lm: List[Dict] = []
        cum_coverage = 0.0

        for m in range(1, self._ell + 1):
            L_m = self._components[:m]

            # normalised harmonic-weighted coverage ∈ (0, 1]
            norm_v = self._components[m - 1]["value"] / self._V_total
            cum_coverage += norm_v / m

            conc_pen = self._concentration_penalty(L_m, self._V_total)

            s_m = (
                cum_coverage
                - self.gamma_C * conc_pen
                - self.gamma_S * self._global_size_penalty
            )
            if s_m > best_score:
                best_score = s_m
                best_m = m
                best_Lm = list(L_m)

        return max(0.0, best_score), best_m, best_Lm


###########################################################################
# §9  SIMULATED ANNEALING PARTITION SEARCH
###########################################################################

class PartitionSearcher:
    """Simulated annealing over the space of connected partitions
    of the heavy-atom graph.

    Every candidate partition is validated for connectedness before
    scoring.  If a move produces a disconnected group, it is
    automatically repaired by splitting into connected components.
    """

    def __init__(
        self,
        mol_graph: MoleculeGraph,
        W: np.ndarray,
        phi: np.ndarray,
        gamma_C: float = 0.4,
        gamma_S: float = 0.3,
        mu_ideal: float = 5.0,
    ):
        self.mol_graph = mol_graph
        self.W = W
        self.phi = phi
        self.gamma_C = gamma_C
        self.gamma_S = gamma_S
        self.mu_ideal = mu_ideal

        self.heavy = mol_graph.heavy_idx
        self.n = len(self.heavy)
        self.adj = mol_graph.heavy_adj

        # pre-detect fragments for merge scoring
        fg = find_functional_groups(mol_graph)
        self._known_fragments: List[Set[int]] = []
        for matches in fg.values():
            for m in matches:
                self._known_fragments.append(set(m))
        ring_info = mol_graph.mol.GetRingInfo()
        for ring in ring_info.AtomRings():
            heavy_in_ring = {a for a in ring if a in set(self.heavy)}
            if heavy_in_ring:
                self._known_fragments.append(heavy_in_ring)

        self._score_cache: Dict[FrozenSet[FrozenSet[int]], float] = {}
        self.best_partition: Optional[List[Set[int]]] = None
        self.best_score = -float("inf")
        self.log: Dict[str, List] = {
            "iteration": [], "temperature": [], "current_score": [],
            "best_score": [], "accepted": [], "n_groups": [],
        }

    # ----- scoring -----

    def _evaluate(self, partition: List[Set[int]]) -> float:
        key = frozenset(frozenset(g) for g in partition)
        if key in self._score_cache:
            return self._score_cache[key]

        part_lists = [sorted(g) for g in partition]
        agg = aggregate_group_stii(
            self.W, self.phi, part_lists, self.heavy,
        )
        sizes = [len(g) for g in partition]
        scorer = CAPScorer(
            group_values=agg["group_values"],
            cross_scores=agg["cross_scores"],
            group_sizes=sizes,
            gamma_C=self.gamma_C,
            gamma_S=self.gamma_S,
            mu_ideal=self.mu_ideal,
        )
        s, _, _ = scorer.score()
        self._score_cache[key] = s
        return s

    # ----- initial partition strategies -----

    def _init_functional_groups(self) -> List[Set[int]]:
        part, _ = functional_groups_to_partition(self.mol_graph)
        return [set(g) for g in part]

    def _init_singletons(self) -> List[Set[int]]:
        return [{h} for h in self.heavy]

    def _init_rings_first(self) -> List[Set[int]]:
        assigned: Set[int] = set()
        partition: List[Set[int]] = []
        ring_info = self.mol_graph.mol.GetRingInfo()
        for ring in ring_info.AtomRings():
            heavy_ring = {a for a in ring if a in set(self.heavy)} - assigned
            if heavy_ring:
                partition.append(heavy_ring)
                assigned.update(heavy_ring)
        for h in self.heavy:
            if h not in assigned:
                partition.append({h})
                assigned.add(h)
        return partition

    # ----- move operators -----

    def _atom_to_group(self, partition: List[Set[int]]) -> Dict[int, int]:
        return {a: gi for gi, g in enumerate(partition) for a in g}

    def _boundary_atoms(
        self, partition: List[Set[int]],
    ) -> List[Tuple[int, int, int]]:
        a2g = self._atom_to_group(partition)
        boundary = []
        for atom, gi in a2g.items():
            for nb in self.adj.get(atom, set()):
                gj = a2g.get(nb)
                if gj is not None and gj != gi:
                    boundary.append((atom, gi, gj))
                    break
        return boundary

    def _move_atom(
        self, partition: List[Set[int]], rng: np.random.RandomState,
    ) -> Optional[List[Set[int]]]:
        boundary = self._boundary_atoms(partition)
        if not boundary:
            return None
        atom, from_g, to_g = boundary[rng.randint(len(boundary))]
        new_part: List[Set[int]] = []
        for i, g in enumerate(partition):
            if i == from_g:
                rem = g - {atom}
                if rem:
                    new_part.append(rem)
            elif i == to_g:
                new_part.append(g | {atom})
            else:
                new_part.append(set(g))
        return new_part

    def _merge_groups(
        self, partition: List[Set[int]], rng: np.random.RandomState,
    ) -> Optional[List[Set[int]]]:
        a2g = self._atom_to_group(partition)
        pairs: Set[Tuple[int, int]] = set()
        for atom, gi in a2g.items():
            for nb in self.adj.get(atom, set()):
                gj = a2g.get(nb)
                if gj is not None and gi < gj:
                    pairs.add((gi, gj))
        if not pairs:
            return None
        pairs_list = list(pairs)
        gi, gj = pairs_list[rng.randint(len(pairs_list))]
        new_part: List[Set[int]] = []
        for i, g in enumerate(partition):
            if i == gi:
                continue
            elif i == gj:
                new_part.append(partition[gi] | partition[gj])
            else:
                new_part.append(set(g))
        return new_part

    def _split_group(
        self, partition: List[Set[int]], rng: np.random.RandomState,
    ) -> Optional[List[Set[int]]]:
        splittable = [
            (i, g) for i, g in enumerate(partition) if len(g) > 1
        ]
        if not splittable:
            return None
        idx = rng.randint(len(splittable))
        gi, group = splittable[idx]
        # find a terminal atom (degree 1 within the group)
        terminals = []
        for atom in group:
            nbs_in_group = sum(1 for nb in self.adj.get(atom, set()) if nb in group)
            if nbs_in_group <= 1:
                terminals.append(atom)
        if terminals:
            atom = terminals[rng.randint(len(terminals))]
        else:
            atom = list(group)[rng.randint(len(group))]

        new_part: List[Set[int]] = []
        for i, g in enumerate(partition):
            if i == gi:
                rem = g - {atom}
                if rem:
                    new_part.append(rem)
                new_part.append({atom})
            else:
                new_part.append(set(g))
        return new_part

    def _propose(
        self, partition: List[Set[int]], rng: np.random.RandomState,
    ) -> List[Set[int]]:
        ops = [self._move_atom, self._merge_groups, self._split_group]
        rng.shuffle(ops)
        for op in ops:
            result = op(partition, rng)
            if result is not None:
                # enforce connectedness
                result = enforce_connected_partition(result, self.adj)
                return result
        return list(partition)  # no move possible

    # ----- main search loop -----

    def search(
        self,
        strategy: str = "functional_groups",
        max_iter: int = 1000,
        T0: float = 1.0,
        alpha: float = 0.99,
        seed: int = 42,
        verbose: bool = True,
    ) -> Tuple[List[Set[int]], float]:
        rng = np.random.RandomState(seed)
        random.seed(seed)

        # initialization
        if strategy == "functional_groups":
            current = self._init_functional_groups()
        elif strategy == "singletons":
            current = self._init_singletons()
        elif strategy == "rings_first":
            current = self._init_rings_first()
        else:
            current = self._init_functional_groups()

        current = enforce_connected_partition(current, self.adj)
        cur_score = self._evaluate(current)
        self.best_partition = copy.deepcopy(current)
        self.best_score = cur_score
        T = T0

        if verbose:
            logger.info(
                f"SA start: {len(current)} groups, score={cur_score:.4f}"
            )

        for it in range(max_iter):
            candidate = self._propose(current, rng)
            cand_score = self._evaluate(candidate)
            delta = cand_score - cur_score

            if delta > 0 or (T > 0 and rng.rand() < math.exp(delta / T)):
                current = candidate
                cur_score = cand_score
                accepted = True
                if cur_score > self.best_score:
                    self.best_partition = copy.deepcopy(current)
                    self.best_score = cur_score
                    if verbose and it % 50 == 0:
                        logger.info(
                            f"  iter {it}: new best {self.best_score:.4f} "
                            f"({len(self.best_partition)} groups)"
                        )
            else:
                accepted = False

            T *= alpha
            self.log["iteration"].append(it)
            self.log["temperature"].append(T)
            self.log["current_score"].append(cur_score)
            self.log["best_score"].append(self.best_score)
            self.log["accepted"].append(accepted)
            self.log["n_groups"].append(len(current))

        if verbose:
            logger.info(
                f"SA done: best score={self.best_score:.4f}, "
                f"{len(self.best_partition)} groups, "
                f"{len(self._score_cache)} partitions evaluated"
            )
        return self.best_partition, self.best_score

    def plot_progress(self):
        if not self.log["iteration"]:
            print("No search data. Run search() first.")
            return
        fig, axes = plt.subplots(2, 2, figsize=(12, 8))
        iters = self.log["iteration"]

        axes[0, 0].plot(iters, self.log["current_score"], alpha=0.5, label="current")
        axes[0, 0].plot(iters, self.log["best_score"], lw=2, label="best")
        axes[0, 0].set_ylabel("CAP score"); axes[0, 0].legend(); axes[0, 0].grid(True, alpha=0.3)

        axes[0, 1].plot(iters, self.log["temperature"])
        axes[0, 1].set_ylabel("temperature"); axes[0, 1].set_yscale("log"); axes[0, 1].grid(True, alpha=0.3)

        win = 50
        if len(iters) > win:
            rates = [
                np.mean(self.log["accepted"][max(0, i - win):i])
                for i in range(win, len(iters))
            ]
            axes[1, 0].plot(range(win, len(iters)), rates)
        axes[1, 0].set_ylabel("accept rate"); axes[1, 0].grid(True, alpha=0.3)

        axes[1, 1].plot(iters, self.log["n_groups"])
        axes[1, 1].set_ylabel("# groups"); axes[1, 1].grid(True, alpha=0.3)

        for ax in axes.flat:
            ax.set_xlabel("iteration")
        plt.suptitle("Simulated Annealing Progress")
        plt.tight_layout(); plt.show()


In [ ]:
###########################################################################
# §10  HIGH-LEVEL API: FULL EXPLANATION PIPELINE
###########################################################################

def explain_molecule(
    model: nn.Module,
    smiles: str,
    device: torch.device,
    n_samples_st2: int = 200,
    n_samples_sv: int = 200,
    l_hop: Optional[int] = None,
    antithetic: bool = True,
    sa_iterations: int = 500,
    sa_T0: float = 1.0,
    sa_alpha: float = 0.99,
    gamma_C: float = 0.4,
    gamma_S: float = 0.3,
    mu_ideal: float = 5.0,
    use_mask_bit: bool = False,
    seed: int = 42,
    verbose: bool = True,
) -> Dict[str, Any]:
    """End-to-end explanation of a single molecule.

    Steps:
      1. Parse SMILES → MoleculeGraph
      2. Build ValueFunction (cached f_GNN evaluations)
      3. Compute degree-1 (main effects) and degree-2 (interactions)
         Shapley–Taylor values over heavy atoms
      4. Build initial partition from functional groups
      5. Run simulated annealing to maximize CAP score
      6. Return everything

    Returns a dict with keys:
        mol_graph, vfunc_stats, W, phi,
        init_partition, init_names, init_score,
        best_partition, best_score, sa_log,
        group_agg (for best partition)
    """
    t0 = time.time()

    # 1. parse
    mg = MoleculeGraph(smiles, use_mask_bit=use_mask_bit)
    if verbose:
        logger.info(
            f"Molecule: {smiles}  heavy={mg.n_heavy}  total={mg.n_total}"
        )

    # 2. value function
    vfunc = ValueFunction(model, mg, device)
    bs = vfunc.baseline_score
    fs = vfunc.full_score
    if verbose:
        logger.info(f"Baseline z={bs:.4f}  Full z={fs:.4f}  Δ={fs - bs:.4f}")

    # 3. Shapley–Taylor
    if verbose:
        logger.info("Computing degree-1 Shapley values...")
    phi = compute_main_effects(vfunc, mg, n_samples=n_samples_sv, seed=seed)

    if verbose:
        logger.info("Computing degree-2 interaction matrix W...")
    W = compute_st2_interaction_matrix(
        vfunc, mg,
        n_samples=n_samples_st2,
        l_hop=l_hop,
        antithetic=antithetic,
        seed=seed,
    )

    # 3b. Adjust main effects for Shapley-Taylor order-2 efficiency
    #     T_a = φ_a − ½ Σ_{b≠a} W[a,b]
    #     ensures: Σ T_a + Σ_{a<b} W_{ab} = Δz  (no double-counting)
    phi_raw = phi.copy()
    phi = shapley_taylor_adjust(phi, W)
    if verbose:
        efficiency_check = phi.sum() + np.triu(W, 1).sum()
        delta_z = fs - bs
        logger.info(
            f"Efficiency check: Σ T_a + Σ W_ab = {efficiency_check:.4f}  "
            f"vs  Δz = {delta_z:.4f}  "
            f"(error = {abs(efficiency_check - delta_z):.4f})"
        )

    # 4. initial partition
    init_part, init_names = functional_groups_to_partition(mg)
    init_part_sets = [set(g) for g in init_part]
    init_part_sets = enforce_connected_partition(init_part_sets, mg.heavy_adj)

    # 5. SA search
    searcher = PartitionSearcher(
        mg, W, phi,
        gamma_C=gamma_C, gamma_S=gamma_S, mu_ideal=mu_ideal,
    )
    # evaluate initial partition score
    init_score = searcher._evaluate(init_part_sets)

    best_part, best_score = searcher.search(
        strategy="functional_groups",
        max_iter=sa_iterations,
        T0=sa_T0,
        alpha=sa_alpha,
        seed=seed,
        verbose=verbose,
    )

    # 6. aggregate for best partition
    best_lists = [sorted(g) for g in best_part]
    best_agg = aggregate_group_stii(W, phi, best_lists, mg.heavy_idx)

    elapsed = time.time() - t0
    if verbose:
        logger.info(f"Explanation complete in {elapsed:.1f}s")
        logger.info(
            f"  model evals: {vfunc.n_evals}  "
            f"(cache hits: {vfunc.n_calls - vfunc.n_evals})"
        )

    return {
        "smiles": smiles,
        "mol_graph": mg,
        "vfunc_stats": vfunc.stats(),
        "W": W,
        "phi": phi,            # adjusted ST2 main effects (T_a)
        "phi_raw": phi_raw,    # raw Shapley values (φ_a)
        "init_partition": init_part,
        "init_names": init_names,
        "init_score": init_score,
        "best_partition": best_lists,
        "best_score": best_score,
        "sa_log": searcher.log,
        "group_agg": best_agg,
        "elapsed_seconds": elapsed,
    }


def print_explanation(result: Dict[str, Any]):
    """Pretty-print the output of explain_molecule."""
    mg: MoleculeGraph = result["mol_graph"]
    stats = result["vfunc_stats"]
    delta_z = stats["total_change"]

    print("=" * 70)
    print(f"EXPLANATION: {result['smiles']}")
    print(f"  heavy atoms: {mg.n_heavy}   total atoms: {mg.n_total}")
    print(f"  baseline z = {stats['baseline_score']:.4f}")
    print(f"  full z     = {stats['full_score']:.4f}")
    print(f"  Δz         = {delta_z:.4f}")
    print(f"  p(toxic)   = {1/(1+math.exp(-stats['full_score'])):.4f}")
    print(f"  model evals: {stats['n_evals']}")
    print()

    agg = result["group_agg"]
    bp = result["best_partition"]
    print(f"Best partition ({len(bp)} groups, CAP={result['best_score']:.4f}):")
    abs_dz = abs(delta_z) if abs(delta_z) > 1e-8 else 1.0
    for i, group in enumerate(bp):
        atoms = [f"{mg.heavy_atom_symbol(a)}{a}" for a in group]
        v = agg["group_values"][i]
        frac = v / abs_dz
        print(f"  G{i}: {{{', '.join(atoms)}}}  "
              f"Value={v:+.4f}  ({frac:+.0%} of Δz)")

    # top cross-group interactions
    cross = agg["cross_scores"]
    K = len(bp)
    interactions = []
    for a in range(K):
        for b in range(a + 1, K):
            v = cross[a][b]
            if abs(v) > 1e-6:
                interactions.append((a, b, v))
    interactions.sort(key=lambda x: abs(x[2]), reverse=True)
    if interactions:
        print(f"\nTop group interactions:")
        for a, b, v in interactions[:5]:
            kind = "synergistic" if v > 0 else "antagonistic"
            frac = v / abs_dz
            print(f"  G{a} ↔ G{b}: {v:+.4f} ({frac:+.0%} of Δz)  ({kind})")

    # efficiency check: values + cross-interactions should sum to Δz
    total_values = sum(agg["group_values"])
    total_cross = sum(
        cross[a][b] for a in range(K) for b in range(a + 1, K)
    )
    reconstructed = total_values + total_cross
    efficiency_err = abs(reconstructed - delta_z) / abs_dz if abs_dz > 1e-8 else 0
    print(f"\n  Σ Values + Σ Cross = {reconstructed:.4f}  "
          f"(Δz = {delta_z:.4f}, error = {efficiency_err:.1%})")

    print(f"\nElapsed: {result['elapsed_seconds']:.1f}s")
    print("=" * 70)


# ─────────────────────────────────────────────────────────────────────────────
# Case-study figure exporter.
# Saves one PDF panel per molecule showing the SA-discovered partition (atom
# colouring by group) alongside a bar chart of intrinsic + pairwise scores.
# Used by the demo cell below to generate supplementary figures for the paper.
# ─────────────────────────────────────────────────────────────────────────────


def save_case_study_figure(result, label, out_path):
    """Render a two-panel figure: (left) molecule drawn with atoms coloured by
    SA group, (right) bar chart of group values + dominant pairwise interactions.

    Parameters
    ----------
    result : dict
        Output of `explain_molecule_for_verification` (or the cell-6 demo).
    label : str
        Short molecule name used in the title.
    out_path : str
        Where to write the PDF.
    """
    try:
        from rdkit.Chem import Draw, AllChem
        from rdkit.Chem.Draw import rdMolDraw2D
    except ImportError:
        print(f"  [{label}] RDKit drawing not available; skipping figure.")
        return False

    mg = result["mol_graph"]
    partition = result["best_partition"]
    agg = result["group_agg"]
    delta_z = result["vfunc_stats"]["total_change"]
    abs_dz = abs(delta_z) if abs(delta_z) > 1e-8 else 1.0

    # Build a color per atom — same color for all atoms in the same group.
    palette = [
        (0.30, 0.30, 0.30),  # G0 grey (acetyl-tail style)
        (0.20, 0.40, 0.85),  # G1 blue
        (0.85, 0.25, 0.20),  # G2 red
        (0.25, 0.70, 0.30),  # G3 green
        (0.85, 0.55, 0.15),  # G4 orange
        (0.55, 0.30, 0.75),  # G5 purple
        (0.20, 0.65, 0.65),  # G6 teal
    ]
    atom_colors = {}
    for gi, group in enumerate(partition):
        c = palette[gi % len(palette)]
        for atom_idx in group:
            atom_colors[int(atom_idx)] = c

    # Draw the molecule.
    mol_for_draw = Chem.Mol(mg.mol)
    try:
        AllChem.Compute2DCoords(mol_for_draw)
    except Exception:
        pass
    drawer = rdMolDraw2D.MolDraw2DCairo(450, 450)
    rdMolDraw2D.PrepareAndDrawMolecule(
        drawer, mol_for_draw,
        highlightAtoms=list(atom_colors.keys()),
        highlightAtomColors=atom_colors,
    )
    drawer.FinishDrawing()
    png_bytes = drawer.GetDrawingText()

    # Render with matplotlib for a side-by-side layout.
    fig, axes = plt.subplots(1, 2, figsize=(11, 5),
                             gridspec_kw={"width_ratios": [1, 1.2]})
    import io
    img = plt.imread(io.BytesIO(png_bytes), format="png")
    axes[0].imshow(img)
    axes[0].axis("off")
    axes[0].set_title(f"{label}: SA partition")

    # Bar chart of intrinsic values + top pairwise interactions.
    K = len(partition)
    bar_labels = [f"G{i}" for i in range(K)]
    bar_values = [v / abs_dz for v in agg["group_values"]]
    bar_colors = [palette[i % len(palette)] for i in range(K)]

    cross = agg["cross_scores"]
    pairs = sorted(
        [(i, j, cross[i][j]) for i in range(K) for j in range(i + 1, K)
         if abs(cross[i][j]) > 1e-6],
        key=lambda x: abs(x[2]), reverse=True,
    )[:5]
    pair_labels = [f"G{i} - G{j}" for i, j, _ in pairs]
    pair_values = [v / abs_dz for _, _, v in pairs]

    all_labels = bar_labels + pair_labels
    all_values = bar_values + pair_values
    all_colors = bar_colors + [(0.4, 0.4, 0.4)] * len(pairs)
    ax = axes[1]
    y = np.arange(len(all_labels))
    ax.barh(y, all_values, color=all_colors)
    ax.set_yticks(y)
    ax.set_yticklabels(all_labels)
    ax.axvline(0, color="black", lw=0.5)
    ax.set_xlabel(r"Contribution / $|\Delta z|$")
    ax.set_title(rf"Decomposition (total $\Delta z = {delta_z:+.2f}$)")
    ax.grid(True, axis="x", alpha=0.3)

    plt.tight_layout()
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"  [{label}] Saved figure to {out_path}")
    return True



In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# §6b  UGROPY GROUND-TRUTH PARTITION HELPERS
#
# ugropy (Flores et al., I&EC Research 2025; pip install ugropy) provides a
# deterministic, Set-Cover-optimal atom-to-functional-group assignment for
# UNIFAC, Joback, Dortmund, and other thermodynamic group-contribution models.
# We use it as an *external, chemistry-grounded ground truth* for our learned
# SA partitions: how often does our explainer recover a single GC group?
# Comparison against the external chemistry-grounded fragmentation.
# ─────────────────────────────────────────────────────────────────────────────


def compute_ugropy_partition(smiles, gc_model="unifac"):
    """Return the canonical atom-to-group partition of `smiles` under a chosen
    group-contribution model. Output is a list of dicts:
        [{"name": <group_name>, "atoms": frozenset[int]}, ...]
    where atom indices match the heavy-atom indices of the RDKit MolFromSmiles
    (without explicit H, matching MoleculeGraph.heavy_idx convention).
    Returns None if ugropy isn't installed or the molecule isn't representable
    in the chosen GC model.
    """
    try:
        import ugropy
    except ImportError:
        return None
    try:
        if gc_model == "unifac":
            groups_obj = ugropy.unifac.get_groups(smiles, identifier_type="smiles")
        elif gc_model == "joback":
            groups_obj = ugropy.joback.get_groups(smiles, identifier_type="smiles")
        elif gc_model in ("dortmund", "modified_unifac"):
            groups_obj = ugropy.psrk.get_groups(smiles, identifier_type="smiles")  # PSRK uses Dortmund
        else:
            return None
    except Exception:
        return None
    if groups_obj is None or not hasattr(groups_obj, "subgroups_atoms"):
        return None
    partition = []
    for grp_name, atom_groups_list in groups_obj.subgroups_atoms.items():
        # ugropy returns list of tuples; flatten across occurrences of same group
        atom_indices = []
        for tup in atom_groups_list:
            atom_indices.extend(tup)
        atoms = frozenset(int(a) for a in atom_indices)
        if atoms:
            partition.append({"name": str(grp_name), "atoms": atoms})
    return partition or None


def partition_agreement(our_partition, gc_partition):
    """Quantify how well our SA-discovered partition aligns with the canonical
    chemistry partition. We use the best-match Jaccard averaged over our
    groups: for each subgraph S_i in our partition, find the GC group g with
    maximum |S_i ∩ g| / |S_i ∪ g|, then average those Jaccards.

    A symmetric score is also returned (averaged in both directions).
    """
    if not our_partition or not gc_partition:
        return {"jaccard_ours_to_gc": 0.0,
                "jaccard_gc_to_ours": 0.0,
                "jaccard_symmetric": 0.0}

    our_sets = [set(g) if not isinstance(g, set) else g for g in our_partition]
    gc_sets = [set(g["atoms"]) for g in gc_partition]

    def _best_jaccard(group, candidates):
        best = 0.0
        for c in candidates:
            inter = len(group & c)
            union = len(group | c)
            if union > 0:
                best = max(best, inter / union)
        return best

    js_ours = [_best_jaccard(s, gc_sets) for s in our_sets]
    js_gc = [_best_jaccard(s, our_sets) for s in gc_sets]
    j_ours = float(np.mean(js_ours)) if js_ours else 0.0
    j_gc = float(np.mean(js_gc)) if js_gc else 0.0
    return {
        "jaccard_ours_to_gc": j_ours,
        "jaccard_gc_to_ours": j_gc,
        "jaccard_symmetric": 0.5 * (j_ours + j_gc),
    }


def evaluate_ugropy_alignment(model, smiles_list, gc_model="unifac",
                              n_samples_st2=100, sa_iterations=300):
    """Run the explainer on each SMILES in `smiles_list`, also compute the
    ugropy partition, and report per-molecule + average Jaccard alignment.
    Skips molecules that ugropy can't represent.
    """
    records = []
    for smi in smiles_list:
        try:
            result = explain_molecule_for_verification(
                model, smi, CFG.DEVICE,
                n_samples_st2=n_samples_st2,
                n_samples_sv=n_samples_st2,
                sa_iterations=sa_iterations,
            )
        except Exception:
            continue
        if result is None:
            continue
        our_part = result["best_partition"]  # list of lists of heavy-atom indices
        gc_part = compute_ugropy_partition(smi, gc_model=gc_model)
        if gc_part is None:
            continue
        agreement = partition_agreement(our_part, gc_part)
        records.append({
            "smiles": smi,
            "n_our_groups": len(our_part),
            "n_gc_groups": len(gc_part),
            **agreement,
        })
    df = pd.DataFrame(records)
    if not df.empty:
        print(f"\n  ugropy alignment on {len(df)} molecules:")
        print(f"    Jaccard (ours → GC):     {df['jaccard_ours_to_gc'].mean():.3f} "
              f"± {df['jaccard_ours_to_gc'].std():.3f}")
        print(f"    Jaccard (GC → ours):     {df['jaccard_gc_to_ours'].mean():.3f} "
              f"± {df['jaccard_gc_to_ours'].std():.3f}")
        print(f"    Symmetric Jaccard:        {df['jaccard_symmetric'].mean():.3f} "
              f"± {df['jaccard_symmetric'].std():.3f}")
    return df


if __name__ == "__main__":
    # Quick demo: align our partitions against UNIFAC for the case-study trio.
    # Requires the Tox21-trained model to be loaded first (cell 8 / DEMO_SMILES).
    _LOAD = "outputs/dumplinggnn_tox21_8dim.pth"
    if os.path.exists(_LOAD):
        _m = DumplingGNN(input_dim=8, hidden_channels=64, dropout=0.1)
        _m.to(CFG.DEVICE if "CFG" in globals() else torch.device("cpu"))
        _m.load_state_dict(torch.load(_LOAD, map_location=_m.device if hasattr(_m, "device") else "cpu"))
        _m.eval()
        _smis = [smi for _name, smi in (DEMO_SMILES if "DEMO_SMILES" in globals() else [])]
        if _smis:
            evaluate_ugropy_alignment(_m, _smis, gc_model="unifac")
    else:
        print(f"ugropy demo skipped: pre-trained model not found at {_LOAD}.")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# §0b  TOX21 SR-MMP TRAINING (real toxicity model used by the case studies)
#
# Paper §5.1: "We train on the Tox21 SR-MMP benchmark (test AUROC = 0.801)
# for qualitative analysis only." This cell loads the SR-MMP assay column of
# the standard MoleculeNet Tox21 dataset, drops rows whose SR-MMP label is NaN,
# trains a DumplingGNN, and saves it to outputs/dumplinggnn_tox21_8dim.pth so the
# downstream demo (next cell) can explain real toxic compounds.
#
# Note: modern SOTA on SR-MMP is ≈0.94 (DeepTox); 0.801 is a plausible AUROC
# for an off-the-shelf small GNN and is used here because the focus of §5.5
# is qualitative interpretability rather than predictive performance.
# ─────────────────────────────────────────────────────────────────────────────

# SR-MMP is column index 10 of the 12-task Tox21 multi-label vector. Standard
# MoleculeNet order: NR-AR, NR-AR-LBD, NR-AhR, NR-Aromatase, NR-ER, NR-ER-LBD,
# NR-PPAR-gamma, SR-ARE, SR-ATAD5, SR-HSE, SR-MMP, SR-p53.
SR_MMP_INDEX = 10


def load_tox21_srmmp(root: str = "outputs/dig_data") -> "Tuple[List[Data], List[float]]":
    """Load the SR-MMP subset of MoleculeNet's Tox21 dataset.

    Drops rows whose SR-MMP label is NaN (the SR-MMP column is sparsely
    annotated). Returns a list of PyG Data objects with `.y` set to the
    binary SR-MMP label, and the corresponding label list (useful for
    pos_weight computation).
    """
    from torch_geometric.datasets import MoleculeNet
    raw = MoleculeNet(root, name="Tox21")
    filtered: List[Data] = []
    labels: List[float] = []
    for d in raw:
        if d.x is None or d.edge_index is None or d.y is None:
            continue
        y = d.y.view(-1)
        if SR_MMP_INDEX >= y.numel():
            continue
        lbl = y[SR_MMP_INDEX].item()
        if math.isnan(lbl):
            continue
        g = d.clone()
        if g.x.dtype != torch.float:
            g.x = g.x.float()
        g.y = torch.tensor([float(lbl)], dtype=torch.float32)
        filtered.append(g)
        labels.append(float(lbl))
    return filtered, labels


def train_tox21_srmmp(
    data_list: List[Data],
    labels: List[float],
    *,
    save_path: str = "outputs/dumplinggnn_tox21_8dim.pth",
    epochs: int = 300,
    lr: float = 1e-3,
    batch_size: int = 64,
    patience: int = 30,
    train_frac: float = 0.80,
    val_frac: float = 0.10,
    seed: int = 42,
    device: Optional[torch.device] = None,
) -> "Tuple[nn.Module, Dict[str, float]]":
    """Train DumplingGNN on the SR-MMP subset of Tox21.

    Random 80/10/10 split (Tox21 has no canonical scaffold split in MoleculeNet's
    loader; production work should use scaffold split — see MoleculeNet paper).
    Uses BCEWithLogitsLoss with pos_weight = n_neg / n_pos, ReduceLROnPlateau,
    early stopping on validation AUROC. Saves the best-val state_dict to
    `save_path` and returns (model, metrics_dict) with the held-out test AUROC.
    """
    if device is None:
        # Use CFG.DEVICE if available, otherwise auto-detect. CFG lives later in
        # the notebook (legacy SMILES pipeline cell); the Tox21 cell must be
        # runnable without it.
        device = globals().get("CFG").DEVICE if "CFG" in globals() else torch.device(
            "cuda" if torch.cuda.is_available() else "cpu")

    # Determine input feature dimension from the data.
    feat_dim = int(data_list[0].x.size(1))

    rng = np.random.RandomState(seed)
    n = len(data_list)
    perm = rng.permutation(n)
    n_tr = int(n * train_frac)
    n_va = int(n * val_frac)
    train_data = [data_list[i] for i in perm[:n_tr]]
    val_data   = [data_list[i] for i in perm[n_tr:n_tr + n_va]]
    test_data  = [data_list[i] for i in perm[n_tr + n_va:]]

    print(f"  Tox21 SR-MMP split: train={len(train_data)} val={len(val_data)} "
          f"test={len(test_data)} feat_dim={feat_dim}")

    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_data,   batch_size=batch_size)
    test_loader  = DataLoader(test_data,  batch_size=batch_size)

    # Match the canonical DumplingGNN hyperparameters used throughout the
    # paper (CFG.HIDDEN = 64, CFG.DROPOUT = 0.1 in the legacy pipeline cell).
    _HIDDEN = globals().get("CFG").HIDDEN if "CFG" in globals() else 64
    _DROPOUT = globals().get("CFG").DROPOUT if "CFG" in globals() else 0.1
    model = DumplingGNN(input_dim=feat_dim, hidden_channels=_HIDDEN,
                        dropout=_DROPOUT).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)

    # Class-ratio weighting on the training set.
    tr_labels = [labels[i] for i in perm[:n_tr]]
    n_pos = sum(1 for l in tr_labels if l > 0.5)
    n_neg = len(tr_labels) - n_pos
    pw = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32, device=device)
    print(f"  pos_weight = {pw.item():.2f}  (n_pos={n_pos}, n_neg={n_neg})")
    crit = nn.BCEWithLogitsLoss(pos_weight=pw)

    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, "max", patience=8, factor=0.5, min_lr=1e-5)

    best_auc, best_state, wait = 0.0, None, 0
    for epoch in range(1, epochs + 1):
        model.train()
        tl = 0.0
        for b in train_loader:
            b = b.to(device); opt.zero_grad()
            loss = crit(model(b), b.y.view(-1)); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            tl += loss.item() * b.num_graphs
        al = tl / len(train_data)

        model.eval(); yy, pp = [], []
        with torch.no_grad():
            for b in val_loader:
                b = b.to(device)
                pp.extend(torch.sigmoid(model(b)).cpu().tolist())
                yy.extend(b.y.view(-1).cpu().tolist())
        va = roc_auc_score(yy, pp) if len(set(yy)) > 1 else 0.5
        sched.step(va)
        if va > best_auc:
            best_auc, best_state, wait = va, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
        if epoch % 10 == 0 or epoch == 1:
            cur_lr = opt.param_groups[0]["lr"]
            print(f"  [tox21] Ep {epoch:3d}  loss={al:.4f}  val_auc={va:.4f}  "
                  f"best={best_auc:.4f}  lr={cur_lr:.1e}")
        if wait >= patience:
            print(f"  [tox21] Early stop epoch {epoch}")
            break

    assert best_state is not None
    model.load_state_dict(best_state)

    # Test-set AUROC
    model.eval(); yy, pp = [], []
    with torch.no_grad():
        for b in test_loader:
            b = b.to(device)
            pp.extend(torch.sigmoid(model(b)).cpu().tolist())
            yy.extend(b.y.view(-1).cpu().tolist())
    test_auc = roc_auc_score(yy, pp) if len(set(yy)) > 1 else 0.5
    print(f"  [tox21] Best val AUROC: {best_auc:.4f} | Test AUROC: {test_auc:.4f}")

    # Save for downstream demo to load.
    os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)
    torch.save(best_state, save_path)
    print(f"  [tox21] Saved model to {save_path}")

    return model, {"val_auroc": best_auc, "test_auroc": float(test_auc)}


if __name__ == "__main__":
    # Skip this training if a Tox21-trained model already exists at the canonical
    # path. Delete or rename the file to force a re-train.
    _TOX21_SAVE = "outputs/dumplinggnn_tox21_8dim.pth"
    if os.path.exists(_TOX21_SAVE):
        print(f"Tox21 model already at {_TOX21_SAVE}; skipping training. "
              f"Delete the file to retrain.")
    else:
        try:
            _tox_data, _tox_labels = load_tox21_srmmp()
            print(f"Tox21 SR-MMP loaded: {len(_tox_data)} graphs "
                  f"(positives: {sum(1 for l in _tox_labels if l > 0.5)})")
            train_tox21_srmmp(_tox_data, _tox_labels, save_path=_TOX21_SAVE)
        except Exception as _e:
            print(f"Tox21 SR-MMP training skipped: {_e}")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# DEMO: run the explainer on a few well-known toxic compounds.
# Requires outputs/dumplinggnn_tox21_8dim.pth (model trained by main() below).
# Comment out or skip when running main() from scratch.
# ─────────────────────────────────────────────────────────────────────────────

# Paper §5.5 presents three case studies: acetaminophen, 2,4-dinitrophenol,
# cyclophosphamide. Two extra demos (paraquat dichloride, terfenadine) were
# previously included but are not part of the paper; they have been removed
# to keep the demo output aligned with the manuscript.
DEMO_SMILES = [
    ("acetaminophen",     "CC(=O)Nc1ccc(cc1)O"),
    ("2,4-dinitrophenol", "Oc1ccc(cc1[N+]([O-])=O)[N+]([O-])=O"),
    ("cyclophosphamide",  "ClCCN(CCCl)P1(=O)NCCCO1"),
]

if __name__ == "__main__":
    _LOAD_PATH = "outputs/dumplinggnn_tox21_8dim.pth"
    _DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    _demo_model = DumplingGNN(input_dim=8, hidden_channels=64, dropout=0.1)
    _demo_model.to(_DEVICE)
    if os.path.exists(_LOAD_PATH):
        _demo_model.load_state_dict(torch.load(_LOAD_PATH, map_location=_DEVICE))
        _demo_model.eval()
        print(f"Demo model loaded on {_DEVICE}")
        for _name, _smi in DEMO_SMILES:
            print(f"\n--- {_name} ---")
            _result = explain_molecule(
                _demo_model, _smi, _DEVICE,
                n_samples_st2=100, n_samples_sv=100,
                sa_iterations=300,
                verbose=True,
            )
            print_explanation(_result)
            # Save a PDF figure per molecule.
            _fig_path = f"outputs/case_study_{_name.replace(' ', '_').replace(',', '')}.pdf"
            try:
                os.makedirs(os.path.dirname(_fig_path), exist_ok=True)
                save_case_study_figure(_result, _name, _fig_path)
            except Exception as _exc:
                print(f"  [{_name}] figure save failed: {_exc}")
    else:
        print(f"Demo skipped: pre-trained model not found at {_LOAD_PATH}. "
              f"Run main() first to train and save the model.")

class CFG:
    SEED              = 42
    DEVICE            = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # dataset
    N_TOTAL_PER_TASK  = 8000
    LABEL_NOISE_RATE  = 0.03
    DIST_THRESH       = 3           # Task 3: close ≤ 3, far > 3 (achievable on linked scaffolds)

    # label integrity
    RELABEL_MODE      = "overwrite"  # "overwrite" or "drop"

    # scaffold
    MURCKO_SPLIT      = True        # True = scaffold-disjoint train/val/test
    SCAFFOLD_BALANCE  = True        # True = downsample to equalize motif combos per scaffold

    # training
    BATCH_SIZE        = 128
    EPOCHS            = 40
    LR                = 1e-3
    HIDDEN            = 64
    DROPOUT           = 0.1
    PATIENCE          = 10

    # explanation
    N_EXPLAIN         = 30
    N_SAMPLES_ST2_LIST = [50, 100]
    N_SAMPLES_SV      = 50
    SA_ITERATIONS     = 100
    MAX_HEAVY_EXPLAIN = 18

    # deletion faithfulness
    DEL_ENABLED       = True        # run for every explained molecule
    DEL_N_RANDOM      = 20          # random orderings for baseline (was 10, increased)
    DEL_BOOTSTRAP_N   = 1000        # bootstrap resamples for 95% CI
    INSERTION_ENABLED = True        # run insertion curve (complement of deletion)

    # baselines
    GRAD_BASELINE     = True        # run Grad×Input baseline
    BASELINE_N_EXPLAIN = 30         # how many molecules for baseline eval (match N_EXPLAIN)

    # artifact diagnostics
    ARTIFACT_CHECK    = True        # logistic regression on nuisance features

    # stability
    STABILITY_SEEDS   = [42, 123, 777]   # repeat explanations with these seeds
    STABILITY_N       = 10                # molecules per seed for stability test

    # main effects tests
    MAIN_EFFECTS_PERM_N = 20             # permutation shuffles for logreg sanity

    # counterfactual experiment (DIST task)
    CF_N              = 30               # molecules for counterfactual pairs
    CF_MAX_ADD        = 5                # max heavy atoms to add for CF_break
    CF_MAX_ATTEMPTS   = 50               # attempts per CF generation

    # output
    OUT_DIR           = "outputs"
    DATA_DIR          = "outputs"

os.makedirs(CFG.OUT_DIR, exist_ok=True)
os.makedirs(CFG.DATA_DIR, exist_ok=True)

def set_seed(seed=CFG.SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed()
print(f"Device: {CFG.DEVICE}")

def save_config():
    """Dump config as JSON for reproducibility."""
    cfg = {k: v for k, v in vars(CFG).items()
           if not k.startswith("_") and k != "DEVICE"}
    cfg["DEVICE"] = str(CFG.DEVICE)
    path = os.path.join(CFG.DATA_DIR, "config.json")
    with open(path, "w") as f:
        json.dump(cfg, f, indent=2, default=str)
    print(f"  Config saved to {path}")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# §1  MOTIF DEFINITIONS & DETECTION
# ═══════════════════════════════════════════════════════════════════════════

MOTIF_A_SMARTS = "c[N+](=O)[O-]"   # nitroaromatic
MOTIF_B_SMARTS = "C#N"              # nitrile

def has_motif(mol, smarts):
    if mol is None:
        return False
    pat = Chem.MolFromSmarts(smarts)
    return mol.HasSubstructMatch(pat) if pat else False

def get_motif_atom_indices(mol, smarts):
    if mol is None:
        return set()
    pat = Chem.MolFromSmarts(smarts)
    if pat is None:
        return set()
    indices = set()
    for match in mol.GetSubstructMatches(pat):
        for idx in match:
            if mol.GetAtomWithIdx(idx).GetAtomicNum() != 1:
                indices.add(idx)
    return indices

def get_motif_heavy_positions(mol_graph, smarts):
    """Map motif atom indices to heavy-atom positional indices in mol_graph.

    SAFE approach: match on mol_graph.mol (the actual molecule), then convert
    atom indices → heavy-atom positions via mol_graph.heavy_idx lookup.
    Never uses RemoveHs which can reindex atoms.
    """
    mol = mol_graph.mol
    raw = get_motif_atom_indices(mol, smarts)
    if not raw:
        return set()
    # Build global→position map for heavy atoms only
    heavy_set = set(mol_graph.heavy_idx)
    g2p = {g: p for p, g in enumerate(mol_graph.heavy_idx)}
    return {g2p[i] for i in raw if i in heavy_set}

def compute_motif_distance(mol, smarts_A, smarts_B):
    atoms_A = get_motif_atom_indices(mol, smarts_A)
    atoms_B = get_motif_atom_indices(mol, smarts_B)
    if not atoms_A or not atoms_B:
        return -1
    try:
        dm = Chem.GetDistanceMatrix(mol)
    except Exception:
        return -1
    return int(min(dm[a][b] for a in atoms_A for b in atoms_B))

def get_murcko_scaffold(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return ""
    try:
        core = MurckoScaffold.GetScaffoldForMol(mol)
        return Chem.MolToSmiles(core)
    except Exception:
        return ""

def count_heavy_atoms(mol):
    if mol is None:
        return 0
    return sum(1 for a in mol.GetAtoms() if a.GetAtomicNum() != 1)


# ═══════════════════════════════════════════════════════════════════════════
# §2  LABEL INTEGRITY
# ═══════════════════════════════════════════════════════════════════════════

def recompute_labels(df, task_type, dist_thresh=3):
    """Recompute has_A, has_B, label from final SMILES via RDKit.

    Parameters
    ----------
    task_type : str  "AND", "OR", "DIST"
    dist_thresh : int  distance threshold for DIST

    Returns
    -------
    df_fixed, mismatch_log (list of dicts)
    """
    df = df.copy()
    mismatch_log = []

    new_has_A, new_has_B, new_dist, new_label = [], [], [], []
    for _, row in df.iterrows():
        smi = row["smiles"]
        mol = Chem.MolFromSmiles(smi)

        a = int(has_motif(mol, MOTIF_A_SMARTS))
        b = int(has_motif(mol, MOTIF_B_SMARTS))
        d = compute_motif_distance(mol, MOTIF_A_SMARTS, MOTIF_B_SMARTS) if (a and b) else -1

        if task_type == "AND":
            lab = int(a == 1 and b == 1)
        elif task_type == "OR":
            lab = int(a == 1 or b == 1)
        elif task_type == "DIST":
            lab = int(a == 1 and b == 1 and 0 <= d <= dist_thresh)
        else:
            lab = int(row["label"])

        new_has_A.append(a)
        new_has_B.append(b)
        new_dist.append(d)
        new_label.append(lab)

    df["has_A_recomp"] = new_has_A
    df["has_B_recomp"] = new_has_B
    df["dist_AB_recomp"] = new_dist
    df["label_recomp"] = new_label

    # Compare with existing
    for i, row in df.iterrows():
        old_lab = row.get("label", -1)
        new_lab = row["label_recomp"]
        if int(old_lab) != int(new_lab):
            mismatch_log.append({
                "smiles": row["smiles"],
                "task": task_type,
                "old_label": int(old_lab),
                "new_label": int(new_lab),
                "has_A": row["has_A_recomp"],
                "has_B": row["has_B_recomp"],
                "dist_AB": row["dist_AB_recomp"],
                "scaffold": row.get("scaffold", ""),
            })

    n_mismatch = len(mismatch_log)

    if CFG.RELABEL_MODE == "overwrite":
        df["label_was_recomputed"] = (df["label"].astype(int) != df["label_recomp"]).astype(int)
        df["label"] = df["label_recomp"]
        df["has_A"] = df["has_A_recomp"]
        df["has_B"] = df["has_B_recomp"]
        if "dist_AB" in df.columns or task_type == "DIST":
            df["dist_AB"] = df["dist_AB_recomp"]
        action = "overwritten"
    elif CFG.RELABEL_MODE == "drop":
        mask = df["label"].astype(int) == df["label_recomp"]
        n_drop = (~mask).sum()
        df = df[mask].reset_index(drop=True)
        action = f"dropped {n_drop}"
    else:
        action = "none"

    # Clean up temp columns
    df.drop(columns=["has_A_recomp", "has_B_recomp", "dist_AB_recomp", "label_recomp"],
            inplace=True, errors="ignore")

    print(f"  Label integrity ({task_type}): {n_mismatch} mismatches → {action}")
    return df, mismatch_log


# ═══════════════════════════════════════════════════════════════════════════
# §3  SYNTHETIC MOLECULE GENERATOR
# ═══════════════════════════════════════════════════════════════════════════

SCAFFOLDS = [
    # --- Simple monocyclic (6 atoms) ---
    "c1ccccc1",       # benzene
    "c1ccncc1",       # pyridine
    "c1ccncn1",       # pyrimidine
    "c1cnccn1",       # pyrazine
    "c1ccnnc1",       # pyridazine
    # --- Simple monocyclic (5 atoms) ---
    "c1ccoc1",        # furan
    "c1ccsc1",        # thiophene
    "c1cc[nH]c1",     # pyrrole
    # --- Fused bicyclic ---
    "c1ccc2ccccc2c1",     # naphthalene
    "c1ccc2ncccc2c1",     # quinoline
    "c1ccc2[nH]ccc2c1",  # indole
    "c1ccc2occc2c1",      # benzofuran
    "c1ccc2sccc2c1",      # benzothiophene
    "c1ccc2nccnc2c1",     # quinoxaline
    "c1cnc2ccccc2n1",     # quinazoline
    # --- Linked diaryl: enable motif distance > 5 ---
    "c1ccc(-c2ccccc2)cc1",           # biphenyl (dist~4-5)
    "c1ccc(Cc2ccccc2)cc1",           # diphenylmethane (dist~5-6)
    "c1ccc(CCc2ccccc2)cc1",          # diphenylethane (dist~6-7)
    "c1ccc(Oc2ccccc2)cc1",           # diphenylether (dist~4-5)
    "c1ccc(-c2ccccn2)cc1",           # 2-phenylpyridine (N-linked)
    "c1ccnc(-c2ccncc2)c1",           # 2,2'-bipyridine (N-linked)
    "c1ccc(Cc2ccncc2)cc1",           # phenylmethylpyridine (N-linked)
    "c1ccc(CCc2ccncc2)cc1",          # phenylethylpyridine (N-linked)
    "c1ccc(-c2ccsc2)cc1",            # 2-phenylthiophene
    "c1ccc(-c2ccoc2)cc1",            # 2-phenylfuran
    "c1ccc(Cc2ccsc2)cc1",            # phenylmethylthiophene
    "c1ccc(CCOc2ccccc2)cc1",         # phenoxyethylbenzene (dist~7)
    "c1ccnc(Cc2ccccc2)c1",           # pyridylmethylphenyl (N-linked)
    "c1ccnc(CCc2ccncc2)c1",          # bipyridylethane (2xN)
]

# Subsets for controlled DIST generation
SCAFFOLDS_SMALL = [s for s in SCAFFOLDS if s.count('c') + s.count('n') + s.count('o') + s.count('s') <= 10]
SCAFFOLDS_LINKED = [s for s in SCAFFOLDS if '(' in s and s.count('c') > 8]

# DIST-specific scaffolds: compact linked diaryl with DIRECT BOND.
# Only direct-bond links (no CH2) to ensure consistent distance semantics:
#   same-ring → d ≤ 3 (close), cross-ring → d ≥ 3-4 (close/far mix).
# Each entry is a UNIQUE molecule (no reversed duplicates).
# Heavy atoms: 10 (5+5), 11 (5+6), or 12 (6+6).
# With both motifs (nitro=3, nitrile=2): 15-17 heavy → fits MAX_HEAVY_EXPLAIN=18.
SCAFFOLDS_DIST = [
    # --- 5+5 direct bond (10 heavy atoms) ---
    "c1cc(-c2ccsc2)cs1",          # bithiophene
    "c1cc(-c2ccoc2)cs1",          # thienyl-furan
    "c1cc(-c2cc[nH]c2)cs1",      # thienyl-pyrrole
    "c1cc(-c2ccoc2)co1",          # bifuran
    "c1cc(-c2cc[nH]c2)co1",      # furanyl-pyrrole
    "c1cc(-c2cc[nH]c2)[nH]c1",   # bipyrrole
    # --- 5+6 direct bond (11 heavy atoms) ---
    "c1ccc(-c2ccsc2)cc1",         # phenyl-thiophene
    "c1ccc(-c2ccoc2)cc1",         # phenyl-furan
    "c1ccc(-c2cc[nH]c2)cc1",     # phenyl-pyrrole
    "c1ccnc(-c2ccsc2)c1",        # pyridyl-thiophene
    "c1ccnc(-c2ccoc2)c1",        # pyridyl-furan
    "c1ccnc(-c2cc[nH]c2)c1",     # pyridyl-pyrrole
    "c1ncc(-c2ccsc2)nc1",         # pyrimidyl-thiophene
    "c1ncc(-c2ccoc2)nc1",         # pyrimidyl-furan
    # --- 6+6 direct bond (12 heavy atoms) ---
    "c1ccc(-c2ccccc2)cc1",        # biphenyl
    "c1ccc(-c2ccccn2)cc1",        # phenyl-pyridine
    "c1ccnc(-c2ccncc2)c1",       # bipyridine
]

NOISE_FRAGMENTS_NEUTRAL = ["F", "Cl", "Br", "C", "CC", "CCC", "OC", "OCC", "O"]
NOISE_FRAGMENTS_N_CONTAINING = [
    "N", "NC", "N(C)C", "NC(C)=O", "NC(=O)C", "N=O", "[N+](C)(C)C",
]
ALL_NOISE = NOISE_FRAGMENTS_NEUTRAL + NOISE_FRAGMENTS_N_CONTAINING

def _safe_sanitize(mol):
    if mol is None: return None
    try:
        Chem.SanitizeMol(mol)
        return mol
    except Exception:
        return None

def attach_fragment(mol, frag, rng):
    rxn_smarts = f"[cH:1]>>[c:1]{frag}"
    try:
        rxn = AllChem.ReactionFromSmarts(rxn_smarts)
    except Exception:
        return None
    if rxn is None: return None
    try:
        products = rxn.RunReactants((mol,))
    except Exception:
        return None
    if not products: return None
    return _safe_sanitize(products[rng.randint(len(products))][0])

def attach_nitro(mol, rng):
    return attach_fragment(mol, "[N+](=O)[O-]", rng)

def attach_nitrile(mol, rng):
    result = attach_fragment(mol, "C#N", rng)
    if result: return result
    rxn = AllChem.ReactionFromSmarts("[CX4H:1]>>[C:1]C#N")
    if rxn is None: return None
    try:
        products = rxn.RunReactants((mol,))
    except Exception:
        return None
    if not products: return None
    return _safe_sanitize(products[rng.randint(len(products))][0])

def attach_noise(mol, rng, n=1):
    for _ in range(n):
        frag = ALL_NOISE[rng.randint(len(ALL_NOISE))]
        r = attach_fragment(mol, frag, rng)
        if r is not None: mol = r
    return mol

def insert_linker(mol, rng, min_len=3, max_len=5):
    linker = "C" * rng.randint(min_len, max_len + 1)
    r = attach_fragment(mol, linker, rng)
    return r if r else mol

def generate_molecule(scaffold_smi, add_A, add_B, rng,
                      n_noise_range=(0, 2),
                      far_attachment=False,
                      target_heavy_range=(8, 22)):
    mol = Chem.MolFromSmiles(scaffold_smi)
    if mol is None: return None, None, None

    # Adaptive noise: more fragments for fewer motifs → equalises n_heavy
    n_motifs = int(add_A) + int(add_B)
    noise_lo = n_noise_range[0] + (2 - n_motifs)  # 0-motif: +2, 1-motif: +1, 2-motif: +0
    noise_hi = n_noise_range[1] + (2 - n_motifs)
    # Bias toward N-containing noise for molecules without motifs (reduces N shortcut)
    n_noise = rng.randint(max(noise_lo, 0), noise_hi + 1)
    for _ in range(n_noise):
        if n_motifs < 2 and rng.random() < 0.5:
            frag = NOISE_FRAGMENTS_N_CONTAINING[rng.randint(len(NOISE_FRAGMENTS_N_CONTAINING))]
        else:
            frag = ALL_NOISE[rng.randint(len(ALL_NOISE))]
        r = attach_fragment(mol, frag, rng)
        if r is not None:
            mol = r

    if add_A and add_B and far_attachment:
        # For far attachment, add motifs hoping they land on different parts
        # (The caller should already be using a LINKED scaffold)
        mol = attach_nitro(mol, rng)
        if mol is None: return None, None, None
        mol = attach_nitrile(mol, rng)
        if mol is None: return None, None, None
    else:
        if add_A:
            mol = attach_nitro(mol, rng)
            if mol is None: return None, None, None
        if add_B:
            mol = attach_nitrile(mol, rng)
            if mol is None: return None, None, None

    mol = _safe_sanitize(mol)
    if mol is None: return None, None, None

    smi = Chem.MolToSmiles(mol)
    mol_check = Chem.MolFromSmiles(smi)
    if mol_check is None: return None, None, None

    if add_A != has_motif(mol_check, MOTIF_A_SMARTS): return None, None, None
    if add_B != has_motif(mol_check, MOTIF_B_SMARTS): return None, None, None

    n_heavy = count_heavy_atoms(mol_check)
    if n_heavy < target_heavy_range[0] or n_heavy > target_heavy_range[1]:
        return None, None, None

    scaffold = get_murcko_scaffold(smi)
    return smi, mol_check, scaffold


def add_label_noise(df, rate, seed=42):
    rng = np.random.RandomState(seed)
    n_flip = int(len(df) * rate)
    idx = rng.choice(len(df), size=n_flip, replace=False)
    df = df.copy()
    df.loc[idx, "label"] = 1 - df.loc[idx, "label"]
    print(f"  Label noise: flipped {n_flip}/{len(df)} ({rate:.1%})")
    return df


def _gen_class(add_A, add_B, target, label, cname, rng, seen):
    records = []
    attempts = 0
    while len(records) < target and attempts < target * 80:
        attempts += 1
        scaff = SCAFFOLDS[rng.randint(len(SCAFFOLDS))]
        smi, mol, scaffold = generate_molecule(scaff, add_A, add_B, rng)
        if smi is None or smi in seen: continue
        seen.add(smi)
        records.append({
            "smiles": smi, "label": label,
            "has_A": int(add_A), "has_B": int(add_B),
            "struct_class": cname, "scaffold": scaffold,
            "n_heavy": count_heavy_atoms(mol),
        })
    print(f"  {cname}: {len(records)}/{target} (attempts={attempts})")
    return records


def balance_per_scaffold(df, seed=42):
    """Balance motif combos per scaffold AND drop shortcut scaffolds.

    Two-phase approach:
      Phase 1: Drop scaffolds that appear in only one label state (shortcuts).
      Phase 2: For each remaining scaffold with ≥2 motif combos, downsample
               to equalise combos. Use the *median* count (not min) to reduce
               data loss, then trim the largest combos to median.

    Returns balanced DataFrame (smaller than input).
    """
    if not CFG.SCAFFOLD_BALANCE:
        return df

    lab_col = "clean_label" if "clean_label" in df.columns else "label"
    rng = np.random.RandomState(seed)
    n_before = len(df)

    # Phase 1: Drop single-label scaffolds (leakage risk)
    scaff_labels = df.groupby("scaffold")[lab_col].nunique()
    multi_label = scaff_labels[scaff_labels >= 2].index
    single_label_scaffs = scaff_labels[scaff_labels < 2].index.tolist()
    n_single_dropped = df[df["scaffold"].isin(single_label_scaffs)].shape[0]
    df = df[df["scaffold"].isin(multi_label)].copy()
    if single_label_scaffs:
        print(f"  Scaffold filter: dropped {n_single_dropped} mols from "
              f"{len(single_label_scaffs)} single-label scaffolds")

    # Phase 2: Equalise motif combos within each scaffold
    keep_idx = []
    for scaff, grp in df.groupby("scaffold"):
        combo_groups = grp.groupby(["has_A", "has_B"])
        counts = combo_groups.size()

        if len(counts) < 2:
            keep_idx.extend(grp.index.tolist())
            continue

        # Use 25th percentile (not min) to reduce aggressive downsampling
        target = max(int(np.percentile(counts.values, 25)), counts.min())
        if target == 0:
            keep_idx.extend(grp.index.tolist())
            continue

        for combo_key, sub in combo_groups:
            if len(sub) <= target:
                keep_idx.extend(sub.index.tolist())
            else:
                sampled = sub.sample(n=target, random_state=rng)
                keep_idx.extend(sampled.index.tolist())

    df_bal = df.loc[keep_idx].reset_index(drop=True)
    n_dropped = n_before - len(df_bal)
    print(f"  Scaffold balance: {n_before} → {len(df_bal)} (dropped {n_dropped})")

    # Log per-scaffold distribution
    combos_per_scaff = df_bal.groupby("scaffold").apply(
        lambda g: g.groupby(["has_A", "has_B"]).size().to_dict()
    )
    n_balanced = sum(1 for v in combos_per_scaff.values if len(v) >= 2)
    print(f"  Scaffolds with ≥2 combos (balanced): {n_balanced}/{df_bal['scaffold'].nunique()}")

    return df_bal


def generate_dataset_task1(N=8000, seed=42, noise_rate=0.03):
    rng = np.random.RandomState(seed)
    n_pos = N // 2
    n_neg_per = N // 6
    seen = set()
    recs = []
    recs += _gen_class(True,  True,  n_pos,     1, "A_and_B", rng, seen)
    recs += _gen_class(False, False, n_neg_per, 0, "none",    rng, seen)
    recs += _gen_class(True,  False, n_neg_per, 0, "A_only",  rng, seen)
    recs += _gen_class(False, True,  n_neg_per, 0, "B_only",  rng, seen)
    remaining = N - len(recs)
    if remaining > 0:
        recs += _gen_class(False, False, remaining, 0, "none_extra", rng, seen)

    df = pd.DataFrame(recs).sample(frac=1, random_state=seed).reset_index(drop=True)
    df = balance_per_scaffold(df, seed=seed)

    # ORDERING IS CRITICAL:
    # 1) Integrity check FIRST — fix any structural mismatches from sanitization
    # 2) Label noise SECOND — deliberately corrupt some labels for regularization
    # If reversed, recompute_labels undoes all noise → defeats the purpose.
    df, mismatch = recompute_labels(df, "AND")
    df["clean_label"] = df["label"].copy()       # ground truth BEFORE noise
    df = add_label_noise(df, noise_rate, seed=seed + 100)
    print(f"  Task1 final: {len(df)}, labels: {df['label'].value_counts().to_dict()}")
    return df, mismatch


def generate_dataset_task2(N=8000, seed=43, noise_rate=0.03):
    rng = np.random.RandomState(seed)
    n_neg = N // 2
    n_pos_per = N // 6
    seen = set()
    recs = []
    recs += _gen_class(False, False, n_neg,     0, "none",    rng, seen)
    recs += _gen_class(True,  False, n_pos_per, 1, "A_only",  rng, seen)
    recs += _gen_class(False, True,  n_pos_per, 1, "B_only",  rng, seen)
    recs += _gen_class(True,  True,  n_pos_per, 1, "A_and_B", rng, seen)
    remaining = N - len(recs)
    if remaining > 0:
        recs += _gen_class(True, False, remaining, 1, "A_only_extra", rng, seen)

    df = pd.DataFrame(recs).sample(frac=1, random_state=seed).reset_index(drop=True)
    df = balance_per_scaffold(df, seed=seed)
    df, mismatch = recompute_labels(df, "OR")
    df["clean_label"] = df["label"].copy()
    df = add_label_noise(df, noise_rate, seed=seed + 100)
    print(f"  Task2 final: {len(df)}, labels: {df['label'].value_counts().to_dict()}")
    return df, mismatch


def generate_dataset_task3(N=8000, seed=44, noise_rate=0.03, dist_thresh=3):
    """DIST task: label=1 iff both motifs present AND close (dist ≤ thresh).

    ALL molecules generated from SCAFFOLDS_DIST (compact linked scaffolds).
    This ensures:
      - No scaffold→label shortcuts (all scaffolds host all struct_classes)
      - No size artifact (same scaffold pool for close vs far)
      - Molecules fit MAX_HEAVY_EXPLAIN (≤16 heavy atoms)
    """
    rng = np.random.RandomState(seed)
    seen = set()
    records = []
    dist_scaffolds = SCAFFOLDS_DIST if SCAFFOLDS_DIST else SCAFFOLDS_LINKED

    # Compact generation parameters for DIST
    # Higher noise range so adaptive compensation (2-n_motifs extra) can fully
    # offset the 5 heavy-atom gap between AB (nitro+nitrile) and none.
    dist_noise = (1, 3)
    dist_heavy = (10, CFG.MAX_HEAVY_EXPLAIN)  # fit within explanation budget

    # ── Phase 1: Generate AB pool — both close and far from same scaffolds ─
    ab_target = N // 2
    ab_pool_close, ab_pool_far = [], []
    att = 0
    while (len(ab_pool_close) + len(ab_pool_far)) < ab_target * 2 and att < ab_target * 100:
        att += 1
        scaff = dist_scaffolds[rng.randint(len(dist_scaffolds))]
        smi, mol, scaffold = generate_molecule(
            scaff, True, True, rng, n_noise_range=dist_noise,
            target_heavy_range=dist_heavy)
        if smi is None or smi in seen: continue
        d = compute_motif_distance(mol, MOTIF_A_SMARTS, MOTIF_B_SMARTS)
        if d < 0: continue
        seen.add(smi)
        rec = {"smiles": smi, "has_A": 1, "has_B": 1, "dist_AB": d,
               "scaffold": scaffold, "n_heavy": count_heavy_atoms(mol)}
        if d <= dist_thresh:
            ab_pool_close.append(rec)
        else:
            ab_pool_far.append(rec)

    # Balance close:far — target at most 2:1 ratio in either direction
    n_close, n_far = len(ab_pool_close), len(ab_pool_far)
    if n_close > n_far * 2:
        idx = rng.choice(n_close, size=n_far * 2, replace=False)
        ab_pool_close = [ab_pool_close[i] for i in sorted(idx)]
    elif n_far > n_close * 2:
        idx = rng.choice(n_far, size=n_close * 2, replace=False)
        ab_pool_far = [ab_pool_far[i] for i in sorted(idx)]

    for r in ab_pool_close:
        r["struct_class"] = "AB_close"
        r["label"] = 1
    for r in ab_pool_far:
        r["struct_class"] = "AB_far"
        r["label"] = 0

    records.extend(ab_pool_close)
    records.extend(ab_pool_far)
    n_ab = len(records)
    print(f"  Task3 AB_close: {len(ab_pool_close)}  AB_far: {len(ab_pool_far)}  "
          f"(compact linked pool)")

    # ── Phase 2: Non-AB negatives — SAME scaffold pool ────────────────────
    # Size each non-AB class to ~N//8 (~1000). This keeps them comparable to
    # expected post-balance AB_close count, preventing extreme class imbalance.
    # (AB_close typically loses ~60% in scaffold balance; non-AB loses ~5%)
    n_neg_per = N // 8

    def _neg_dist(aA, aB, target, cname):
        c2, att2 = 0, 0
        while c2 < target and att2 < target * 100:
            att2 += 1
            scaff = dist_scaffolds[rng.randint(len(dist_scaffolds))]
            smi, mol, scaffold = generate_molecule(
                scaff, aA, aB, rng, n_noise_range=dist_noise,
                target_heavy_range=dist_heavy)
            if smi is None or smi in seen: continue
            d = -1
            if aA and aB:
                d = compute_motif_distance(mol, MOTIF_A_SMARTS, MOTIF_B_SMARTS)
            seen.add(smi)
            records.append({"smiles": smi, "label": 0, "has_A": int(aA), "has_B": int(aB),
                            "struct_class": cname, "dist_AB": d,
                            "scaffold": scaffold, "n_heavy": count_heavy_atoms(mol)})
            c2 += 1
        print(f"  Task3 {cname}: {c2}/{target}")

    _neg_dist(False, False, n_neg_per, "none")
    _neg_dist(True,  False, n_neg_per, "A_only")
    _neg_dist(False, True,  n_neg_per, "B_only")

    # Post-generation checks
    df_pre = pd.DataFrame(records)
    n_ab_far = (df_pre["struct_class"] == "AB_far").sum()
    n_ab_close = (df_pre["struct_class"] == "AB_close").sum()
    print(f"  Pre-balance: AB_close={n_ab_close}  AB_far={n_ab_far}")
    if n_ab_far < 100:
        print(f"  ⚠ WARNING: Only {n_ab_far} AB_far molecules generated!")

    df = df_pre.sample(frac=1, random_state=seed).reset_index(drop=True)
    df = balance_per_scaffold(df, seed=seed)

    # Verify AB_far survived balancing
    n_ab_far_post = (df["struct_class"] == "AB_far").sum()
    print(f"  Post-balance: AB_far={n_ab_far_post}")

    df, mismatch = recompute_labels(df, "DIST", dist_thresh)
    df["clean_label"] = df["label"].copy()
    df = add_label_noise(df, noise_rate, seed=seed + 100)
    print(f"  Task3 final: {len(df)}, labels: {df['label'].value_counts().to_dict()}")
    return df, mismatch


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# §4  DATASET DIAGNOSTICS & INTEGRITY REPORT
# ═══════════════════════════════════════════════════════════════════════════

def dataset_diagnostics(df, task_name):
    print(f"\n  ── DIAGNOSTICS: {task_name} ──")
    print(f"  Total: {len(df)}  Labels (noisy): {df['label'].value_counts().to_dict()}")
    if "clean_label" in df.columns:
        n_flipped = (df["label"] != df["clean_label"]).sum()
        print(f"  Labels (clean):  {df['clean_label'].value_counts().to_dict()}")
        print(f"  Noise-flipped:   {n_flipped} ({100*n_flipped/len(df):.1f}%)")
    if "struct_class" in df.columns:
        print(f"  Classes: {df['struct_class'].value_counts().to_dict()}")
    for lab in [0, 1]:
        sub = df[df["label"] == lab]
        if "n_heavy" in sub.columns and len(sub):
            print(f"  Label {lab}: n_heavy mean={sub['n_heavy'].mean():.1f} std={sub['n_heavy'].std():.1f}")
    if "scaffold" in df.columns:
        print(f"  Unique scaffolds: {df['scaffold'].nunique()}")
    # Nitrogen overlap check
    for lab in [0, 1]:
        sub = df[df["label"] == lab]
        n_with_N = sum(1 for s in sub["smiles"]
                       if Chem.MolFromSmiles(s) is not None and
                       any(a.GetAtomicNum() == 7 for a in Chem.MolFromSmiles(s).GetAtoms()))
        print(f"  Label {lab}: {n_with_N}/{len(sub)} ({100*n_with_N/max(len(sub),1):.0f}%) have nitrogen")


def build_integrity_report(task_dfs, mismatch_logs):
    """Build dataset_integrity_report.csv"""
    task_map = {"task1_AND": "AND", "task2_OR": "OR", "task3_DIST": "DIST"}
    rows = []
    for task_name, df in task_dfs.items():
        wanted = task_map.get(task_name, task_name)
        n_mm = sum(1 for m in mismatch_logs if m.get("task") == wanted)
        # Count (has_A, has_B, label) combos
        for (a, b, l), grp in df.groupby(["has_A", "has_B", "label"]):
            rows.append({
                "task": task_name, "has_A": a, "has_B": b, "label": l,
                "count": len(grp), "mismatches_total": n_mm,
            })
        # DIST distance breakdown
        if "dist_AB" in df.columns:
            both = df[(df["has_A"] == 1) & (df["has_B"] == 1) & (df["dist_AB"] >= 0)]
            for lab in [0, 1]:
                sub = both[both["label"] == lab]
                if len(sub):
                    rows.append({
                        "task": task_name, "has_A": "AB", "has_B": "dist_stats",
                        "label": lab,
                        "count": len(sub),
                        "mismatches_total": f"dist mean={sub['dist_AB'].mean():.1f} med={sub['dist_AB'].median():.0f}",
                    })
    return pd.DataFrame(rows)


def scaffold_leakage_report(task_dfs):
    """Report scaffold-label-state distribution to detect leakage.

    For each scaffold, counts how many distinct (has_A, has_B, label) states exist.
    Scaffolds with only one label state can be shortcuts.

    Returns DataFrame with per-scaffold stats and a printed summary.
    """
    all_rows = []
    for task_name, df in task_dfs.items():
        if "scaffold" not in df.columns:
            continue
        lab_col = "clean_label" if "clean_label" in df.columns else "label"

        scaff_stats = df.groupby("scaffold").apply(
            lambda g: pd.Series({
                "n_mols": len(g),
                "n_label_states": g[lab_col].nunique(),
                "n_motif_combos": g.groupby(["has_A", "has_B"]).ngroups,
                "label_mean": g[lab_col].mean(),
            })
        ).reset_index()
        scaff_stats["task"] = task_name

        n_total = len(scaff_stats)
        n_single_label = (scaff_stats["n_label_states"] == 1).sum()
        mols_in_single = scaff_stats.loc[scaff_stats["n_label_states"] == 1, "n_mols"].sum()
        n_single_combo = (scaff_stats["n_motif_combos"] == 1).sum()

        print(f"\n  [SCAFFOLD LEAKAGE] {task_name}")
        print(f"    Total scaffolds: {n_total}")
        print(f"    Single-label scaffolds: {n_single_label} ({n_single_label/max(n_total,1):.1%})"
              f"  covering {mols_in_single} molecules")
        print(f"    Single-combo scaffolds: {n_single_combo} ({n_single_combo/max(n_total,1):.1%})")

        if n_single_label / max(n_total, 1) > 0.3:
            print(f"    ⚠ HIGH LEAKAGE RISK: >30% of scaffolds are single-label")
        else:
            print(f"    ✓ Leakage appears controlled")

        all_rows.append(scaff_stats)

    return pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()


# ═══════════════════════════════════════════════════════════════════════════
# §5  SMILES → PyG Data
# ═══════════════════════════════════════════════════════════════════════════

_HYB_MAP = {
    Chem.rdchem.HybridizationType.S: 0,
    Chem.rdchem.HybridizationType.SP: 1,
    Chem.rdchem.HybridizationType.SP2: 2,
    Chem.rdchem.HybridizationType.SP3: 3,
    Chem.rdchem.HybridizationType.SP3D: 4,
    Chem.rdchem.HybridizationType.SP3D2: 5,
}

def smiles_to_pyg_data(smiles, label=None):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    feats = []
    for atom in mol.GetAtoms():
        hyb = _HYB_MAP.get(atom.GetHybridization(), 6)
        feats.append([atom.GetAtomicNum(), atom.GetDegree(), atom.GetTotalNumHs(),
                      atom.GetTotalValence(), int(atom.GetIsAromatic()),
                      atom.GetFormalCharge(), hyb, int(atom.IsInRing())])
    x = torch.tensor(feats, dtype=torch.float32)
    edge_list = []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        edge_list += [[i, j], [j, i]]
    ei = torch.tensor(edge_list, dtype=torch.long).t().contiguous() if edge_list else torch.zeros((2, 0), dtype=torch.long)
    d = Data(x=x, edge_index=ei)
    if label is not None:
        d.y = torch.tensor([label], dtype=torch.float32)
    return d


# §6 MODEL — canonical MPNN / DumplingGNN definitions live earlier in this file.
# (The duplicate class block that used to live here has been removed.)
# ═══════════════════════════════════════════════════════════════════════════
# §7  TRAINING & EVALUATION
# ═══════════════════════════════════════════════════════════════════════════

def build_pyg_dataset(df):
    data_list, valid_idx = [], []
    for i, row in df.iterrows():
        d = smiles_to_pyg_data(row["smiles"], row["label"])
        if d is not None:
            data_list.append(d); valid_idx.append(i)
    return data_list, valid_idx

def stratified_split(df, data_list, valid_idx, train_f=0.70, val_f=0.15, seed=42):
    rng = np.random.RandomState(seed)
    idx_map = {vi: pos for pos, vi in enumerate(valid_idx)}
    strat_col = "struct_class" if "struct_class" in df.columns else "label"

    if CFG.MURCKO_SPLIT and "scaffold" in df.columns:
        # Scaffold-disjoint split
        scaffolds = df["scaffold"].unique().tolist()
        rng.shuffle(scaffolds)
        n_s = len(scaffolds)
        s_train = set(scaffolds[:int(n_s * train_f)])
        s_val = set(scaffolds[int(n_s * train_f):int(n_s * (train_f + val_f))])
        s_test = set(scaffolds[int(n_s * (train_f + val_f)):])

        splits = {"train": ([], []), "val": ([], []), "test": ([], [])}
        for r in df.index:
            if r not in idx_map: continue
            sc = df.loc[r, "scaffold"]
            if sc in s_train: key = "train"
            elif sc in s_val: key = "val"
            else: key = "test"
            splits[key][0].append(data_list[idx_map[r]])
            splits[key][1].append(r)

        overlap = s_train & s_test
        print(f"  Scaffold split: train_scaff={len(s_train)} val={len(s_val)} test={len(s_test)} overlap={len(overlap)}")
        return {f"{k}": splits[k][0] for k in splits} | {f"{k}_rows": splits[k][1] for k in splits}

    # Stratified by struct_class
    result = {k: [] for k in ["train", "val", "test", "train_rows", "val_rows", "test_rows"]}
    for cls_val in df[strat_col].unique():
        cls_valid = [r for r in df[df[strat_col] == cls_val].index if r in idx_map]
        rng.shuffle(cls_valid)
        n = len(cls_valid)
        n_tr, n_va = int(n * train_f), int(n * val_f)
        for r in cls_valid[:n_tr]:
            result["train"].append(data_list[idx_map[r]]); result["train_rows"].append(r)
        for r in cls_valid[n_tr:n_tr + n_va]:
            result["val"].append(data_list[idx_map[r]]); result["val_rows"].append(r)
        for r in cls_valid[n_tr + n_va:]:
            result["test"].append(data_list[idx_map[r]]); result["test_rows"].append(r)
    return result

def train_model(train_data, val_data, task_name, epochs=40, lr=1e-3,
                batch_size=128, patience=10, device=None):
    if device is None: device = CFG.DEVICE
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=batch_size)
    model = DumplingGNN(input_dim=8, hidden_channels=CFG.HIDDEN, dropout=CFG.DROPOUT).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    crit = nn.BCEWithLogitsLoss()
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, "max", patience=4, factor=0.5)
    best_auc, best_state, wait = 0.0, None, 0
    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        tl = 0
        for b in train_loader:
            b = b.to(device); opt.zero_grad()
            loss = crit(model(b), b.y.view(-1)); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            tl += loss.item() * b.num_graphs
        al = tl / len(train_data)
        model.eval()
        yy, pp = [], []
        with torch.no_grad():
            for b in val_loader:
                b = b.to(device)
                pp.extend(torch.sigmoid(model(b)).cpu().tolist())
                yy.extend(b.y.view(-1).cpu().tolist())
        va = roc_auc_score(yy, pp) if len(set(yy)) > 1 else 0.5
        sched.step(va)
        history.append({"epoch": epoch, "train_loss": al, "val_auc": va})
        if va > best_auc: best_auc = va; best_state = copy.deepcopy(model.state_dict()); wait = 0
        else: wait += 1
        if epoch % 5 == 0 or epoch == 1:
            print(f"  [{task_name}] Ep {epoch:3d}  loss={al:.4f}  val_auc={va:.4f}  best={best_auc:.4f}")
        if wait >= patience:
            print(f"  [{task_name}] Early stop ep {epoch}"); break
    model.load_state_dict(best_state)
    p = os.path.join(CFG.DATA_DIR, f"dumplinggnn_{task_name}.pth")
    torch.save(best_state, p)
    print(f"  [{task_name}] Best val AUC: {best_auc:.4f}")
    return model, history

def evaluate_model(model, test_data, device=None):
    if device is None: device = CFG.DEVICE
    loader = DataLoader(test_data, batch_size=256)
    model.eval(); yy, pp = [], []
    with torch.no_grad():
        for b in loader:
            b = b.to(device)
            pp.extend(torch.sigmoid(model(b)).cpu().tolist())
            yy.extend(b.y.view(-1).cpu().tolist())
    y, p = np.array(yy), np.array(pp)
    preds = (p >= 0.5).astype(int)
    return {"auroc": roc_auc_score(y, p) if len(set(yy)) > 1 else 0.5,
            "accuracy": accuracy_score(y, preds),
            "precision": precision_score(y, preds, zero_division=0),
            "recall": recall_score(y, preds, zero_division=0),
            "f1": f1_score(y, preds, zero_division=0)}, y, p


def scaffold_held_out_eval(model, dataset_info, df, device):
    """Evaluate model on test molecules whose scaffold was NOT seen in training.

    Returns dict with novel_scaffold_auroc, scaffold_overlap_pct, n_novel, n_test.
    """
    train_rows = dataset_info["train_rows"]
    test_rows = dataset_info["test_rows"]

    train_scaffolds = set(df.loc[train_rows, "scaffold"].unique())
    test_df_subset = df.loc[test_rows]
    test_scaffolds = set(test_df_subset["scaffold"].unique())

    overlap = train_scaffolds & test_scaffolds
    novel = test_scaffolds - train_scaffolds
    overlap_pct = len(overlap) / max(len(test_scaffolds), 1) * 100

    # Novel-scaffold test molecules
    novel_mask = test_df_subset["scaffold"].isin(novel)
    n_novel = novel_mask.sum()
    n_test = len(test_df_subset)

    result = {
        "n_test_scaffolds": len(test_scaffolds),
        "n_train_scaffolds": len(train_scaffolds),
        "n_overlap": len(overlap),
        "overlap_pct": overlap_pct,
        "n_novel_mols": int(n_novel),
        "n_test_mols": n_test,
    }

    if n_novel >= 10:
        # Get indices into the test data list
        novel_rows = test_df_subset[novel_mask].index.tolist()
        test_row_to_pos = {r: i for i, r in enumerate(test_rows)}
        novel_data = [dataset_info["test"][test_row_to_pos[r]]
                      for r in novel_rows if r in test_row_to_pos]
        if len(novel_data) >= 5:
            met, _, _ = evaluate_model(model, novel_data, device)
            result["novel_auroc"] = met["auroc"]
            result["novel_acc"] = met["accuracy"]
        else:
            result["novel_auroc"] = None
    else:
        result["novel_auroc"] = None

    return result


def per_class_size_diagnostic(df, task_name):
    """Report n_heavy distribution per struct_class to verify adaptive noise works."""
    print(f"\n  [SIZE BALANCE] {task_name}")
    for cls, grp in df.groupby("struct_class"):
        nh = grp["n_heavy"]
        has_n = grp["smiles"].apply(lambda s: "N" in s or "n" in s).mean()
        print(f"    {cls:12s}  n={len(grp):5d}  heavy={nh.mean():.1f}±{nh.std():.1f}  "
              f"range=[{nh.min()},{nh.max()}]  has_nitrogen={has_n:.0%}")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# §8  EXPLANATION WRAPPER  (calls existing pipeline)
#
#     evaluate_coalition fallback: if the main pipeline isn't loaded, we
#     provide a minimal reimplementation so deletion faithfulness works.
# ═══════════════════════════════════════════════════════════════════════════

def _evaluate_coalition_fallback(model, mol_graph, active_heavy, device):
    """Minimal reimplementation of evaluate_coalition for deletion metric.
    Masks inactive heavy atoms (and their H) to baseline features."""
    active_total = set()
    for h in active_heavy:
        active_total.add(h)
        active_total.update(mol_graph.heavy_to_h.get(h, []))

    x_masked = mol_graph.x.clone()
    for i in range(mol_graph.n_total):
        if i not in active_total:
            x_masked[i] = mol_graph.baseline

    data = Data(x=x_masked, edge_index=mol_graph.edge_index)
    batch = Batch.from_data_list([data]).to(device)
    with torch.no_grad():
        model.eval()
        z = model(batch)
        return z.item()

def _get_evaluate_coalition():
    """Return the pipeline's evaluate_coalition if available, else fallback."""
    try:
        # Try to use the one from the main pipeline
        return evaluate_coalition
    except NameError:
        logger.info("Using fallback evaluate_coalition (main pipeline not loaded)")
        return _evaluate_coalition_fallback

def explain_molecule_for_verification(model, smiles, device,
                                       n_samples_st2=50, n_samples_sv=50,
                                       sa_iterations=100, seed=42):
    try:
        return explain_molecule(
            model=model, smiles=smiles, device=device,
            n_samples_st2=n_samples_st2, n_samples_sv=n_samples_sv,
            sa_iterations=sa_iterations, sa_T0=1.0, sa_alpha=0.99,
            gamma_C=0.4, gamma_S=0.3, mu_ideal=5.0,
            use_mask_bit=False, seed=seed, verbose=False,
        )
    except Exception as e:
        logger.warning(f"Explanation failed for {smiles}: {e}")
        return None


# ═══════════════════════════════════════════════════════════════════════════
# §9  VERIFICATION METRICS  (deletion faithfulness FIXED)
# ═══════════════════════════════════════════════════════════════════════════

def compute_loc_main(phi, motif_pos):
    total = np.sum(np.abs(phi))
    if total < 1e-12: return 0.0
    return float(sum(abs(phi[p]) for p in motif_pos) / total)

def find_best_group_for_motif(partition, heavy_idx, motif_pos):
    g2p = {g: p for p, g in enumerate(heavy_idx)}
    best_gi, best_ov = -1, 0
    for gi, group in enumerate(partition):
        ov = len({g2p[g] for g in group if g in g2p} & motif_pos)
        if ov > best_ov: best_ov = ov; best_gi = gi
    return best_gi

def compute_interaction_dominance(group_agg, gi_A, gi_B):
    if gi_A < 0 or gi_B < 0: return 0.0
    I = abs(group_agg["cross_scores"][gi_A][gi_B])
    VA = abs(group_agg["group_values"][gi_A])
    VB = abs(group_agg["group_values"][gi_B])
    return I / (I + VA + VB + 1e-8)

def compute_interaction_rank(group_agg, gi_A, gi_B):
    if gi_A < 0 or gi_B < 0: return -1
    cross = group_agg["cross_scores"]
    K = len(cross)
    target = abs(cross[gi_A][gi_B])
    all_c = sorted([abs(cross[i][j]) for i in range(K) for j in range(i+1, K)], reverse=True)
    for r, v in enumerate(all_c, 1):
        if abs(v - target) < 1e-12: return r
    return len(all_c)


def bootstrap_ci(data, n_bootstrap=1000, ci=0.95, seed=42):
    """Bootstrap confidence interval for the mean of data."""
    rng = np.random.RandomState(seed)
    means = []
    for _ in range(n_bootstrap):
        sample = rng.choice(data, size=len(data), replace=True)
        means.append(np.mean(sample))
    lo = np.percentile(means, (1 - ci) / 2 * 100)
    hi = np.percentile(means, (1 + ci) / 2 * 100)
    return float(lo), float(hi)


def compute_deletion_faithfulness(model, mol_graph, phi, partition,
                                  group_agg, device, n_random=20,
                                  label=None):
    """Group-level deletion faithfulness with bootstrap CIs.

    Uses LOGIT space (not probability) to avoid saturation when p0 ≈ 0 or 1.
    This is critical for tasks like DIST where model outputs extreme logits.

    Direction-aware: for predicted-positive (z>0), deletion should decrease z.
    For predicted-negative (z<0), deletion should make z more negative.
    We track abs(full_z) - abs(z_k): how much confidence (in either direction) drops.

    Returns dict with del_auc, del_rand_mean/std, ins_auc, ins_rand_mean/std,
    bootstrap CI for (del_auc - del_rand), p0 (probability), p_final, K.
    """
    eval_coal = _get_evaluate_coalition()
    heavy = mol_graph.heavy_idx
    K = len(partition)
    empty = {"del_auc": 0., "del_auc_rand": 0., "del_auc_rand_std": 0.,
             "ins_auc": 0., "ins_auc_rand": 0., "ins_auc_rand_std": 0.,
             "del_diff_ci_lo": 0., "del_diff_ci_hi": 0.,
             "p0": 0., "p_final": 0., "K_groups": 0}
    if K == 0:
        return empty

    full_z = eval_coal(model, mol_graph, set(heavy), device)
    base_z = eval_coal(model, mol_graph, set(), device)
    p0 = 1.0 / (1.0 + math.exp(-max(min(full_z, 30), -30)))

    # Direction of model's prediction
    sign = 1.0 if full_z >= 0 else -1.0
    # Signed logit: positive = model is confident in its prediction
    # For z>0: signed_z = z (high is confident positive)
    # For z<0: signed_z = -z (high magnitude is confident negative)
    def _signed_logit(z):
        return sign * z

    sz_full = _signed_logit(full_z)
    sz_base = _signed_logit(base_z)

    def _deletion_curve(order):
        active = set(heavy)
        szvals = [sz_full]
        for gi in order:
            for a in partition[gi]:
                active.discard(a)
            z = eval_coal(model, mol_graph, active, device)
            szvals.append(_signed_logit(z))
        return szvals

    def _insertion_curve(order):
        active = set()
        szvals = [sz_base]
        for gi in order:
            for a in partition[gi]:
                active.add(a)
            z = eval_coal(model, mol_graph, active, device)
            szvals.append(_signed_logit(z))
        return szvals

    # Explanation-guided ordering
    order_expl = sorted(range(K), key=lambda i: abs(group_agg["group_values"][i]),
                        reverse=True)

    # Deletion (signed-logit space)
    szvals_del = _deletion_curve(order_expl)
    x_ax = np.linspace(0, 1, len(szvals_del))
    drops = [sz_full - sk for sk in szvals_del]
    del_auc = float(np.trapz(drops, x_ax))

    # Insertion (signed-logit space)
    ins_auc = 0.0
    if CFG.INSERTION_ENABLED:
        szvals_ins = _insertion_curve(order_expl)
        x_ins = np.linspace(0, 1, len(szvals_ins))
        gains = [sk - sz_base for sk in szvals_ins]
        ins_auc = float(np.trapz(gains, x_ins))

    # Random baselines
    rng = np.random.RandomState(42)
    rand_del_aucs, rand_ins_aucs = [], []
    for _ in range(n_random):
        ro = rng.permutation(K).tolist()
        sv = _deletion_curve(ro)
        x_r = np.linspace(0, 1, len(sv))
        rand_del_aucs.append(float(np.trapz([sz_full - sk for sk in sv], x_r)))
        if CFG.INSERTION_ENABLED:
            svi = _insertion_curve(ro)
            x_ri = np.linspace(0, 1, len(svi))
            rand_ins_aucs.append(float(np.trapz([sk - sz_base for sk in svi], x_ri)))

    del_rand_mean = float(np.mean(rand_del_aucs))
    del_rand_std = float(np.std(rand_del_aucs))
    ins_rand_mean = float(np.mean(rand_ins_aucs)) if rand_ins_aucs else 0.0
    ins_rand_std = float(np.std(rand_ins_aucs)) if rand_ins_aucs else 0.0

    # Bootstrap 95% CI for (del_auc - rand_auc)
    diffs = [del_auc - ra for ra in rand_del_aucs]
    if len(diffs) >= 5:
        ci_lo, ci_hi = bootstrap_ci(diffs, n_bootstrap=CFG.DEL_BOOTSTRAP_N)
    else:
        ci_lo, ci_hi = 0.0, 0.0

    # Convert final signed-logit back to probability for p_final
    z_final = szvals_del[-1] * sign   # undo sign: sign² = 1
    p_final = 1.0 / (1.0 + math.exp(-max(min(z_final, 30), -30)))

    return {
        "del_auc": del_auc, "del_auc_rand": del_rand_mean,
        "del_auc_rand_std": del_rand_std,
        "ins_auc": ins_auc, "ins_auc_rand": ins_rand_mean,
        "ins_auc_rand_std": ins_rand_std,
        "del_diff_ci_lo": ci_lo, "del_diff_ci_hi": ci_hi,
        "p0": p0, "p_final": p_final, "K_groups": K,
    }


def deletion_sanity_check(results_df):
    """Quick check: for confident predictions, del_auc should exceed random.
    Now direction-aware: confident = |p0 - 0.5| > 0.3 (either class)."""
    confident = results_df[abs(results_df["p0"] - 0.5) > 0.3]
    if len(confident) < 3:
        print("  [DEL SANITY] Too few confident samples to check")
        return
    mean_expl = confident["del_auc"].mean()
    mean_rand = confident["del_auc_rand"].mean()
    if mean_expl > mean_rand:
        print(f"  [DEL SANITY] ✓ PASS: expl_auc={mean_expl:.4f} > rand_auc={mean_rand:.4f} "
              f"(n={len(confident)})")
    else:
        print(f"  [DEL SANITY] ⚠ FAIL: expl_auc={mean_expl:.4f} <= rand_auc={mean_rand:.4f} "
              f"— explanation ordering no better than random!")


# ═══════════════════════════════════════════════════════════════════════════
# §10  NEW INTERACTION-QUALITY METRICS
# ═══════════════════════════════════════════════════════════════════════════

def compute_synergy_sign_rate(results_df):
    """For both-motif molecules: fraction where I_AB > 0.
    Uses clean_label if available."""
    both = results_df[(results_df["has_A"] == 1) & (results_df["has_B"] == 1)]
    if both.empty: return {}
    lab_col = "clean_label" if "clean_label" in both.columns else "label"
    out = {}
    for lab in both[lab_col].unique():
        sub = both[both[lab_col] == lab]
        if len(sub):
            out[f"synergy_positive_rate_label{lab}"] = float((sub["I_AB"] > 0).mean())
            out[f"n_both_label{lab}"] = len(sub)
    return out


def compute_interaction_separability(results_df):
    """AUROC of using I_AB to separate label=1 vs label=0 within both-motif subset.
    Uses clean_label (ground truth) if available."""
    both = results_df[(results_df["has_A"] == 1) & (results_df["has_B"] == 1)]
    if both.empty:
        return {"interaction_sep_auroc": float("nan"), "interaction_sep_n": 0}
    lab_col = "clean_label" if "clean_label" in both.columns else "label"
    if both[lab_col].nunique() < 2:
        return {"interaction_sep_auroc": float("nan"),
                "interaction_sep_n": len(both),
                "interaction_sep_note": f"only label={both[lab_col].iloc[0]} in both-motif subset"}
    try:
        auc = roc_auc_score(both[lab_col].values, both["I_AB"].values)
    except Exception:
        auc = float("nan")
    return {"interaction_sep_auroc": auc, "interaction_sep_n": len(both)}


def compute_distance_correlation(results_df):
    """Spearman correlation between I_AB and dist_AB (DIST task only)."""
    both = results_df[(results_df["has_A"] == 1) & (results_df["has_B"] == 1)]
    if "dist_AB" not in both.columns or both.empty:
        return {"spearman_I_AB_dist": float("nan"), "spearman_pvalue": float("nan")}
    valid = both[both["dist_AB"] >= 0]
    if len(valid) < 5:
        return {"spearman_I_AB_dist": float("nan"), "spearman_pvalue": float("nan")}
    rho, pval = spearmanr(valid["I_AB"].values, valid["dist_AB"].values)
    return {"spearman_I_AB_dist": float(rho), "spearman_pvalue": float(pval)}


# ═══════════════════════════════════════════════════════════════════════════
# §11  EXPLANATION VERIFICATION LOOP
# ═══════════════════════════════════════════════════════════════════════════

def verify_explanations(model, test_df, task_name, device,
                        n_explain=30, n_samples_st2=50,
                        n_samples_sv=50, sa_iterations=100):
    print(f"\n{'='*60}")
    print(f"VERIFICATION: {task_name}  n_st2={n_samples_st2}")
    print(f"{'='*60}")

    MAX_H = CFG.MAX_HEAVY_EXPLAIN
    small = test_df[test_df["smiles"].apply(
        lambda s: (Chem.MolFromSmiles(s) is not None and
                   sum(1 for a in Chem.MolFromSmiles(s).GetAtoms()
                       if a.GetAtomicNum() != 1) <= MAX_H))]
    if len(small) < 10:
        print(f"  ⚠ Only {len(small)} with ≤{MAX_H} heavy → using all")
        small = test_df

    toxic = small[small["label"] == 1].sample(n=min(n_explain, (small["label"]==1).sum()), random_state=42)
    nontoxic = small[small["label"] == 0].sample(n=min(n_explain, (small["label"]==0).sum()), random_state=42)
    subset = pd.concat([toxic, nontoxic]).reset_index(drop=True)
    print(f"  Explaining {len(subset)} ({len(toxic)} toxic, {len(nontoxic)} non-toxic)")

    results = []
    t_start = time.time()

    for mol_i, (idx, row) in enumerate(subset.iterrows()):
        smi = row["smiles"]
        label = int(row["label"])
        clean_label = int(row["clean_label"]) if "clean_label" in row.index else label
        has_A = int(row.get("has_A", 0))
        has_B = int(row.get("has_B", 0))
        dist_AB = float(row.get("dist_AB", -1)) if "dist_AB" in row.index else -1.0

        t_mol = time.time()
        res = explain_molecule_for_verification(
            model, smi, device,
            n_samples_st2=n_samples_st2,
            n_samples_sv=n_samples_sv,
            sa_iterations=sa_iterations,
        )
        elapsed_mol = time.time() - t_mol

        if res is None:
            print(f"  [{mol_i+1}/{len(subset)}] FAIL  {smi[:40]}…  ({elapsed_mol:.1f}s)")
            continue

        mg = res["mol_graph"]
        phi = res["phi"]
        W = res["W"]
        partition = res["best_partition"]
        group_agg = res["group_agg"]

        A_pos = get_motif_heavy_positions(mg, MOTIF_A_SMARTS)
        B_pos = get_motif_heavy_positions(mg, MOTIF_B_SMARTS)
        motif_pos = A_pos | B_pos

        loc = compute_loc_main(phi, motif_pos)

        dom, rank, I_AB_val = 0.0, -1, 0.0
        gi_A = find_best_group_for_motif(partition, mg.heavy_idx, A_pos) if A_pos else -1
        gi_B = find_best_group_for_motif(partition, mg.heavy_idx, B_pos) if B_pos else -1
        if gi_A >= 0 and gi_B >= 0 and gi_A != gi_B:
            dom = compute_interaction_dominance(group_agg, gi_A, gi_B)
            rank = compute_interaction_rank(group_agg, gi_A, gi_B)
            I_AB_val = group_agg["cross_scores"][gi_A][gi_B]

        # Deletion faithfulness — ALWAYS runs now (returns dict)
        del_dict = {"del_auc": 0., "del_auc_rand": 0., "del_auc_rand_std": 0.,
                    "ins_auc": 0., "ins_auc_rand": 0., "ins_auc_rand_std": 0.,
                    "del_diff_ci_lo": 0., "del_diff_ci_hi": 0.,
                    "p0": 0., "p_final": 0., "K_groups": 0}
        if CFG.DEL_ENABLED:
            try:
                del_dict = compute_deletion_faithfulness(
                    model, mg, phi, partition, group_agg, device,
                    n_random=CFG.DEL_N_RANDOM)
            except Exception as e:
                logger.warning(f"Deletion failed for {smi}: {e}")

        rec = {
            "smiles": smi, "label": label, "clean_label": clean_label,
            "has_A": has_A, "has_B": has_B,
            "method": "ours",
            "n_samples_st2": n_samples_st2, "n_heavy": mg.n_heavy,
            "loc_main": loc, "dom": dom, "rank": rank, "I_AB": I_AB_val,
            **del_dict,
        }
        if dist_AB >= 0:
            rec["dist_AB"] = dist_AB

        results.append(rec)

        elapsed_total = time.time() - t_start
        avg = elapsed_total / (mol_i + 1)
        eta = avg * (len(subset) - mol_i - 1)
        print(f"  [{mol_i+1}/{len(subset)}]  {elapsed_mol:.1f}s  "
              f"heavy={mg.n_heavy}  loc={loc:.2f}  dom={dom:.2f}  "
              f"rank={rank}  del={del_dict['del_auc']:.3f}  p0={del_dict['p0']:.2f}  (ETA {eta:.0f}s)")

    df_out = pd.DataFrame(results)
    print(f"  Done: {len(df_out)} explained")

    # Run deletion sanity check
    if CFG.DEL_ENABLED and len(df_out):
        deletion_sanity_check(df_out)

    return df_out


# ═══════════════════════════════════════════════════════════════════════════
# §12  PLOTS
# ═══════════════════════════════════════════════════════════════════════════

def plot_roc_curves(roc_data, save_path):
    fig, ax = plt.subplots(figsize=(8, 6))
    for n, (y, p) in roc_data.items():
        fpr, tpr, _ = roc_curve(y, p)
        ax.plot(fpr, tpr, label=f"{n} ({roc_auc_score(y,p):.3f})", lw=2)
    ax.plot([0,1],[0,1],"k--",alpha=.5)
    ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
    ax.set_title("ROC Curves"); ax.legend(); ax.grid(True, alpha=.3)
    plt.tight_layout(); plt.savefig(save_path, dpi=150); plt.close()

def plot_interaction_histogram(df, task, path):
    fig, ax = plt.subplots(figsize=(8, 5))
    t = df[df["label"]==1]["I_AB"].dropna(); nt = df[df["label"]==0]["I_AB"].dropna()
    all_vals = pd.concat([t, nt])
    if all_vals.empty: plt.close(); return
    bins = np.linspace(all_vals.min()-0.1, all_vals.max()+0.1, 30)
    ax.hist(t, bins=bins, alpha=.7, label="Toxic", color="red", edgecolor="darkred")
    ax.hist(nt, bins=bins, alpha=.7, label="Non-toxic", color="blue", edgecolor="darkblue")
    ax.set_xlabel("I(A,B)"); ax.set_ylabel("Count"); ax.set_title(f"I(A,B) — {task}")
    ax.legend(); ax.grid(True, alpha=.3)
    plt.tight_layout(); plt.savefig(path, dpi=150); plt.close()

def plot_convergence(all_conv, path):
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for n, cdf in all_conv.items():
        if cdf.empty: continue
        ns = cdf["n_samples_st2"].values
        axes[0].plot(ns, cdf["loc_main_mean"].values, "o-", label=n)
        axes[1].plot(ns, cdf["dom_mean"].values, "o-", label=n)
        axes[2].plot(ns, cdf["top1_rate"].values, "o-", label=n)
    axes[0].set_ylabel("Localization"); axes[1].set_ylabel("Dominance"); axes[2].set_ylabel("Top-1")
    for ax in axes: ax.set_xlabel("n_st2"); ax.legend(fontsize=9); ax.grid(True, alpha=.3)
    plt.suptitle("Convergence"); plt.tight_layout(); plt.savefig(path, dpi=150); plt.close()

def plot_deletion_comparison(all_exp, path):
    """Bar chart: mean del_auc (explanation) vs del_auc_rand per task."""
    tasks = all_exp["task"].unique()
    fig, ax = plt.subplots(figsize=(8, 5))
    x = np.arange(len(tasks))
    w = 0.35
    expl_means, rand_means, rand_stds = [], [], []
    for t in tasks:
        sub = all_exp[all_exp["task"] == t]
        expl_means.append(sub["del_auc"].mean())
        rand_means.append(sub["del_auc_rand"].mean())
        rand_stds.append(sub["del_auc_rand_std"].mean())
    ax.bar(x - w/2, expl_means, w, label="Explanation order", color="coral")
    ax.bar(x + w/2, rand_means, w, yerr=rand_stds, label="Random order", color="steelblue", capsize=3)
    ax.set_xticks(x); ax.set_xticklabels(tasks, fontsize=9)
    ax.set_ylabel("Deletion AUC (drop curve)")
    ax.set_title("Deletion Faithfulness: Explanation vs Random")
    ax.legend(); ax.grid(True, alpha=.3, axis="y")
    plt.tight_layout(); plt.savefig(path, dpi=150); plt.close()


# ═══════════════════════════════════════════════════════════════════════════
# §13  MAIN
# ═══════════════════════════════════════════════════════════════════════════

# ═══════════════════════════════════════════════════════════════════════════
# §13  BASELINE A: Grad×Input ATOM SALIENCY
# ═══════════════════════════════════════════════════════════════════════════

def grad_x_input_attribution(model, smiles, device):
    """Compute per-atom importance via Grad×Input.

    For each atom i, score_i = sum_d |grad(output, x_i^d) * x_i^d|.
    Returns heavy-atom-only scores (matching our pipeline's indexing).
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None, None
    d = smiles_to_pyg_data(smiles, label=0)
    if d is None:
        return None, None

    d = d.to(device)
    d.x.requires_grad_(True)
    d.batch = torch.zeros(d.x.size(0), dtype=torch.long, device=device)

    model.eval()
    out = model(d)
    out.backward()

    grad = d.x.grad  # [n_atoms, feat_dim]
    gxi = (grad * d.x).abs().sum(dim=1).detach().cpu().numpy()  # [n_atoms]

    # Filter to heavy atoms only
    heavy_mask = [i for i, a in enumerate(mol.GetAtoms()) if a.GetAtomicNum() != 1]
    heavy_scores = gxi[heavy_mask] if len(heavy_mask) <= len(gxi) else gxi
    return heavy_scores, heavy_mask


class _LiteMolGraph:
    """Lightweight mol_graph for baseline deletion without running explanation pipeline.

    CRITICAL: Uses heavy-atom-only graph to match smiles_to_pyg_data / model training.
    No AddHs — the model was never trained on H atoms.
    Provides the minimal interface needed by _evaluate_coalition_fallback.
    """

    def __init__(self, smiles):
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            raise ValueError(f"Invalid SMILES: {smiles}")
        # Do NOT add H — model trains on heavy-atom-only graphs
        self.mol = mol
        self.heavy_idx = list(range(mol.GetNumAtoms()))  # all atoms are heavy
        self.n_heavy = mol.GetNumAtoms()
        self.n_total = mol.GetNumAtoms()

        # No H neighbors (heavy-only graph)
        self.heavy_to_h = {i: [] for i in range(mol.GetNumAtoms())}

        # Build features matching smiles_to_pyg_data exactly
        feats = []
        for atom in mol.GetAtoms():
            hyb = _HYB_MAP.get(atom.GetHybridization(), 6)
            feats.append([atom.GetAtomicNum(), atom.GetDegree(), atom.GetTotalNumHs(),
                          atom.GetTotalValence(), int(atom.GetIsAromatic()),
                          atom.GetFormalCharge(), hyb, int(atom.IsInRing())])
        self.x = torch.tensor(feats, dtype=torch.float32)
        self.baseline = torch.zeros(self.x.size(1), dtype=torch.float32)

        edge_list = []
        for bond in mol.GetBonds():
            i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
            edge_list += [[i, j], [j, i]]
        self.edge_index = (torch.tensor(edge_list, dtype=torch.long).t().contiguous()
                           if edge_list else torch.zeros((2, 0), dtype=torch.long))


def evaluate_baseline_grad(model, test_df, task_name, device, n_explain=30):
    """Run Grad×Input baseline with PURE atom-level deletion (no borrowed partition).

    Each heavy atom is its own group. Deletion order = descending |grad_score|.
    This is a strict baseline — no dependence on our explanation method.
    """
    print(f"\n  [GRAD×INPUT] Evaluating on {task_name}...")

    MAX_H = CFG.MAX_HEAVY_EXPLAIN
    small = test_df[test_df["smiles"].apply(
        lambda s: (Chem.MolFromSmiles(s) is not None and
                   count_heavy_atoms(Chem.MolFromSmiles(s)) <= MAX_H))]
    if len(small) < 10:
        small = test_df

    toxic = small[small["label"] == 1].sample(n=min(n_explain, (small["label"]==1).sum()), random_state=42)
    nontoxic = small[small["label"] == 0].sample(n=min(n_explain, (small["label"]==0).sum()), random_state=42)
    subset = pd.concat([toxic, nontoxic]).reset_index(drop=True)

    results = []
    for mol_i, (_, row) in enumerate(subset.iterrows()):
        smi = row["smiles"]
        scores, heavy_mask = grad_x_input_attribution(model, smi, device)
        if scores is None:
            continue

        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue

        # Localization: fraction of attribution on motif atoms
        A_idx = get_motif_atom_indices(mol, MOTIF_A_SMARTS)
        B_idx = get_motif_atom_indices(mol, MOTIF_B_SMARTS)
        motif_heavy = set()
        for mi, gi in enumerate(heavy_mask):
            if gi in A_idx or gi in B_idx:
                motif_heavy.add(mi)

        total = np.sum(np.abs(scores))
        loc = float(sum(abs(scores[p]) for p in motif_heavy) / total) if total > 1e-12 else 0.0

        # Atom-level deletion: each heavy atom = its own group
        del_dict = {"del_auc": 0., "del_auc_rand": 0., "del_auc_rand_std": 0.,
                    "ins_auc": 0., "ins_auc_rand": 0., "ins_auc_rand_std": 0.,
                    "del_diff_ci_lo": 0., "del_diff_ci_hi": 0.,
                    "p0": 0., "p_final": 0., "K_groups": 0}
        try:
            mg = _LiteMolGraph(smi)
            # Atom-level partition: each heavy atom in its own group
            partition = [[h] for h in mg.heavy_idx]
            # Group values = per-atom |grad score|, mapped via heavy_mask
            g2p = {g: p for p, g in enumerate(mg.heavy_idx)}
            grad_group_vals = []
            for grp in partition:
                a = grp[0]  # single atom per group
                p_idx = g2p.get(a)
                val = float(abs(scores[p_idx])) if p_idx is not None and p_idx < len(scores) else 0.0
                grad_group_vals.append(val)
            mock_group_agg = {
                "group_values": grad_group_vals,
                "cross_scores": [[0.0] * len(partition) for _ in partition],
            }
            del_dict = compute_deletion_faithfulness(
                model, mg, scores, partition, mock_group_agg, device,
                n_random=CFG.DEL_N_RANDOM)
        except Exception as e:
            logger.warning(f"Grad deletion failed for {smi}: {e}")

        results.append({
            "smiles": smi, "label": int(row["label"]),
            "clean_label": int(row.get("clean_label", row["label"])),
            "has_A": int(row.get("has_A", 0)), "has_B": int(row.get("has_B", 0)),
            "method": "grad_x_input",
            "n_samples_st2": 0, "n_heavy": len(heavy_mask),
            "loc_main": loc, "dom": 0.0, "rank": -1, "I_AB": 0.0,
            **del_dict,
        })
        if (mol_i + 1) % 10 == 0:
            print(f"    [{mol_i+1}/{len(subset)}] loc={loc:.2f} del={del_dict['del_auc']:.3f}")

    df = pd.DataFrame(results)
    print(f"  [GRAD×INPUT] Done: {len(df)} molecules, mean loc={df['loc_main'].mean():.3f}")
    return df


# ═══════════════════════════════════════════════════════════════════════════
# §14  BASELINE B: SYNERGY PROXY  f(A∪B) - f(A) - f(B) + f(∅)
# ═══════════════════════════════════════════════════════════════════════════

def compute_synergy_proxy(model, smiles, device):
    """Compute difference-in-differences synergy proxy for a molecule.

    synergy = f(A∪B) - f(A) - f(B) + f(∅)
    where f is sigmoid(model_logit) evaluated on coalitions defined by
    motif atom groups A and B.

    Returns synergy value, or None if motifs not found.
    """
    eval_coal = _get_evaluate_coalition()

    try:
        mg = MoleculeGraph(smiles)
    except NameError:
        # MoleculeGraph not loaded from main pipeline — use explain fallback
        try:
            res = explain_molecule_for_verification(model, smiles, device, 10, 10, 10)
            if res is None:
                return None
            mg = res["mol_graph"]
        except Exception:
            return None
    except Exception:
        return None

    mol = mg.mol
    A_atoms = get_motif_atom_indices(mol, MOTIF_A_SMARTS)
    B_atoms = get_motif_atom_indices(mol, MOTIF_B_SMARTS)
    if not A_atoms or not B_atoms:
        return None

    # Map to heavy atom indices used by MoleculeGraph
    heavy_set = set(mg.heavy_idx)
    A_heavy = {a for a in A_atoms if a in heavy_set}
    B_heavy = {a for a in B_atoms if a in heavy_set}
    AB_heavy = A_heavy | B_heavy
    all_heavy = set(mg.heavy_idx)

    def _p(coalition):
        z = eval_coal(model, mg, coalition, device)
        return 1.0 / (1.0 + math.exp(-z))

    # f(∅), f(A), f(B), f(A∪B) — keep non-motif atoms always active
    background = all_heavy - AB_heavy
    p_empty = _p(background)
    p_A = _p(background | A_heavy)
    p_B = _p(background | B_heavy)
    p_AB = _p(background | AB_heavy)

    synergy = p_AB - p_A - p_B + p_empty
    return synergy


def evaluate_synergy_proxy(model, test_df, task_name, device, n_explain=30):
    """Compute synergy proxy for both-motif molecules."""
    print(f"\n  [SYNERGY PROXY] Computing for {task_name}...")
    both = test_df[(test_df["has_A"] == 1) & (test_df["has_B"] == 1)]
    if both.empty:
        print("    No both-motif molecules")
        return pd.DataFrame()

    MAX_H = CFG.MAX_HEAVY_EXPLAIN
    small = both[both["smiles"].apply(
        lambda s: (Chem.MolFromSmiles(s) is not None and
                   count_heavy_atoms(Chem.MolFromSmiles(s)) <= MAX_H))]
    subset = small.sample(n=min(n_explain * 2, len(small)), random_state=42)

    results = []
    for mol_i, (_, row) in enumerate(subset.iterrows()):
        syn = compute_synergy_proxy(model, row["smiles"], device)
        if syn is None:
            continue
        rec = {"smiles": row["smiles"], "label": int(row["label"]),
               "clean_label": int(row.get("clean_label", row["label"])),
               "has_A": 1, "has_B": 1,
               "synergy_proxy": syn, "task": task_name}
        if "dist_AB" in row.index and row["dist_AB"] >= 0:
            rec["dist_AB"] = float(row["dist_AB"])
        results.append(rec)

    df = pd.DataFrame(results)
    if not df.empty:
        print(f"    Computed {len(df)} synergy values, mean={df['synergy_proxy'].mean():.4f}")
    return df


# ═══════════════════════════════════════════════════════════════════════════
# §15  ARTIFACT DIAGNOSTICS
# ═══════════════════════════════════════════════════════════════════════════

def artifact_diagnostics(df, task_name):
    """Fit logistic regression on nuisance features to check for shortcuts.

    If AUROC >> 0.5, the label is predictable from cheap features → artifact.
    """
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import cross_val_score

    print(f"\n  [ARTIFACT CHECK] {task_name}")

    feats = []
    for _, row in df.iterrows():
        mol = Chem.MolFromSmiles(row["smiles"])
        if mol is None:
            feats.append([0, 0, 0, 0])
            continue
        n_heavy = count_heavy_atoms(mol)
        n_nitrogen = sum(1 for a in mol.GetAtoms() if a.GetAtomicNum() == 7)
        n_aromatic = sum(1 for a in mol.GetAtoms() if a.GetIsAromatic())
        n_rings = mol.GetRingInfo().NumRings()
        feats.append([n_heavy, n_nitrogen, n_aromatic, n_rings])

    X = np.array(feats)
    y = df["label"].values if "clean_label" not in df.columns else df["clean_label"].values

    # Also check scaffold-only predictiveness
    scaffold_auroc = 0.5
    if "scaffold" in df.columns:
        from sklearn.preprocessing import LabelEncoder
        scaff_enc = LabelEncoder().fit_transform(df["scaffold"].values).reshape(-1, 1)
        try:
            lr_s = LogisticRegression(max_iter=500, random_state=42)
            sc_s = cross_val_score(lr_s, scaff_enc, y, cv=5, scoring="roc_auc")
            scaffold_auroc = sc_s.mean()
        except Exception:
            scaffold_auroc = 0.5

    try:
        lr = LogisticRegression(max_iter=500, random_state=42)
        scores = cross_val_score(lr, X, y, cv=5, scoring="roc_auc")
        mean_auc = scores.mean()
        std_auc = scores.std()
    except Exception as e:
        logger.warning(f"Artifact check failed: {e}")
        mean_auc, std_auc = 0.5, 0.0

    status = "✓ OK (no artifact)" if mean_auc < 0.6 else "⚠ ARTIFACT RISK"
    print(f"    Nuisance-feature AUROC: {mean_auc:.3f} ± {std_auc:.3f}  {status}")
    if scaffold_auroc > 0.55:
        print(f"    Scaffold-only AUROC:    {scaffold_auroc:.3f}  ⚠ scaffold shortcut")
    else:
        print(f"    Scaffold-only AUROC:    {scaffold_auroc:.3f}  ✓ OK")

    return {"task": task_name,
            "nuisance_auroc_mean": mean_auc, "nuisance_auroc_std": std_auc,
            "scaffold_auroc": scaffold_auroc,
            "features": "n_heavy,n_nitrogen,n_aromatic,n_rings",
            "status": status}


# ═══════════════════════════════════════════════════════════════════════════
# §16  EXPLANATION STABILITY
# ═══════════════════════════════════════════════════════════════════════════

def stability_test(model, test_df, task_name, device):
    """Repeat explanations with different seeds, measure consistency."""
    print(f"\n  [STABILITY] {task_name}")
    seeds = CFG.STABILITY_SEEDS
    n_mol = CFG.STABILITY_N

    MAX_H = CFG.MAX_HEAVY_EXPLAIN
    small = test_df[test_df["smiles"].apply(
        lambda s: (Chem.MolFromSmiles(s) is not None and
                   count_heavy_atoms(Chem.MolFromSmiles(s)) <= MAX_H))]
    both = small[(small["has_A"] == 1) & (small["has_B"] == 1)]
    pool = both if len(both) >= n_mol else small
    subset = pool.sample(n=min(n_mol, len(pool)), random_state=42)

    rows = []
    for _, mol_row in subset.iterrows():
        smi = mol_row["smiles"]
        per_seed = []
        for seed in seeds:
            res = explain_molecule_for_verification(
                model, smi, device,
                n_samples_st2=CFG.N_SAMPLES_ST2_LIST[0],
                n_samples_sv=CFG.N_SAMPLES_SV,
                sa_iterations=CFG.SA_ITERATIONS,
                seed=seed)
            if res is None:
                continue
            mg = res["mol_graph"]
            phi = res["phi"]
            partition = res["best_partition"]
            group_agg = res["group_agg"]

            A_pos = get_motif_heavy_positions(mg, MOTIF_A_SMARTS)
            B_pos = get_motif_heavy_positions(mg, MOTIF_B_SMARTS)
            motif_pos = A_pos | B_pos
            loc = compute_loc_main(phi, motif_pos)

            gi_A = find_best_group_for_motif(partition, mg.heavy_idx, A_pos) if A_pos else -1
            gi_B = find_best_group_for_motif(partition, mg.heavy_idx, B_pos) if B_pos else -1
            I_AB = 0.0
            if gi_A >= 0 and gi_B >= 0 and gi_A != gi_B:
                I_AB = group_agg["cross_scores"][gi_A][gi_B]

            per_seed.append({"seed": seed, "loc": loc, "I_AB": I_AB})

        if len(per_seed) >= 2:
            locs = [s["loc"] for s in per_seed]
            iabs = [s["I_AB"] for s in per_seed]
            rows.append({
                "smiles": smi, "task": task_name,
                "loc_mean": np.mean(locs), "loc_std": np.std(locs),
                "I_AB_mean": np.mean(iabs), "I_AB_std": np.std(iabs),
                "n_seeds": len(per_seed),
            })

    df = pd.DataFrame(rows)
    if not df.empty:
        print(f"    loc: mean_std={df['loc_std'].mean():.3f}  "
              f"I_AB: mean_std={df['I_AB_std'].mean():.4f}  (n={len(df)})")
    else:
        print("    No stable results")
    return df


# ═══════════════════════════════════════════════════════════════════════════
# §17  DISTANCE BUCKET ANALYSIS (DIST task)
# ═══════════════════════════════════════════════════════════════════════════

def distance_bucket_analysis(exp_df, synergy_df=None):
    """Analyze |I_AB| vs distance buckets for DIST task molecules."""
    both = exp_df[(exp_df["has_A"] == 1) & (exp_df["has_B"] == 1)].copy()
    if "dist_AB" not in both.columns or both.empty:
        return pd.DataFrame()

    valid = both[both["dist_AB"] >= 0].copy()
    if valid.empty:
        return pd.DataFrame()

    # Create distance buckets
    valid["dist_bucket"] = valid["dist_AB"].apply(
        lambda d: str(int(d)) if d <= 4 else "5+")

    rows = []
    for bucket, grp in valid.groupby("dist_bucket"):
        rec = {
            "dist_bucket": bucket, "n": len(grp),
            "I_AB_mean": grp["I_AB"].mean(), "I_AB_std": grp["I_AB"].std(),
            "abs_I_AB_mean": grp["I_AB"].abs().mean(),
            "abs_I_AB_std": grp["I_AB"].abs().std(),
            "label_mean": grp["label"].mean(),
        }
        rows.append(rec)

    # Add synergy proxy column if available
    if synergy_df is not None and not synergy_df.empty and "dist_AB" in synergy_df.columns:
        sv = synergy_df[synergy_df["dist_AB"] >= 0].copy()
        sv["dist_bucket"] = sv["dist_AB"].apply(lambda d: str(int(d)) if d <= 4 else "5+")
        for rec in rows:
            bucket_syn = sv[sv["dist_bucket"] == rec["dist_bucket"]]
            if not bucket_syn.empty:
                rec["synergy_proxy_mean"] = bucket_syn["synergy_proxy"].mean()
                rec["synergy_proxy_std"] = bucket_syn["synergy_proxy"].std()

    return pd.DataFrame(rows)


# ═══════════════════════════════════════════════════════════════════════════
# §18  ADDITIONAL PLOTS
# ═══════════════════════════════════════════════════════════════════════════

def plot_distance_buckets(bucket_df, path):
    """Bar chart: mean |I_AB| per distance bucket with error bars."""
    if bucket_df.empty:
        return
    fig, ax = plt.subplots(figsize=(8, 5))
    bk = bucket_df.sort_values("dist_bucket")
    x = np.arange(len(bk))
    ax.bar(x, bk["abs_I_AB_mean"], yerr=bk["abs_I_AB_std"],
           capsize=4, color="coral", edgecolor="darkred", alpha=0.8)
    if "synergy_proxy_mean" in bk.columns:
        ax.bar(x + 0.35, bk["synergy_proxy_mean"].fillna(0),
               yerr=bk["synergy_proxy_std"].fillna(0), width=0.3,
               capsize=4, color="steelblue", edgecolor="navy", alpha=0.8,
               label="Synergy proxy")
        ax.legend()
    ax.set_xticks(x)
    ax.set_xticklabels(bk["dist_bucket"])
    ax.set_xlabel("Distance (bonds)")
    ax.set_ylabel("|I(A,B)| or synergy proxy")
    ax.set_title("Interaction Strength vs Motif Distance")
    ax.grid(True, alpha=0.3, axis="y")
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.close()


def plot_method_comparison(all_exp_df, path):
    """Bar chart comparing localization and deletion AUC across methods."""
    if all_exp_df.empty:
        return
    methods = all_exp_df["method"].unique()
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Localization
    for m in methods:
        sub = all_exp_df[all_exp_df["method"] == m]
        axes[0].bar(m, sub["loc_main"].mean(),
                    yerr=sub["loc_main"].std(), capsize=5, alpha=0.8)
    axes[0].set_ylabel("Motif Localization")
    axes[0].set_title("Localization by Method")
    axes[0].grid(True, alpha=0.3, axis="y")

    # Deletion AUC
    for m in methods:
        sub = all_exp_df[all_exp_df["method"] == m]
        axes[1].bar(m, sub["del_auc"].mean(),
                    yerr=sub["del_auc"].std(), capsize=5, alpha=0.8)
    axes[1].set_ylabel("Deletion AUC")
    axes[1].set_title("Deletion Faithfulness by Method")
    axes[1].grid(True, alpha=0.3, axis="y")

    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.close()


# ═══════════════════════════════════════════════════════════════════════════
# §19  DELETION & INTERACTION SUMMARY BUILDERS
# ═══════════════════════════════════════════════════════════════════════════

def build_deletion_summary(all_exp_df):
    """Build deletion_summary.csv: means, std, CI per task × method."""
    rows = []
    for (task, method), grp in all_exp_df.groupby(["task", "method"]):
        rec = {"task": task, "method": method, "n": len(grp)}
        for col in ["del_auc", "del_auc_rand", "ins_auc", "ins_auc_rand"]:
            if col in grp.columns:
                rec[f"{col}_mean"] = grp[col].mean()
                rec[f"{col}_std"] = grp[col].std()
        # Aggregate bootstrap CI from individual molecules
        if "del_diff_ci_lo" in grp.columns:
            rec["del_diff_ci_lo_mean"] = grp["del_diff_ci_lo"].mean()
            rec["del_diff_ci_hi_mean"] = grp["del_diff_ci_hi"].mean()
        # Overall bootstrap CI from the sample of del_auc - del_auc_rand
        diffs = (grp["del_auc"] - grp["del_auc_rand"]).values
        if len(diffs) >= 5:
            ci_lo, ci_hi = bootstrap_ci(diffs, n_bootstrap=CFG.DEL_BOOTSTRAP_N)
            rec["overall_diff_ci_lo"] = ci_lo
            rec["overall_diff_ci_hi"] = ci_hi
        rows.append(rec)
    return pd.DataFrame(rows)


def build_interaction_distance_analysis(exp_df, synergy_df, task_name):
    """Build interaction_distance_analysis.csv for DIST task."""
    both = exp_df[(exp_df["has_A"] == 1) & (exp_df["has_B"] == 1)].copy()
    result = {"task": task_name}

    if "dist_AB" not in both.columns or both.empty:
        return result

    valid = both[both["dist_AB"] >= 0]
    if len(valid) >= 5:
        rho, pval = spearmanr(valid["I_AB"].abs().values, valid["dist_AB"].values)
        result["spearman_absI_vs_dist"] = rho
        result["spearman_pvalue"] = pval

        # Separability: AUROC of I_AB for label=1 vs label=0 (within both-motif)
        lab_col = "clean_label" if "clean_label" in valid.columns else "label"
        if valid[lab_col].nunique() >= 2:
            try:
                result["sep_auroc_I_AB"] = roc_auc_score(valid[lab_col], valid["I_AB"])
                result["sep_auroc_absI_AB"] = roc_auc_score(valid[lab_col], valid["I_AB"].abs())
            except Exception:
                pass

    # Synergy proxy separability
    if synergy_df is not None and not synergy_df.empty:
        sv = synergy_df[synergy_df.get("dist_AB", pd.Series(dtype=float)) >= 0]
        lab_col = "clean_label" if "clean_label" in sv.columns else "label"
        if len(sv) >= 5 and sv[lab_col].nunique() >= 2:
            try:
                result["sep_auroc_synergy"] = roc_auc_score(sv[lab_col], sv["synergy_proxy"])
            except Exception:
                pass

    return result


# ═══════════════════════════════════════════════════════════════════════════
# §20  MAIN EFFECTS USELESSNESS TESTS
# ═══════════════════════════════════════════════════════════════════════════

def _discrete_mi(x, y):
    """Mutual information between two discrete arrays (bits or small ints)."""
    from sklearn.metrics import mutual_info_score
    return mutual_info_score(x, y)


def _chi2_test(x, y):
    """Chi-square test of independence. Returns chi2, p, cramers_v."""
    ct = pd.crosstab(pd.Series(x, name="x"), pd.Series(y, name="y"))
    chi2, p, dof, _ = chi2_contingency(ct)
    n = len(x)
    k = min(ct.shape) - 1
    cramers_v = math.sqrt(chi2 / (n * max(k, 1))) if n > 0 and k > 0 else 0.0
    return float(chi2), float(p), float(cramers_v)


def main_effects_useless_tests(df, task_name):
    """Test whether labels are predictable from main effects alone.

    Returns (stats_rows, logreg_rows) — lists of dicts for CSV output.
    """
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import StratifiedKFold, cross_val_score

    lab_col = "clean_label" if "clean_label" in df.columns else "label"
    y_clean = df[lab_col].values.astype(int)
    y_noisy = df["label"].values.astype(int)
    has_A = df["has_A"].values.astype(int)
    has_B = df["has_B"].values.astype(int)
    state = has_A * 2 + has_B  # 0,1,2,3 encoding of (has_A, has_B)

    print(f"\n  [MAIN EFFECTS] {task_name}")

    # ── Part A: MI and chi-square ─────────────────────────────────────────
    stats_rows = []
    for feat_name, feat_vals in [("has_A", has_A), ("has_B", has_B), ("state_AB", state)]:
        for lab_name, lab_vals in [("clean_label", y_clean), ("noisy_label", y_noisy)]:
            mi = _discrete_mi(feat_vals, lab_vals)
            chi2, p_chi, cramers = _chi2_test(feat_vals, lab_vals)
            stats_rows.append({
                "task": task_name, "feature": feat_name, "label_type": lab_name,
                "MI": mi, "chi2": chi2, "chi2_p": p_chi, "cramers_v": cramers,
            })
            if lab_name == "clean_label":
                print(f"    {feat_name:10s}  MI={mi:.4f}  χ²={chi2:.1f}  p={p_chi:.2e}  V={cramers:.3f}")

    # ── Part B: Logistic regression AUROC baselines ───────────────────────
    logreg_rows = []
    feature_sets = {
        "has_A_only":       has_A.reshape(-1, 1),
        "has_B_only":       has_B.reshape(-1, 1),
        "has_A_has_B":      np.column_stack([has_A, has_B]),
        "A_B_interaction":  np.column_stack([has_A, has_B, has_A * has_B]),
    }
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    for fs_name, X in feature_sets.items():
        lr = LogisticRegression(max_iter=500, random_state=42, solver="lbfgs")
        try:
            scores = cross_val_score(lr, X, y_clean, cv=cv, scoring="roc_auc")
            auc_mean, auc_std = scores.mean(), scores.std()
        except Exception:
            auc_mean, auc_std = 0.5, 0.0

        rec = {"task": task_name, "feature_set": fs_name,
               "auroc_mean": auc_mean, "auroc_std": auc_std}

        # Permutation sanity check for the interaction model
        if fs_name == "A_B_interaction":
            perm_aucs = []
            rng = np.random.RandomState(42)
            for _ in range(CFG.MAIN_EFFECTS_PERM_N):
                y_perm = rng.permutation(y_clean)
                try:
                    ps = cross_val_score(lr, X, y_perm, cv=cv, scoring="roc_auc")
                    perm_aucs.append(ps.mean())
                except Exception:
                    perm_aucs.append(0.5)
            rec["perm_auroc_mean"] = np.mean(perm_aucs)
            rec["perm_auroc_std"] = np.std(perm_aucs)

        logreg_rows.append(rec)
        tag = f"  perm={rec.get('perm_auroc_mean', ''):.3f}" if "perm_auroc_mean" in rec else ""
        print(f"    LR {fs_name:20s}  AUROC={auc_mean:.3f} ± {auc_std:.3f}{tag}")

    # ── Part C: Both-motif-only subset (critical for DIST task) ───────────
    # Within has_A=has_B=1, main effects are constant → should be AUROC ~0.5
    both = df[(df["has_A"] == 1) & (df["has_B"] == 1)]
    if len(both) >= 20 and both[lab_col].nunique() >= 2:
        y_both = both[lab_col].values.astype(int)
        n_both = len(both)
        n_folds = min(5, min((y_both == 0).sum(), (y_both == 1).sum()))
        if n_folds < 2:
            print(f"    AB-only: skipped — too few of minority class "
                  f"(n={n_both}, pos={y_both.sum()}, neg={n_both - y_both.sum()})")
        else:
            cv_ab = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

            # n_heavy within AB-only
            if "n_heavy" in both.columns:
                n_heavy_both = both["n_heavy"].values.reshape(-1, 1)
                try:
                    lr_nh = LogisticRegression(max_iter=500, random_state=42, solver="lbfgs")
                    sc = cross_val_score(lr_nh, n_heavy_both, y_both, cv=cv_ab, scoring="roc_auc")
                    auc_nh = sc.mean()
                except Exception:
                    auc_nh = 0.5
                logreg_rows.append({
                    "task": task_name, "feature_set": "AB_only__n_heavy",
                    "auroc_mean": auc_nh, "auroc_std": 0.0, "subset": "both_motif",
                })
                print(f"    LR AB-only n_heavy       AUROC={auc_nh:.3f}  (n={n_both}, should be ~0.5)")

            # dist_AB within AB-only (should be high for DIST)
            if "dist_AB" in both.columns:
                valid_d = both[both["dist_AB"] >= 0]
                if len(valid_d) >= 20 and valid_d[lab_col].nunique() >= 2:
                    y_d = valid_d[lab_col].values.astype(int)
                    d_both = valid_d["dist_AB"].values.reshape(-1, 1)
                    n_folds_d = min(5, min((y_d == 0).sum(), (y_d == 1).sum()))
                    if n_folds_d >= 2:
                        try:
                            cv_d = StratifiedKFold(n_splits=n_folds_d, shuffle=True, random_state=42)
                            lr_d = LogisticRegression(max_iter=500, random_state=42, solver="lbfgs")
                            sc = cross_val_score(lr_d, d_both, y_d, cv=cv_d, scoring="roc_auc")
                            auc_dist = sc.mean()
                        except Exception:
                            auc_dist = 0.5
                        logreg_rows.append({
                            "task": task_name, "feature_set": "AB_only__dist_AB",
                            "auroc_mean": auc_dist, "auroc_std": 0.0, "subset": "both_motif",
                        })
                        print(f"    LR AB-only dist_AB       AUROC={auc_dist:.3f}  "
                              f"(n={len(valid_d)}, should be high for DIST)")
                    else:
                        print(f"    AB-only dist_AB: skipped — too few minority class "
                              f"(n={len(valid_d)}, pos={y_d.sum()})")
    elif len(both) >= 2:
        print(f"    AB-only: n={len(both)}, labels={both[lab_col].nunique()} — "
              f"need ≥20 with ≥2 labels for CV")

    return stats_rows, logreg_rows


# ═══════════════════════════════════════════════════════════════════════════
# §21  COUNTERFACTUAL EXPERIMENT (DIST task)
# ═══════════════════════════════════════════════════════════════════════════

def _score_smiles(model, smiles, device):
    """Get p(toxic) for a SMILES string using the trained model."""
    d = smiles_to_pyg_data(smiles, label=0)
    if d is None:
        return None
    d = d.to(device)
    d.batch = torch.zeros(d.x.size(0), dtype=torch.long, device=device)
    model.eval()
    with torch.no_grad():
        z = model(d)
        return float(torch.sigmoid(z).item())


def _get_closest_motif_pair(mol, smarts_A, smarts_B):
    """Find the closest pair of atoms (a_star, b_star) between motifs A and B.
    Returns (a_star, b_star, min_dist, shortest_path) or (None, None, -1, []).
    """
    A_atoms = get_motif_atom_indices(mol, smarts_A)
    B_atoms = get_motif_atom_indices(mol, smarts_B)
    if not A_atoms or not B_atoms:
        return None, None, -1, []
    try:
        dm = Chem.GetDistanceMatrix(mol)
    except Exception:
        return None, None, -1, []

    best_a, best_b, best_d = None, None, float("inf")
    for a in A_atoms:
        for b in B_atoms:
            if dm[a][b] < best_d:
                best_a, best_b, best_d = a, b, int(dm[a][b])

    if best_a is None:
        return None, None, -1, []

    try:
        path = list(Chem.rdmolops.GetShortestPath(mol, int(best_a), int(best_b)))
    except Exception:
        path = []
    return best_a, best_b, best_d, path


def _insert_carbons_into_bond(mol, bond_begin, bond_end, k):
    """Insert k carbon atoms into the bond between bond_begin and bond_end.

    Uses RWMol to:
      1. Remove the bond between bond_begin and bond_end
      2. Add k new carbon atoms as a chain
      3. Bond: bond_begin - C1 - C2 - ... - Ck - bond_end

    Returns sanitized mol or None.
    """
    rw = Chem.RWMol(mol)

    # Get existing bond type
    bond = rw.GetBondBetweenAtoms(int(bond_begin), int(bond_end))
    if bond is None:
        return None
    bond_type = bond.GetBondType()

    # Remove the bond
    rw.RemoveBond(int(bond_begin), int(bond_end))

    # Add k carbon atoms in a chain
    new_indices = []
    for _ in range(k):
        idx = rw.AddAtom(Chem.Atom(6))  # carbon
        new_indices.append(idx)

    # Wire up: bond_begin -> new[0] -> new[1] -> ... -> new[k-1] -> bond_end
    # Use SINGLE bonds for the inserted chain
    rw.AddBond(int(bond_begin), new_indices[0], Chem.BondType.SINGLE)
    for i in range(len(new_indices) - 1):
        rw.AddBond(new_indices[i], new_indices[i + 1], Chem.BondType.SINGLE)
    rw.AddBond(new_indices[-1], int(bond_end), Chem.BondType.SINGLE)

    return _safe_sanitize(rw.GetMol())


def generate_cf_break_dist(smiles, tau, max_add=5, seed=42):
    """Generate minimal-edit counterfactual that pushes motif distance > tau.

    CORRECT APPROACH: identifies the shortest path between closest motif atoms,
    then inserts k carbon atoms into a bond ALONG that path. This guarantees
    the shortest-path distance increases by ~k.

    Returns (cf_smiles, edit_size, new_dist) or (None, 0, -1).
    """
    rng = np.random.RandomState(seed)
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None, 0, -1

    a_star, b_star, orig_dist, path = _get_closest_motif_pair(
        mol, MOTIF_A_SMARTS, MOTIF_B_SMARTS)
    if a_star is None or orig_dist < 0 or orig_dist > tau:
        return None, 0, -1
    if len(path) < 2:
        return None, 0, -1

    orig_heavy = count_heavy_atoms(mol)

    # Candidate bonds along the shortest path (pairs of consecutive path atoms)
    path_bonds = [(path[i], path[i + 1]) for i in range(len(path) - 1)]

    # Try inserting progressively more carbons
    for k in range(1, max_add + 1):
        # Try each bond along the path (shuffle for variety)
        bond_order = list(range(len(path_bonds)))
        rng.shuffle(bond_order)

        for bi in bond_order:
            b_begin, b_end = path_bonds[bi]
            try:
                cf_mol = _insert_carbons_into_bond(mol, b_begin, b_end, k)
                if cf_mol is None:
                    continue

                cf_smi = Chem.MolToSmiles(cf_mol)
                cf_mol2 = Chem.MolFromSmiles(cf_smi)
                if cf_mol2 is None:
                    continue

                # Verify motifs still present
                if not has_motif(cf_mol2, MOTIF_A_SMARTS):
                    continue
                if not has_motif(cf_mol2, MOTIF_B_SMARTS):
                    continue

                new_dist = compute_motif_distance(cf_mol2, MOTIF_A_SMARTS, MOTIF_B_SMARTS)
                cf_heavy = count_heavy_atoms(cf_mol2)
                edit = cf_heavy - orig_heavy

                if new_dist > tau and edit > 0:
                    return cf_smi, edit, new_dist
            except Exception:
                continue

    return None, 0, -1


def generate_cf_ctrl_matched(smiles, add_k, tau, original_side, seed=42):
    """Generate control counterfactual: add exactly add_k heavy atoms
    WITHOUT changing the distance bucket.

    APPROACH: attach an alkyl chain (C×add_k) to an aromatic site that is
    NOT on the shortest path between motifs. This ensures the edit is
    size-matched but does not affect the A↔B distance.

    Returns (cf_smiles, new_dist) or (None, -1).
    """
    rng = np.random.RandomState(seed)
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None, -1

    orig_heavy = count_heavy_atoms(mol)

    # Find shortest-path atoms to avoid
    _, _, _, path = _get_closest_motif_pair(mol, MOTIF_A_SMARTS, MOTIF_B_SMARTS)
    path_set = set(path)

    # Also find all motif atoms to avoid
    A_atoms = get_motif_atom_indices(mol, MOTIF_A_SMARTS)
    B_atoms = get_motif_atom_indices(mol, MOTIF_B_SMARTS)
    avoid = path_set | A_atoms | B_atoms

    # Find candidate aromatic CH sites NOT on the path / motifs
    cH_sites = [a.GetIdx() for a in mol.GetAtoms()
                if a.GetIsAromatic() and a.GetTotalNumHs() > 0
                and a.GetIdx() not in avoid]

    # Fallback: any aromatic CH if all are on path
    if not cH_sites:
        cH_sites = [a.GetIdx() for a in mol.GetAtoms()
                    if a.GetIsAromatic() and a.GetTotalNumHs() > 0]
    if not cH_sites:
        return None, -1

    linker = "C" * add_k
    rxn_smarts = f"[cH:1]>>[c:1]{linker}"

    for attempt in range(CFG.CF_MAX_ATTEMPTS):
        try:
            rxn = AllChem.ReactionFromSmarts(rxn_smarts)
            if rxn is None:
                continue
            products = rxn.RunReactants((mol,))
            if not products:
                continue

            # Pick a product — shuffle to get different attachment sites
            prod_idx = (attempt + rng.randint(len(products))) % len(products)
            cf = _safe_sanitize(products[prod_idx][0])
            if cf is None:
                continue
            cf_smi = Chem.MolToSmiles(cf)
            cf_mol = Chem.MolFromSmiles(cf_smi)
            if cf_mol is None:
                continue

            # Verify motifs
            if not has_motif(cf_mol, MOTIF_A_SMARTS):
                continue
            if not has_motif(cf_mol, MOTIF_B_SMARTS):
                continue

            # Check exact edit size
            cf_heavy = count_heavy_atoms(cf_mol)
            if cf_heavy - orig_heavy != add_k:
                continue

            # Check distance stays on correct side of tau
            new_dist = compute_motif_distance(cf_mol, MOTIF_A_SMARTS, MOTIF_B_SMARTS)
            if original_side == "close" and new_dist <= tau:
                return cf_smi, new_dist
            elif original_side == "far" and new_dist > tau:
                return cf_smi, new_dist
        except Exception:
            continue

    return None, -1


def run_counterfactual_experiment(model, test_df, tau, device, N=30, seed=42):
    """Run paired counterfactual experiment on DIST task.

    For each AB_close molecule:
      1. Generate CF_break (push dist > tau)
      2. Generate CF_ctrl (same edit size, dist stays ≤ tau)
      3. Score all three with model
      4. Compare |Δp_break| vs |Δp_ctrl|

    Returns (pairs_df, summary_dict).
    """
    print(f"\n  [COUNTERFACTUAL] Running on DIST task (tau={tau}, N={N})")

    # Select AB_close molecules (label=1, both motifs)
    lab_col = "clean_label" if "clean_label" in test_df.columns else "label"
    candidates = test_df[(test_df["has_A"] == 1) & (test_df["has_B"] == 1) &
                         (test_df[lab_col] == 1)].copy()

    MAX_H = CFG.MAX_HEAVY_EXPLAIN
    candidates = candidates[candidates["smiles"].apply(
        lambda s: (Chem.MolFromSmiles(s) is not None and
                   count_heavy_atoms(Chem.MolFromSmiles(s)) <= MAX_H))]

    subset = candidates.sample(n=min(N, len(candidates)), random_state=seed)
    print(f"    Selected {len(subset)} AB_close molecules")

    pairs = []
    n_fail_break, n_fail_ctrl = 0, 0

    for mol_i, (_, row) in enumerate(subset.iterrows()):
        smi = row["smiles"]
        orig_mol = Chem.MolFromSmiles(smi)
        orig_dist = compute_motif_distance(orig_mol, MOTIF_A_SMARTS, MOTIF_B_SMARTS)
        orig_heavy = count_heavy_atoms(orig_mol)
        orig_p = _score_smiles(model, smi, device)
        if orig_p is None:
            continue

        # Generate CF_break
        cf_break_smi, edit_size, break_dist = generate_cf_break_dist(
            smi, tau, max_add=CFG.CF_MAX_ADD, seed=seed + mol_i)
        if cf_break_smi is None:
            n_fail_break += 1
            continue

        # Generate CF_ctrl with matched edit size
        cf_ctrl_smi, ctrl_dist = generate_cf_ctrl_matched(
            smi, edit_size, tau, "close", seed=seed + mol_i + 1000)
        if cf_ctrl_smi is None:
            n_fail_ctrl += 1
            continue

        # Sanity assertions — verify constraints before scoring
        cf_break_heavy = count_heavy_atoms(Chem.MolFromSmiles(cf_break_smi))
        cf_ctrl_heavy = count_heavy_atoms(Chem.MolFromSmiles(cf_ctrl_smi))
        ctrl_edit = cf_ctrl_heavy - orig_heavy

        sane = (orig_dist <= tau and
                break_dist > tau and
                ctrl_dist <= tau and
                edit_size == cf_break_heavy - orig_heavy and
                ctrl_edit == edit_size)
        if not sane:
            logger.warning(f"CF sanity fail: orig_d={orig_dist} break_d={break_dist} "
                           f"ctrl_d={ctrl_dist} edit={edit_size} ctrl_edit={ctrl_edit}")
            continue

        # Score counterfactuals
        p_break = _score_smiles(model, cf_break_smi, device)
        p_ctrl = _score_smiles(model, cf_ctrl_smi, device)
        if p_break is None or p_ctrl is None:
            continue

        delta_break = p_break - orig_p
        delta_ctrl = p_ctrl - orig_p

        pairs.append({
            "original_smiles": smi, "original_dist": orig_dist,
            "original_heavy": orig_heavy, "original_p": orig_p,
            "cf_break_smiles": cf_break_smi, "cf_break_dist": break_dist,
            "cf_break_heavy": cf_break_heavy,
            "p_break": p_break, "delta_break": delta_break,
            "cf_ctrl_smiles": cf_ctrl_smi, "cf_ctrl_dist": ctrl_dist,
            "cf_ctrl_heavy": cf_ctrl_heavy,
            "p_ctrl": p_ctrl, "delta_ctrl": delta_ctrl,
            "edit_size": edit_size,
            "abs_delta_break": abs(delta_break), "abs_delta_ctrl": abs(delta_ctrl),
            "sanity_pass": True,
        })

        if (mol_i + 1) % 10 == 0 or mol_i == 0:
            print(f"    [{mol_i+1}/{len(subset)}] pairs={len(pairs)} "
                  f"Δbreak={delta_break:+.3f} Δctrl={delta_ctrl:+.3f}")

    pairs_df = pd.DataFrame(pairs)
    n_sane = int(pairs_df["sanity_pass"].sum()) if not pairs_df.empty else 0
    print(f"    Completed: {len(pairs_df)} pairs (all sanity_pass=True), "
          f"break_fail={n_fail_break}, ctrl_fail={n_fail_ctrl}")

    # Build summary
    summary = {"n_pairs": len(pairs_df), "n_sanity_pass": n_sane,
               "n_fail_break": n_fail_break,
               "n_fail_ctrl": n_fail_ctrl, "tau": tau}

    if len(pairs_df) >= 3:
        abs_db = pairs_df["abs_delta_break"].values
        abs_dc = pairs_df["abs_delta_ctrl"].values
        diffs = abs_db - abs_dc

        summary["mean_abs_delta_break"] = float(np.mean(abs_db))
        summary["mean_abs_delta_ctrl"] = float(np.mean(abs_dc))
        summary["mean_diff"] = float(np.mean(diffs))
        summary["frac_break_bigger"] = float((abs_db > abs_dc).mean())

        # Bootstrap 95% CI for mean difference
        if len(diffs) >= 5:
            ci_lo, ci_hi = bootstrap_ci(diffs, n_bootstrap=CFG.DEL_BOOTSTRAP_N)
            summary["diff_ci_lo"] = ci_lo
            summary["diff_ci_hi"] = ci_hi

        # Wilcoxon signed-rank test (nonparametric paired test)
        try:
            stat, p_val = wilcoxon(abs_db, abs_dc, alternative="greater")
            summary["wilcoxon_stat"] = float(stat)
            summary["wilcoxon_p"] = float(p_val)
        except Exception:
            pass

        print(f"    |Δbreak|={summary['mean_abs_delta_break']:.4f}  "
              f"|Δctrl|={summary['mean_abs_delta_ctrl']:.4f}  "
              f"diff={summary['mean_diff']:.4f}  "
              f"frac_bigger={summary['frac_break_bigger']:.2f}")
        if "wilcoxon_p" in summary:
            print(f"    Wilcoxon p={summary['wilcoxon_p']:.4f}")
        if "diff_ci_lo" in summary:
            print(f"    95% CI for diff: [{summary['diff_ci_lo']:.4f}, {summary['diff_ci_hi']:.4f}]")
    else:
        print("    Too few pairs for statistics")

    return pairs_df, summary


def plot_counterfactual_effects(pairs_df, path):
    """Paired bar/box plot of |Δp_break| vs |Δp_ctrl|."""
    if pairs_df.empty or len(pairs_df) < 3:
        return
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Left: paired scatter
    ax = axes[0]
    ax.scatter(pairs_df["abs_delta_ctrl"], pairs_df["abs_delta_break"],
               alpha=0.7, edgecolors="k", s=50)
    lim = max(pairs_df["abs_delta_break"].max(), pairs_df["abs_delta_ctrl"].max()) * 1.1
    ax.plot([0, lim], [0, lim], "k--", alpha=0.4, label="y=x")
    ax.set_xlabel("|Δp_ctrl| (matched control)")
    ax.set_ylabel("|Δp_break| (interaction break)")
    ax.set_title("Counterfactual Paired Effects")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Right: box plot
    ax = axes[1]
    data = [pairs_df["abs_delta_break"].values, pairs_df["abs_delta_ctrl"].values]
    bp = ax.boxplot(data, labels=["Break\n(flip interaction)", "Control\n(matched edit)"],
                    patch_artist=True)
    bp["boxes"][0].set_facecolor("coral")
    bp["boxes"][1].set_facecolor("steelblue")
    ax.set_ylabel("|Δp|")
    ax.set_title(f"Counterfactual Effect Sizes (n={len(pairs_df)})")
    ax.grid(True, alpha=0.3, axis="y")

    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.close()


In [ ]:
def main():
    set_seed()
    device = CFG.DEVICE
    save_config()
    all_mismatches = []

    # ── STEP 1: Generate ──────────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("STEP 1: GENERATING SYNTHETIC DATASETS")
    print("=" * 70)

    print("\n--- Task 1: AND ---")
    df1, mm1 = generate_dataset_task1(CFG.N_TOTAL_PER_TASK, 42, CFG.LABEL_NOISE_RATE)
    df1.to_csv(os.path.join(CFG.DATA_DIR, "synthetic_task1.csv"), index=False)
    dataset_diagnostics(df1, "Task 1 (AND)")
    all_mismatches.extend(mm1)

    print("\n--- Task 2: OR ---")
    df2, mm2 = generate_dataset_task2(CFG.N_TOTAL_PER_TASK, 43, CFG.LABEL_NOISE_RATE)
    df2.to_csv(os.path.join(CFG.DATA_DIR, "synthetic_task2.csv"), index=False)
    dataset_diagnostics(df2, "Task 2 (OR)")
    all_mismatches.extend(mm2)

    print("\n--- Task 3: DIST ---")
    df3, mm3 = generate_dataset_task3(CFG.N_TOTAL_PER_TASK, 44, CFG.LABEL_NOISE_RATE, CFG.DIST_THRESH)
    df3.to_csv(os.path.join(CFG.DATA_DIR, "synthetic_task3.csv"), index=False)
    dataset_diagnostics(df3, "Task 3 (DIST)")
    all_mismatches.extend(mm3)

    # Mismatch log + integrity report
    if all_mismatches:
        pd.DataFrame(all_mismatches).to_csv(
            os.path.join(CFG.DATA_DIR, "label_mismatch_log.csv"), index=False)
    print(f"  Total label mismatches: {len(all_mismatches)}")

    tasks_df = {"task1_AND": df1, "task2_OR": df2, "task3_DIST": df3}
    integrity = build_integrity_report(tasks_df, all_mismatches)
    integrity.to_csv(os.path.join(CFG.DATA_DIR, "dataset_integrity_report.csv"), index=False)

    # Scaffold leakage report
    scaff_leak = scaffold_leakage_report(tasks_df)
    if not scaff_leak.empty:
        scaff_leak.to_csv(os.path.join(CFG.DATA_DIR, "scaffold_leakage_report.csv"), index=False)

    # ── STEP 1b: Artifact diagnostics ─────────────────────────────────────
    artifact_rows = []
    if CFG.ARTIFACT_CHECK:
        print("\n" + "=" * 70)
        print("STEP 1b: ARTIFACT DIAGNOSTICS")
        print("=" * 70)
        for tname, tdf in tasks_df.items():
            artifact_rows.append(artifact_diagnostics(tdf, tname))
        pd.DataFrame(artifact_rows).to_csv(
            os.path.join(CFG.DATA_DIR, "artifact_diagnostics.csv"), index=False)

    # ── STEP 1b2: Per-struct-class size balance ──────────────────────────
    print("\n  --- Size balance across struct_classes ---")
    for tname, tdf in tasks_df.items():
        per_class_size_diagnostic(tdf, tname)

    # ── STEP 1c: Main effects uselessness tests ──────────────────────────
    print("\n" + "=" * 70)
    print("STEP 1c: MAIN EFFECTS USELESSNESS TESTS")
    print("=" * 70)

    all_me_stats, all_me_logreg = [], []
    for tname, tdf in tasks_df.items():
        stats, logreg = main_effects_useless_tests(tdf, tname)
        all_me_stats.extend(stats)
        all_me_logreg.extend(logreg)
    pd.DataFrame(all_me_stats).to_csv(
        os.path.join(CFG.DATA_DIR, "main_effects_stats.csv"), index=False)
    pd.DataFrame(all_me_logreg).to_csv(
        os.path.join(CFG.DATA_DIR, "main_effects_logreg.csv"), index=False)

    # ── STEP 2: Convert & split ───────────────────────────────────────────
    print("\n" + "=" * 70)
    print("STEP 2: PyG CONVERSION & SPLITTING")
    print("=" * 70)

    datasets = {}
    for tname, df in tasks_df.items():
        dl, vi = build_pyg_dataset(df)
        sp = stratified_split(df, dl, vi, seed=42)
        datasets[tname] = {**sp, "df": df}
        for s in ["train", "val", "test"]:
            rows = sp[f"{s}_rows"]
            pos = df.loc[rows, "label"].mean() if len(rows) else 0
            print(f"  {tname} {s:5s}: n={len(rows)}  pos={pos:.2f}")

    # ── STEP 3: Train ─────────────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("STEP 3: TRAINING")
    print("=" * 70)

    models, training_log = {}, []
    for tname, ds in datasets.items():
        print(f"\n--- {tname} ---")
        m, h = train_model(ds["train"], ds["val"], tname,
                           epochs=CFG.EPOCHS, lr=CFG.LR,
                           batch_size=CFG.BATCH_SIZE, patience=CFG.PATIENCE, device=device)
        models[tname] = m
        for r in h: r["task"] = tname
        training_log.extend(h)
    pd.DataFrame(training_log).to_csv(os.path.join(CFG.DATA_DIR, "training_log.csv"), index=False)

    # ── STEP 4: Evaluate ──────────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("STEP 4: TEST EVALUATION")
    print("=" * 70)

    roc_data, eval_summary = {}, []
    for tname, ds in datasets.items():
        met, y, p = evaluate_model(models[tname], ds["test"], device)
        roc_data[tname] = (y, p)
        eval_summary.append({"task": tname, **met})
        print(f"  {tname}: AUROC={met['auroc']:.4f}  Acc={met['accuracy']:.4f}  F1={met['f1']:.4f}")
    pd.DataFrame(eval_summary).to_csv(os.path.join(CFG.DATA_DIR, "test_evaluation.csv"), index=False)
    plot_roc_curves(roc_data, os.path.join(CFG.DATA_DIR, "roc_curves.png"))

    # ── STEP 4a: Scaffold-held-out evaluation ─────────────────────────────
    print("\n  --- Scaffold generalisation ---")
    scaff_eval_rows = []
    for tname, ds in datasets.items():
        she = scaffold_held_out_eval(models[tname], ds, ds["df"], device)
        scaff_eval_rows.append({"task": tname, **she})
        novel_auc = she.get("novel_auroc")
        auc_str = f"{novel_auc:.4f}" if novel_auc is not None else "N/A (too few novel)"
        print(f"  {tname}: test_scaffolds={she['n_test_scaffolds']} "
              f"overlap={she['overlap_pct']:.0f}%  novel_mols={she['n_novel_mols']} "
              f"novel_AUROC={auc_str}")
    pd.DataFrame(scaff_eval_rows).to_csv(
        os.path.join(CFG.DATA_DIR, "scaffold_held_out_eval.csv"), index=False)

    # ── STEP 4b: Counterfactual experiment (DIST task) ────────────────────
    cf_pairs_df, cf_summary = pd.DataFrame(), {}
    if "task3_DIST" in datasets:
        print("\n" + "=" * 70)
        print("STEP 4b: COUNTERFACTUAL EXPERIMENT (DIST)")
        print("=" * 70)
        ds3 = datasets["task3_DIST"]
        test_df3 = ds3["df"].loc[ds3["test_rows"]].reset_index(drop=True)
        cf_pairs_df, cf_summary = run_counterfactual_experiment(
            models["task3_DIST"], test_df3, CFG.DIST_THRESH, device,
            N=CFG.CF_N, seed=CFG.SEED)
        if not cf_pairs_df.empty:
            cf_pairs_df.to_csv(os.path.join(CFG.DATA_DIR, "counterfactual_pairs.csv"), index=False)
            pd.DataFrame([cf_summary]).to_csv(
                os.path.join(CFG.DATA_DIR, "counterfactual_summary.csv"), index=False)
            plot_counterfactual_effects(cf_pairs_df,
                                        os.path.join(CFG.DATA_DIR, "counterfactual_effects.png"))

    # ── STEP 5: Explanation verification (OUR METHOD) ─────────────────────
    print("\n" + "=" * 70)
    print("STEP 5: EXPLANATION VERIFICATION (OUR METHOD)")
    print("=" * 70)

    all_exp, all_conv = [], {}
    all_synergy = []

    for tname, ds in datasets.items():
        model = models[tname]
        test_df = ds["df"].loc[ds["test_rows"]].reset_index(drop=True)
        conv_rows = []

        for n_st2 in CFG.N_SAMPLES_ST2_LIST:
            rdf = verify_explanations(
                model, test_df, tname, device,
                n_explain=CFG.N_EXPLAIN, n_samples_st2=n_st2,
                n_samples_sv=CFG.N_SAMPLES_SV, sa_iterations=CFG.SA_ITERATIONS)
            if rdf.empty: continue
            rdf["task"] = tname
            all_exp.append(rdf)

            # Aggregate
            agg = {"task": tname, "n_samples_st2": n_st2,
                   "loc_main_mean": rdf["loc_main"].mean(),
                   "loc_main_std": rdf["loc_main"].std(),
                   "del_auc_mean": rdf["del_auc"].mean(),
                   "del_auc_rand_mean": rdf["del_auc_rand"].mean(),
                   "ins_auc_mean": rdf.get("ins_auc", pd.Series([0])).mean(),
                   "ins_auc_rand_mean": rdf.get("ins_auc_rand", pd.Series([0])).mean()}

            both = rdf[(rdf["has_A"]==1) & (rdf["has_B"]==1)]
            if not both.empty:
                agg["dom_mean"] = both["dom"].mean()
                vr = both[both["rank"]>0]["rank"]
                agg["rank_median"] = vr.median() if len(vr) else -1
                agg["top1_rate"] = float((vr==1).sum())/max(len(vr),1)
                agg["top3_rate"] = float((vr<=3).sum())/max(len(vr),1)
            else:
                agg.update({"dom_mean": 0, "rank_median": -1, "top1_rate": 0, "top3_rate": 0})

            agg.update(compute_synergy_sign_rate(rdf))
            agg.update(compute_interaction_separability(rdf))
            if "dist_AB" in rdf.columns:
                agg.update(compute_distance_correlation(rdf))

            conv_rows.append(agg)
            print(f"    loc={agg['loc_main_mean']:.3f}  dom={agg.get('dom_mean',0):.3f}  "
                  f"top1={agg.get('top1_rate',0):.2f}  del={agg['del_auc_mean']:.3f}  "
                  f"ins={agg.get('ins_auc_mean',0):.3f}")

        # Synergy proxy for this task
        syn_df = evaluate_synergy_proxy(model, test_df, tname, device, n_explain=CFG.N_EXPLAIN)
        if not syn_df.empty:
            all_synergy.append(syn_df)

        if conv_rows:
            all_conv[tname] = pd.DataFrame(conv_rows)

    # ── STEP 6: Baseline evaluations ──────────────────────────────────────
    print("\n" + "=" * 70)
    print("STEP 6: BASELINE EVALUATIONS")
    print("=" * 70)

    all_baseline_exp = []
    if CFG.GRAD_BASELINE:
        for tname, ds in datasets.items():
            model = models[tname]
            test_df = ds["df"].loc[ds["test_rows"]].reset_index(drop=True)
            grad_df = evaluate_baseline_grad(
                model, test_df, tname, device,
                n_explain=CFG.BASELINE_N_EXPLAIN)
            if not grad_df.empty:
                grad_df["task"] = tname
                all_baseline_exp.append(grad_df)

    # ── STEP 7: Stability test ────────────────────────────────────────────
    stability_rows = []
    if CFG.STABILITY_SEEDS and len(CFG.STABILITY_SEEDS) >= 2:
        print("\n" + "=" * 70)
        print("STEP 7: EXPLANATION STABILITY")
        print("=" * 70)
        for tname, ds in datasets.items():
            model = models[tname]
            test_df = ds["df"].loc[ds["test_rows"]].reset_index(drop=True)
            sdf = stability_test(model, test_df, tname, device)
            if not sdf.empty:
                stability_rows.append(sdf)

    # ── STEP 8: Save all CSVs ─────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("STEP 8: SAVING OUTPUTS")
    print("=" * 70)

    # Merge our method + baselines into one explanation_metrics.csv
    all_exp_combined = []
    if all_exp:
        all_exp_combined.extend(all_exp)
    if all_baseline_exp:
        all_exp_combined.extend(all_baseline_exp)

    if all_exp_combined:
        all_exp_df = pd.concat(all_exp_combined, ignore_index=True)
        all_exp_df.to_csv(os.path.join(CFG.DATA_DIR, "explanation_metrics.csv"), index=False)
        print(f"  explanation_metrics.csv: {len(all_exp_df)} rows, methods={all_exp_df['method'].unique().tolist()}")

        # deletion_summary.csv
        del_summary = build_deletion_summary(all_exp_df)
        del_summary.to_csv(os.path.join(CFG.DATA_DIR, "deletion_summary.csv"), index=False)
        print(f"  deletion_summary.csv: {len(del_summary)} rows")
    else:
        all_exp_df = pd.DataFrame()

    # convergence_summary.csv
    if all_conv:
        pd.concat(all_conv.values(), ignore_index=True).to_csv(
            os.path.join(CFG.DATA_DIR, "convergence_summary.csv"), index=False)

    # synergy proxy
    synergy_combined = pd.concat(all_synergy, ignore_index=True) if all_synergy else pd.DataFrame()
    if not synergy_combined.empty:
        synergy_combined.to_csv(os.path.join(CFG.DATA_DIR, "synergy_proxy.csv"), index=False)

    # interaction_distance_analysis.csv (DIST task)
    dist_analysis_rows = []
    if all_exp:
        ours_df = pd.concat(all_exp, ignore_index=True)
        for tname in ["task3_DIST"]:
            t_ours = ours_df[ours_df["task"] == tname]
            t_syn = synergy_combined[synergy_combined["task"] == tname] if not synergy_combined.empty else pd.DataFrame()
            if not t_ours.empty:
                dist_analysis_rows.append(
                    build_interaction_distance_analysis(t_ours, t_syn, tname))
                # Distance bucket CSV
                buckets = distance_bucket_analysis(t_ours, t_syn)
                if not buckets.empty:
                    buckets.to_csv(os.path.join(CFG.DATA_DIR, "distance_buckets.csv"), index=False)
    if dist_analysis_rows:
        pd.DataFrame(dist_analysis_rows).to_csv(
            os.path.join(CFG.DATA_DIR, "interaction_distance_analysis.csv"), index=False)

    # stability
    if stability_rows:
        pd.concat(stability_rows, ignore_index=True).to_csv(
            os.path.join(CFG.DATA_DIR, "stability.csv"), index=False)

    # ── STEP 9: Plots ─────────────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("STEP 9: PLOTS")
    print("=" * 70)

    if not all_exp_df.empty:
        # ROC curves already saved in step 4
        t1 = all_exp_df[(all_exp_df["task"]=="task1_AND") & (all_exp_df["method"]=="ours")]
        if not t1.empty:
            best_ns = t1["n_samples_st2"].max()
            plot_interaction_histogram(t1[t1["n_samples_st2"]==best_ns], "Task1 AND",
                                       os.path.join(CFG.DATA_DIR, "interaction_hist_task1.png"))

        plot_deletion_comparison(all_exp_df, os.path.join(CFG.DATA_DIR, "deletion_comparison.png"))
        plot_method_comparison(all_exp_df, os.path.join(CFG.DATA_DIR, "method_comparison.png"))

    if all_conv:
        plot_convergence(all_conv, os.path.join(CFG.DATA_DIR, "convergence_plot.png"))

    # Distance bucket plot (DIST task)
    if all_exp:
        ours_df = pd.concat(all_exp, ignore_index=True)
        t3 = ours_df[ours_df["task"] == "task3_DIST"]
        t3_syn = synergy_combined[synergy_combined["task"] == "task3_DIST"] if not synergy_combined.empty else None
        if not t3.empty:
            buckets = distance_bucket_analysis(t3, t3_syn)
            if not buckets.empty:
                plot_distance_buckets(buckets, os.path.join(CFG.DATA_DIR, "distance_buckets.png"))

    # ── STEP 10: Sanity check on real data ────────────────────────────────
    real_csv = "EXP_RES.csv"
    if os.path.exists(real_csv):
        print("\n" + "=" * 70)
        print("STEP 10: SANITY CHECK ON REAL DATA")
        print("=" * 70)
        try:
            rdf = pd.read_csv(real_csv)
            m = list(models.values())[0]
            for _, row in rdf.sample(3, random_state=42).iterrows():
                smi = row["smiles"]
                try:
                    r = explain_molecule_for_verification(m, smi, device, 50, 50, 100)
                    if r:
                        dz = r["vfunc_stats"]["total_change"]
                        print(f"  {smi[:50]}…  Δz={dz:.4f}  groups={len(r['best_partition'])}  ✓")
                except Exception as e:
                    print(f"  {smi[:50]}…  FAIL: {e}")
        except Exception as e:
            print(f"  {e}")

    # ── RESULTS SUMMARY ───────────────────────────────────────────────────
    print("\n" + "═" * 70)
    print("          RESULTS SUMMARY")
    print("═" * 70)

    print("\n[MODEL PERFORMANCE]")
    for r in eval_summary:
        print(f"  {r['task']:15s}  AUROC={r['auroc']:.4f}  F1={r['f1']:.4f}")

    if artifact_rows:
        print(f"\n[ARTIFACT DIAGNOSTICS]")
        for ar in artifact_rows:
            print(f"  {ar['task']:15s}  nuisance_AUROC={ar['nuisance_auroc_mean']:.3f} ± "
                  f"{ar['nuisance_auroc_std']:.3f}  {ar['status']}")

    if not scaff_leak.empty:
        print(f"\n[SCAFFOLD LEAKAGE]")
        for task in scaff_leak["task"].unique():
            t = scaff_leak[scaff_leak["task"] == task]
            n_single = (t["n_label_states"] == 1).sum()
            print(f"  {task:15s}  {len(t)} scaffolds, {n_single} single-label "
                  f"({n_single/max(len(t),1):.0%})")

    if scaff_eval_rows:
        print(f"\n[SCAFFOLD GENERALISATION]")
        for sr in scaff_eval_rows:
            novel_auc = sr.get("novel_auroc")
            auc_str = f"novel_AUROC={novel_auc:.4f}" if novel_auc is not None else "too few novel scaffolds"
            print(f"  {sr['task']:15s}  overlap={sr['overlap_pct']:.0f}%  "
                  f"novel_mols={sr['n_novel_mols']}  {auc_str}")

    print(f"\n[LABEL INTEGRITY]")
    print(f"  Total mismatches detected & fixed: {len(all_mismatches)}")

    if not all_exp_df.empty:
        print(f"\n[EXPLANATION METRICS BY METHOD]")
        for method in all_exp_df["method"].unique():
            msub = all_exp_df[all_exp_df["method"] == method]
            print(f"  Method: {method}  (n={len(msub)})")
            print(f"    Localization:    {msub['loc_main'].mean():.3f} ± {msub['loc_main'].std():.3f}")
            print(f"    Del AUC:         {msub['del_auc'].mean():.4f} ± {msub['del_auc'].std():.4f}")
            print(f"    Del rand:        {msub['del_auc_rand'].mean():.4f}")
            if "ins_auc" in msub.columns:
                print(f"    Ins AUC:         {msub['ins_auc'].mean():.4f}")

    if all_conv:
        print("\n[INTERACTION METRICS — OUR METHOD, highest n_st2]")
        for tn, cdf in all_conv.items():
            b = cdf.iloc[-1]
            print(f"  {tn}:")
            print(f"    Dominance:     {b.get('dom_mean',0):.3f}")
            print(f"    Top-1 rate:    {b.get('top1_rate',0):.2f}")
            if "synergy_positive_rate_label1" in b:
                print(f"    Synergy+ (y=1): {b.get('synergy_positive_rate_label1',0):.2f}")
            if "interaction_sep_auroc" in b and not pd.isna(b.get("interaction_sep_auroc")):
                print(f"    Separability AUROC: {b['interaction_sep_auroc']:.3f}")
            if "spearman_absI_vs_dist" in b.index if hasattr(b, 'index') else "spearman_I_AB_dist" in b:
                key = "spearman_I_AB_dist" if "spearman_I_AB_dist" in b else "spearman_absI_vs_dist"
                if not pd.isna(b.get(key)):
                    print(f"    Spearman(|I_AB|, dist): {b[key]:.3f}")

    if dist_analysis_rows:
        print(f"\n[DIST TASK — INTERACTION vs DISTANCE]")
        for d in dist_analysis_rows:
            for k, v in d.items():
                if k != "task" and not pd.isna(v):
                    print(f"    {k}: {v:.4f}" if isinstance(v, float) else f"    {k}: {v}")

    if stability_rows:
        stab_df = pd.concat(stability_rows, ignore_index=True)
        print(f"\n[STABILITY — across {len(CFG.STABILITY_SEEDS)} seeds]")
        for task in stab_df["task"].unique():
            ts = stab_df[stab_df["task"] == task]
            print(f"  {task}: loc_std={ts['loc_std'].mean():.3f}  I_AB_std={ts['I_AB_std'].mean():.4f}  (n={len(ts)})")

    # Main effects uselessness summary
    if all_me_logreg:
        print(f"\n[MAIN EFFECTS — USELESSNESS TEST]")
        me_df = pd.DataFrame(all_me_logreg)
        for tname in me_df["task"].unique():
            t = me_df[me_df["task"] == tname]
            print(f"  {tname}:")
            for _, r in t.iterrows():
                tag = ""
                if "perm_auroc_mean" in r and not pd.isna(r.get("perm_auroc_mean")):
                    tag = f"  [perm={r['perm_auroc_mean']:.3f}]"
                print(f"    {r['feature_set']:20s}  AUROC={r['auroc_mean']:.3f} ± {r['auroc_std']:.3f}{tag}")

    # Counterfactual summary
    if cf_summary and cf_summary.get("n_pairs", 0) >= 3:
        print(f"\n[COUNTERFACTUAL — DIST TASK]")
        print(f"  Pairs: {cf_summary['n_pairs']}  "
              f"|Δbreak|={cf_summary.get('mean_abs_delta_break',0):.4f}  "
              f"|Δctrl|={cf_summary.get('mean_abs_delta_ctrl',0):.4f}")
        print(f"  Diff: {cf_summary.get('mean_diff',0):.4f}  "
              f"Frac(break>ctrl): {cf_summary.get('frac_break_bigger',0):.2f}")
        if "diff_ci_lo" in cf_summary:
            print(f"  95% CI: [{cf_summary['diff_ci_lo']:.4f}, {cf_summary['diff_ci_hi']:.4f}]")
        if "wilcoxon_p" in cf_summary:
            print(f"  Wilcoxon p={cf_summary['wilcoxon_p']:.4f}")

    print("\n[VERDICT]")
    if not all_exp_df.empty:
        ours = all_exp_df[all_exp_df["method"] == "ours"]
        grad = all_exp_df[all_exp_df["method"] == "grad_x_input"]
        loc_ours = ours["loc_main"].mean() if not ours.empty else 0
        loc_grad = grad["loc_main"].mean() if not grad.empty else 0
        del_ours = ours["del_auc"].mean() if not ours.empty else 0
        del_grad = grad["del_auc"].mean() if not grad.empty else 0

        print(f"  Localization:   ours={loc_ours:.3f}  grad={loc_grad:.3f}  "
              f"{'✓ ours better' if loc_ours > loc_grad else '— grad better or tied'}")
        print(f"  Faithfulness:   ours={del_ours:.4f}  grad={del_grad:.4f}  "
              f"{'✓ ours better' if del_ours > del_grad else '— grad better or tied'}")

    # Interaction claim verdict
    if all_me_logreg:
        me_df = pd.DataFrame(all_me_logreg)
        and_int = me_df[(me_df["task"]=="task1_AND") & (me_df["feature_set"]=="A_B_interaction")]
        and_main = me_df[(me_df["task"]=="task1_AND") & (me_df["feature_set"]=="has_A_has_B")]
        if not and_int.empty and not and_main.empty:
            gap = and_int["auroc_mean"].iloc[0] - and_main["auroc_mean"].iloc[0]
            print(f"  Interaction signal: AND task interaction_AUROC - main_AUROC = {gap:.3f}  "
                  f"{'✓ interaction needed' if gap > 0.1 else '— weak signal'}")

    if cf_summary and cf_summary.get("n_pairs", 0) >= 3:
        diff = cf_summary.get("mean_diff", 0)
        p = cf_summary.get("wilcoxon_p", 1.0)
        print(f"  Counterfactual:  |Δbreak|-|Δctrl|={diff:.4f}  p={p:.4f}  "
              f"{'✓ interaction causal' if diff > 0 and p < 0.05 else '— not significant'}")

    print("\n[OUTPUT FILES]")
    for f in sorted(os.listdir(CFG.DATA_DIR)):
        fp = os.path.join(CFG.DATA_DIR, f)
        if os.path.isfile(fp):
            print(f"  {f:45s} {os.path.getsize(fp):>10,} bytes")

    print("\n" + "═" * 70)
    print("  Part 5 v5 — COMPLETE")
    print("═" * 70)


In [ ]:
if __name__ == "__main__":
    main()
